# Эксперимент 2: объединение токенов на входе

**Статус: no_go**

## Гипотеза

После неудачи с расширением словаря (см. [эксперимент 1](experiment-1-tokenizer-vocab-extension.ipynb))
выбран более консервативный механизм: не создавать новые "слова", а
объединять только те последовательности токенов, которые уже
встречаются при стандартной токенизации — и обучать отдельную
небольшую таблицу входных эмбеддингов для этих объединений. Модель,
выходной словарь и остальные веса остаются замороженными.

Почему именно этот механизм:
- не требует отдельной сети-энкодера на каждый запрос;
- не сворачивает вопрос в один усреднённый вектор — сохраняется
  локальная структура текста;
- не разрушает существующую сегментацию (объединения строятся строго
  поверх реальных токенов baseline, а не поверх новых "слов", как в
  эксперименте 1).


In [1]:
%pip install -q "transformers>=4.46" tokenizers



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Метод

- До 4096 объединений из 2–4 исходных token ID, встреченных ≥5 раз на
  обучающем корпусе (10 000 документов Wikipedia RU)
- Объединяются только части одного русского слова; числа, пунктуация,
  границы слов защищены
- Критерий перехода к GPU-проверке: сокращение числа позиций ≥5% как
  на корпусе, так и на полных MCQ-промптах validation

Список объединений (результат отбора по частоте на train-корпусе)
встроен ниже как данные. Код объединения (`Merger`, `encode`) приведён
полностью, без внешних зависимостей от репозитория.


In [2]:
import json

from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

# 4096 объединений — результат отбора по частоте на train-корпусе
# (10 000 документов Wikipedia RU), встроены как данные.
MERGES_JSON = '''[{"ids":[51844,15630,53887,36916],"train_count":4443},{"ids":[51844,15630,53887],"train_count":4489},{"ids":[15630,53887,36916],"train_count":4443},{"ids":[18727,63469,16256],"train_count":4164},{"ids":[150704,32884,77779,26125],"train_count":1783},{"ids":[29189,152950,37835],"train_count":2500},{"ids":[15630,53887],"train_count":4842},{"ids":[151447,150704,32884,77779],"train_count":1598},{"ids":[51844,15630],"train_count":4649},{"ids":[61757,9080,7358,154182],"train_count":1521},{"ids":[53887,36916],"train_count":4485},{"ids":[150704,32884,77779],"train_count":2201},{"ids":[63469,16256],"train_count":4183},{"ids":[18727,63469],"train_count":4168},{"ids":[17417,11777,16669],"train_count":1976},{"ids":[32884,77779,26125],"train_count":1784},{"ids":[63046,148660],"train_count":3561},{"ids":[151447,150704,32884],"train_count":1650},{"ids":[61757,9080,7358],"train_count":1639},{"ids":[148309,16467,13113,2175],"train_count":1084},{"ids":[16467,13113,2175],"train_count":1617},{"ids":[33638,17417,11777,16669],"train_count":1074},{"ids":[16467,13113,2175,156204],"train_count":1060},{"ids":[9080,7358,154182],"train_count":1521},{"ids":[15481,19937],"train_count":2885},{"ids":[19582,17417,11777],"train_count":1442},{"ids":[207347,1403,181923],"train_count":1436},{"ids":[19582,17417,11777,16669],"train_count":897},{"ids":[151728,149800],"train_count":2691},{"ids":[29189,152950],"train_count":2690},{"ids":[206694,7358,15481],"train_count":1336},{"ids":[77436,149537],"train_count":2596},{"ids":[152950,37835],"train_count":2579},{"ids":[17417,11777],"train_count":2572},{"ids":[150704,32884],"train_count":2449},{"ids":[148309,16467,13113],"train_count":1159},{"ids":[16362,22078,148284],"train_count":1147},{"ids":[32884,77779],"train_count":2292},{"ids":[160847,157759,6389],"train_count":1141},{"ids":[46145,149208],"train_count":2280},{"ids":[50806,13603,6240,26046],"train_count":756},{"ids":[33638,17417,11777],"train_count":1100},{"ids":[151988,37855],"train_count":2138},{"ids":[13113,2175,156204],"train_count":1060},{"ids":[11777,16669],"train_count":2113},{"ids":[33207,149243],"train_count":2036},{"ids":[206694,7358],"train_count":2032},{"ids":[148275,31889],"train_count":2006},{"ids":[82383,153298,13113,1424],"train_count":668},{"ids":[33207,149243,152027],"train_count":991},{"ids":[77969,6240],"train_count":1981},{"ids":[33207,149243,231448],"train_count":986},{"ids":[18727,6240],"train_count":1963},{"ids":[148378,14942],"train_count":1884},{"ids":[43800,5301,13302],"train_count":937},{"ids":[77779,26125],"train_count":1864},{"ids":[13113,2175],"train_count":1859},{"ids":[29189,177501,16256],"train_count":893},{"ids":[16797,58361],"train_count":1773},{"ids":[9080,7358],"train_count":1763},{"ids":[7358,15481],"train_count":1733},{"ids":[77436,150621],"train_count":1733},{"ids":[46145,149208,44633],"train_count":863},{"ids":[6240,7721],"train_count":1719},{"ids":[16467,13113],"train_count":1710},{"ids":[82383,153298,13113],"train_count":854},{"ids":[6240,7721,2115,26125],"train_count":566},{"ids":[61757,9080],"train_count":1682},{"ids":[149167,42565,171144],"train_count":838},{"ids":[58427,149717],"train_count":1664},{"ids":[151447,150704],"train_count":1660},{"ids":[149746,8789],"train_count":1648},{"ids":[50806,13603,6240],"train_count":822},{"ids":[17417,11777,42565,155130],"train_count":541},{"ids":[19582,17417,11777,42565],"train_count":541},{"ids":[16467,13113,2175,42565],"train_count":540},{"ids":[16362,22078],"train_count":1586},{"ids":[1403,181923],"train_count":1583},{"ids":[23514,36103],"train_count":1582},{"ids":[5301,13302],"train_count":1568},{"ids":[19582,17417],"train_count":1564},{"ids":[13603,6240,26046],"train_count":778},{"ids":[7358,154182],"train_count":1543},{"ids":[15481,152993],"train_count":1542},{"ids":[2175,12751],"train_count":1540},{"ids":[149269,8789,54025],"train_count":760},{"ids":[9080,77969,6240],"train_count":753},{"ids":[7721,77969,6240],"train_count":734},{"ids":[207347,1403],"train_count":1467},{"ids":[174856,149800],"train_count":1459},{"ids":[153298,13113,1424],"train_count":727},{"ids":[21445,28696,45527],"train_count":726},{"ids":[35884,148367,38183],"train_count":720},{"ids":[28696,45527,7754],"train_count":706},{"ids":[42395,28696],"train_count":1410},{"ids":[159489,151639,7754,148276],"train_count":469},{"ids":[28696,1739],"train_count":1378},{"ids":[148309,16467],"train_count":1344},{"ids":[26269,11796],"train_count":1342},{"ids":[33422,15481],"train_count":1332},{"ids":[160847,157759],"train_count":1332},{"ids":[28376,16806],"train_count":1317},{"ids":[231238,220449,32392],"train_count":656},{"ids":[61057,152993],"train_count":1306},{"ids":[157759,6389],"train_count":1301},{"ids":[22078,148284],"train_count":1297},{"ids":[150387,33422],"train_count":1290},{"ids":[2175,156204],"train_count":1289},{"ids":[46145,149208,55389],"train_count":635},{"ids":[7422,148292,15481,7358],"train_count":422},{"ids":[13113,1424],"train_count":1255},{"ids":[33638,17417],"train_count":1251},{"ids":[28696,45527],"train_count":1247},{"ids":[150424,18768,148292,149452],"train_count":415},{"ids":[18727,149095,149827,13292],"train_count":414},{"ids":[150828,42317,67958],"train_count":620},{"ids":[150424,18768,148292],"train_count":619},{"ids":[150704,32884,77779,167596],"train_count":412},{"ids":[32724,10161,149926],"train_count":611},{"ids":[21445,28696,45527,7754],"train_count":407},{"ids":[23514,36103,148317],"train_count":610},{"ids":[26607,18727,32392],"train_count":606},{"ids":[52337,64257,148706],"train_count":604},{"ids":[32884,77779,26125,32401],"train_count":402},{"ids":[7958,159881,48623],"train_count":602},{"ids":[7550,26125,39427],"train_count":599},{"ids":[42565,170741],"train_count":1178},{"ids":[193863,149562,41546],"train_count":576},{"ids":[7550,26125],"train_count":1143},{"ids":[15481,95306],"train_count":1141},{"ids":[7550,26125,39427,44633],"train_count":380},{"ids":[42565,155130],"train_count":1137},{"ids":[7721,2115,26125],"train_count":568},{"ids":[6240,7721,2115],"train_count":566},{"ids":[160761,173910],"train_count":1132},{"ids":[17417,11777,42565],"train_count":562},{"ids":[35884,151709,181923],"train_count":562},{"ids":[5624,42317],"train_count":1123},{"ids":[5624,23514,36103],"train_count":561},{"ids":[149591,7105],"train_count":1107},{"ids":[148721,6240,7721,2115],"train_count":366},{"ids":[5241,149147,16763],"train_count":548},{"ids":[43800,5301,13302,151397],"train_count":364},{"ids":[149501,14666,45679,6389],"train_count":364},{"ids":[8789,16806,5092],"train_count":543},{"ids":[11777,42565,155130],"train_count":542},{"ids":[149243,152027],"train_count":1083},{"ids":[13113,2175,42565],"train_count":540},{"ids":[150387,33422,153056],"train_count":540},{"ids":[7105,148275],"train_count":1074},{"ids":[150175,151203,1739],"train_count":535},{"ids":[157695,47552,28696,45527],"train_count":356},{"ids":[156211,10064,16763],"train_count":533},{"ids":[8789,54025],"train_count":1061},{"ids":[149413,156211,10064,16763],"train_count":352},{"ids":[16362,148294,9080,77436],"train_count":350},{"ids":[148489,15630,53887,57868],"train_count":350},{"ids":[198168,156204],"train_count":1050},{"ids":[2115,34981],"train_count":1048},{"ids":[148294,9080,77436,150621],"train_count":349},{"ids":[151247,57868],"train_count":1047},{"ids":[45527,7754],"train_count":1031},{"ids":[8789,16806],"train_count":1028},{"ids":[32392,148277,154838],"train_count":513},{"ids":[151982,8789,41635],"train_count":510},{"ids":[32392,148277],"train_count":1012},{"ids":[36103,148317],"train_count":1012},{"ids":[148378,14942,77436],"train_count":504},{"ids":[148275,7105],"train_count":1006},{"ids":[149208,55389],"train_count":1001},{"ids":[149243,231448],"train_count":997},{"ids":[22220,23514,36103],"train_count":498},{"ids":[33207,204086,150584],"train_count":498},{"ids":[42565,171144],"train_count":996},{"ids":[77436,153232],"train_count":994},{"ids":[32884,77779,26125,42308],"train_count":330},{"ids":[186747,148634,9080,48623],"train_count":330},{"ids":[234875,148687,150006],"train_count":494},{"ids":[17417,2941,54025,5092],"train_count":329},{"ids":[43800,5301],"train_count":972},{"ids":[151639,7754,148276],"train_count":486},{"ids":[210138,150704,32884,77779],"train_count":324},{"ids":[16362,148294,9080],"train_count":484},{"ids":[44093,159859,149665],"train_count":483},{"ids":[5624,42317,221196],"train_count":482},{"ids":[149191,28696,1739],"train_count":481},{"ids":[42317,67958],"train_count":960},{"ids":[5092,32401],"train_count":959},{"ids":[33422,1424],"train_count":956},{"ids":[159489,151639,7754],"train_count":477},{"ids":[7422,148292,15481],"train_count":475},{"ids":[22346,6733],"train_count":949},{"ids":[23947,61628,12014],"train_count":472},{"ids":[20289,159489,151639,7754],"train_count":314},{"ids":[153298,13113],"train_count":942},{"ids":[148604,148303,158559],"train_count":470},{"ids":[53555,160259,32884,2115],"train_count":312},{"ids":[58297,53555,160259,32884],"train_count":312},{"ids":[149037,148685],"train_count":936},{"ids":[148292,15481,7358],"train_count":466},{"ids":[7958,159881],"train_count":929},{"ids":[149191,15630,69729],"train_count":464},{"ids":[54025,5092,32401],"train_count":462},{"ids":[176215,148275,31889],"train_count":461},{"ids":[13603,6240],"train_count":918},{"ids":[14666,45679,6389,158638],"train_count":306},{"ids":[151183,2175,12751],"train_count":459},{"ids":[160149,4640,34981],"train_count":458},{"ids":[149208,44633],"train_count":914},{"ids":[7721,26125],"train_count":906},{"ids":[7550,26125,158668],"train_count":452},{"ids":[5241,149147,16763,32884],"train_count":300},{"ids":[177501,16256],"train_count":899},{"ids":[29189,177501],"train_count":898},{"ids":[149233,148464,6733],"train_count":449},{"ids":[148634,9080],"train_count":896},{"ids":[16797,58361,148288],"train_count":445},{"ids":[151706,14942,47745],"train_count":444},{"ids":[7958,28696],"train_count":878},{"ids":[82383,153298],"train_count":873},{"ids":[158401,148390,151828,7105],"train_count":291},{"ids":[151982,8789,41635,151883],"train_count":290},{"ids":[2175,42565],"train_count":869},{"ids":[42395,28696,1739],"train_count":434},{"ids":[10064,16763],"train_count":867},{"ids":[194874,150095],"train_count":865},{"ids":[148337,148489,18372,15630],"train_count":287},{"ids":[2115,26125],"train_count":860},{"ids":[17050,148318],"train_count":859},{"ids":[149746,8789,36473],"train_count":429},{"ids":[15630,53887,57868,65360],"train_count":285},{"ids":[87018,21785],"train_count":854},{"ids":[7721,2115],"train_count":853},{"ids":[42565,5301],"train_count":850},{"ids":[95169,152590,160195],"train_count":425},{"ids":[4901,93834,12666],"train_count":422},{"ids":[18727,149095,149827],"train_count":422},{"ids":[14666,45679,6389],"train_count":420},{"ids":[149167,42565],"train_count":839},{"ids":[5241,149147],"train_count":838},{"ids":[18372,14942],"train_count":838},{"ids":[32868,148561,16806,15630],"train_count":279},{"ids":[52337,64257],"train_count":836},{"ids":[7105,1729],"train_count":833},{"ids":[50806,13603],"train_count":833},{"ids":[18768,148292,149452],"train_count":415},{"ids":[149095,149827,13292],"train_count":414},{"ids":[156427,12014],"train_count":828},{"ids":[19774,65131,148283],"train_count":412},{"ids":[32884,77779,167596],"train_count":412},{"ids":[26607,18727],"train_count":822},{"ids":[148275,15481],"train_count":821},{"ids":[5301,13302,151397],"train_count":410},{"ids":[148337,148489,18372],"train_count":410},{"ids":[186747,148634,9080],"train_count":409},{"ids":[10161,8789,16806,5092],"train_count":271},{"ids":[155113,231736],"train_count":809},{"ids":[149282,36621,9080],"train_count":403},{"ids":[16362,148294],"train_count":804},{"ids":[16362,154822,8789,16806],"train_count":268},{"ids":[77779,26125,32401],"train_count":402},{"ids":[5624,42317,163406],"train_count":401},{"ids":[155113,231736,7921],"train_count":401},{"ids":[65445,1739,28376,16806],"train_count":266},{"ids":[13113,2175,42565,21441],"train_count":265},{"ids":[68553,40794,5092],"train_count":397},{"ids":[10161,8789,16806],"train_count":396},{"ids":[153020,148276],"train_count":790},{"ids":[5624,23514],"train_count":788},{"ids":[36771,45480],"train_count":788},{"ids":[148489,18372,15630],"train_count":394},{"ids":[6240,26046],"train_count":780},{"ids":[7754,148276],"train_count":780},{"ids":[154822,8789,16806,5092],"train_count":260},{"ids":[59627,71658],"train_count":775},{"ids":[9080,77969],"train_count":773},{"ids":[11777,42565],"train_count":772},{"ids":[26125,39427,44633],"train_count":386},{"ids":[152377,53887,21785],"train_count":385},{"ids":[26149,47745,10161],"train_count":384},{"ids":[38859,5301,162292],"train_count":384},{"ids":[47745,5301],"train_count":768},{"ids":[155782,150704,32884,77779],"train_count":256},{"ids":[26149,44432,162292],"train_count":383},{"ids":[64611,153051],"train_count":764},{"ids":[151755,77436,149537],"train_count":382},{"ids":[7721,77969],"train_count":763},{"ids":[7958,159881,69729,6240],"train_count":254},{"ids":[18727,22078,148276,150528],"train_count":254},{"ids":[20177,2040,54025],"train_count":381},{"ids":[151290,148489,15630,53887],"train_count":254},{"ids":[149269,8789],"train_count":760},{"ids":[23355,151290,148489,15630],"train_count":253},{"ids":[149167,162292],"train_count":758},{"ids":[149413,156211,10064],"train_count":379},{"ids":[8789,54025,42565],"train_count":378},{"ids":[9080,77436,150621],"train_count":378},{"ids":[33422,153056],"train_count":753},{"ids":[156211,10064,16763,44633],"train_count":251},{"ids":[155260,15630,42565],"train_count":376},{"ids":[15630,53887,57868],"train_count":373},{"ids":[158401,148390,151828],"train_count":373},{"ids":[54025,5092],"train_count":744},{"ids":[156242,64257,198326],"train_count":372},{"ids":[148489,18372],"train_count":743},{"ids":[148367,38183],"train_count":742},{"ids":[65445,1739],"train_count":741},{"ids":[154158,148293],"train_count":741},{"ids":[47745,159380],"train_count":738},{"ids":[50806,17417,2941,54025],"train_count":246},{"ids":[148518,2175],"train_count":738},{"ids":[149501,14666,45679],"train_count":369},{"ids":[26149,44432],"train_count":736},{"ids":[148294,9080,77436],"train_count":368},{"ids":[2941,54025,5092,32401],"train_count":245},{"ids":[61628,12014],"train_count":733},{"ids":[148721,6240,7721],"train_count":366},{"ids":[35884,148367],"train_count":730},{"ids":[149746,8789,77436],"train_count":365},{"ids":[21445,28696],"train_count":726},{"ids":[2941,54025],"train_count":725},{"ids":[22220,64257,225639],"train_count":362},{"ids":[152911,5092],"train_count":724},{"ids":[32884,77779,26125,45480],"train_count":241},{"ids":[7105,149800],"train_count":721},{"ids":[153944,151011],"train_count":721},{"ids":[77436,153199],"train_count":720},{"ids":[176215,148275],"train_count":720},{"ids":[150387,33422,15481],"train_count":358},{"ids":[150530,187942],"train_count":716},{"ids":[152211,159759],"train_count":715},{"ids":[6733,57868],"train_count":714},{"ids":[18727,32392],"train_count":714},{"ids":[64535,25497],"train_count":714},{"ids":[157695,47552,28696],"train_count":357},{"ids":[47552,28696,45527],"train_count":356},{"ids":[16763,4669],"train_count":708},{"ids":[81573,148510,149055],"train_count":354},{"ids":[22346,6733,57868],"train_count":353},{"ids":[32884,77779,26125,149891],"train_count":235},{"ids":[151639,7754,148276,37835],"train_count":235},{"ids":[161420,42565],"train_count":705},{"ids":[156476,148454,200154],"train_count":352},{"ids":[148489,15630,53887],"train_count":351},{"ids":[149167,162292,12751],"train_count":351},{"ids":[157391,163687],"train_count":702},{"ids":[12666,4403],"train_count":700},{"ids":[35884,151709],"train_count":699},{"ids":[7422,2115],"train_count":698},{"ids":[45679,6389,158638],"train_count":348},{"ids":[64535,42565,186408],"train_count":348},{"ids":[149147,16763,32884],"train_count":348},{"ids":[23947,16806],"train_count":695},{"ids":[150423,215753],"train_count":694},{"ids":[193863,149562],"train_count":693},{"ids":[7105,149193],"train_count":692},{"ids":[34279,148294,12666],"train_count":346},{"ids":[150580,201647,7721],"train_count":346},{"ids":[22220,23514,36103,148317],"train_count":230},{"ids":[150175,151203],"train_count":689},{"ids":[219874,29196],"train_count":689},{"ids":[55055,149454,150140],"train_count":344},{"ids":[61326,148292,16763],"train_count":344},{"ids":[218394,5092,13292],"train_count":344},{"ids":[150194,16467,13113,2175],"train_count":229},{"ids":[182670,150198],"train_count":685},{"ids":[5092,41546],"train_count":684},{"ids":[149269,8789,54025,42565],"train_count":228},{"ids":[35884,15481,148579,4096],"train_count":227},{"ids":[10161,149926],"train_count":680},{"ids":[22220,64257],"train_count":680},{"ids":[45679,6389],"train_count":680},{"ids":[159881,48623],"train_count":680},{"ids":[64535,42565],"train_count":678},{"ids":[161446,14942],"train_count":676},{"ids":[64535,42565,186408,31889],"train_count":225},{"ids":[148604,148303],"train_count":674},{"ids":[38241,53062],"train_count":673},{"ids":[2115,149097,16806,6733],"train_count":224},{"ids":[13659,148518,2175],"train_count":336},{"ids":[17417,2941,54025],"train_count":336},{"ids":[26607,148318,177567],"train_count":335},{"ids":[163961,159999],"train_count":670},{"ids":[41635,3652,27112,155066],"train_count":223},{"ids":[77436,148558],"train_count":668},{"ids":[148634,9080,48623],"train_count":334},{"ids":[14942,77436],"train_count":667},{"ids":[13659,148518,2175,156204],"train_count":222},{"ids":[18727,11577,159032],"train_count":333},{"ids":[148277,26125,150366,19937],"train_count":222},{"ids":[156242,64257],"train_count":665},{"ids":[220449,32392],"train_count":665},{"ids":[16797,58361,148275],"train_count":332},{"ids":[5241,21441],"train_count":663},{"ids":[10983,41635,3652,27112],"train_count":221},{"ids":[7422,2115,34981],"train_count":330},{"ids":[77779,26125,42308],"train_count":330},{"ids":[2941,54025,5092],"train_count":329},{"ids":[59627,71658,54201],"train_count":329},{"ids":[4901,215194,182503,15630],"train_count":219},{"ids":[7347,151744,7721,26125],"train_count":219},{"ids":[8789,54025,42565,170741],"train_count":219},{"ids":[20177,2040,54025,149637],"train_count":219},{"ids":[32392,148277,26125,150366],"train_count":219},{"ids":[41381,7358,41635,45679],"train_count":219},{"ids":[148343,185709],"train_count":657},{"ids":[231238,220449],"train_count":656},{"ids":[26149,21785],"train_count":655},{"ids":[33638,36473],"train_count":655},{"ids":[22220,23514],"train_count":654},{"ids":[26125,39427],"train_count":649},{"ids":[149147,16763],"train_count":649},{"ids":[160847,170946],"train_count":649},{"ids":[210138,150704,32884],"train_count":324},{"ids":[68553,40794],"train_count":647},{"ids":[5241,54724,149591],"train_count":323},{"ids":[53887,21785],"train_count":646},{"ids":[4901,151853],"train_count":645},{"ids":[151706,14942,47745,159380],"train_count":215},{"ids":[153589,2115,149097,16806],"train_count":215},{"ids":[8789,41635],"train_count":642},{"ids":[149560,34981,148275,7105],"train_count":214},{"ids":[222663,54025,5092,32401],"train_count":214},{"ids":[18768,148292],"train_count":641},{"ids":[151001,63046],"train_count":641},{"ids":[21720,150396,2115],"train_count":320},{"ids":[160259,32884,2115],"train_count":320},{"ids":[161420,42565,170741],"train_count":320},{"ids":[157695,1450,148378,14942],"train_count":213},{"ids":[148874,11577,159541],"train_count":319},{"ids":[20177,66951],"train_count":637},{"ids":[151980,14942,5301,32884],"train_count":212},{"ids":[154245,6240,150849,14942],"train_count":212},{"ids":[42565,171096],"train_count":634},{"ids":[15481,7358],"train_count":632},{"ids":[151755,77436],"train_count":632},{"ids":[20289,159489,151639],"train_count":315},{"ids":[40794,5092],"train_count":628},{"ids":[53555,160259,32884],"train_count":314},{"ids":[64257,148706],"train_count":628},{"ids":[225678,155016],"train_count":627},{"ids":[7105,148317],"train_count":626},{"ids":[7958,48941,150937],"train_count":313},{"ids":[148275,1403],"train_count":626},{"ids":[23355,151438,181923],"train_count":312},{"ids":[58297,53555,160259],"train_count":312},{"ids":[179837,6733,40794],"train_count":312},{"ids":[148292,15481],"train_count":623},{"ids":[148561,16806,15630],"train_count":311},{"ids":[8789,16806,5092,41546],"train_count":207},{"ids":[61326,148464,149037,1739],"train_count":207},{"ids":[33422,15481,19937],"train_count":310},{"ids":[150828,42317],"train_count":620},{"ids":[150424,18768],"train_count":619},{"ids":[4640,34981],"train_count":618},{"ids":[17050,2941,150475],"train_count":309},{"ids":[5092,13292],"train_count":617},{"ids":[34279,148294,12666,148293],"train_count":205},{"ids":[152134,150947,7105,1729],"train_count":205},{"ids":[32724,10161],"train_count":614},{"ids":[149009,18768],"train_count":614},{"ids":[42886,7721],"train_count":613},{"ids":[162593,152974],"train_count":613},{"ids":[16806,5092],"train_count":612},{"ids":[149560,34981,148275],"train_count":306},{"ids":[148294,9080],"train_count":611},{"ids":[9629,53555],"train_count":609},{"ids":[26149,47745,10161,15481],"train_count":203},{"ids":[34208,148646],"train_count":609},{"ids":[18372,149800],"train_count":608},{"ids":[64535,30206,3652],"train_count":304},{"ids":[186643,149537],"train_count":608},{"ids":[32884,2115],"train_count":606},{"ids":[148303,158559],"train_count":605},{"ids":[150455,2175,153922],"train_count":302},{"ids":[33207,204086],"train_count":602},{"ids":[182670,67494],"train_count":601},{"ids":[26125,150366,19937],"train_count":300},{"ids":[155164,16806,45587],"train_count":300},{"ids":[5092,42240],"train_count":599},{"ids":[45587,7358,200641],"train_count":299},{"ids":[10983,9080,148283,148634],"train_count":199},{"ids":[14942,14942],"train_count":597},{"ids":[57272,14942,45768],"train_count":298},{"ids":[148390,151828,7105],"train_count":298},{"ids":[148464,6733],"train_count":595},{"ids":[54858,158401,148390,151828],"train_count":198},{"ids":[6240,84706],"train_count":593},{"ids":[18372,1739],"train_count":593},{"ids":[46807,148277],"train_count":592},{"ids":[15481,148579,4096],"train_count":295},{"ids":[46145,149208,168640],"train_count":295},{"ids":[151283,10064,149095],"train_count":295},{"ids":[7958,159881,69729],"train_count":294},{"ids":[149156,154299,3652],"train_count":294},{"ids":[150947,7105,1729],"train_count":294},{"ids":[7721,77436],"train_count":587},{"ids":[237856,42565],"train_count":587},{"ids":[2175,42565,21441],"train_count":293},{"ids":[234875,148687],"train_count":586},{"ids":[5301,23514],"train_count":585},{"ids":[149191,28696],"train_count":582},{"ids":[42565,21441],"train_count":581},{"ids":[8789,41635,151883],"train_count":290},{"ids":[59627,71658,68335],"train_count":290},{"ids":[148309,12751],"train_count":580},{"ids":[155113,231736,149005],"train_count":290},{"ids":[173286,87018,21785],"train_count":290},{"ids":[181622,148343,185709],"train_count":290},{"ids":[32884,4403],"train_count":579},{"ids":[149269,8789,54025,16669],"train_count":193},{"ids":[158722,12666],"train_count":579},{"ids":[215471,163045,6733,149591],"train_count":193},{"ids":[26607,23355,203141],"train_count":289},{"ids":[243458,66050],"train_count":578},{"ids":[23514,7105],"train_count":577},{"ids":[36103,42308],"train_count":576},{"ids":[148294,12666],"train_count":576},{"ids":[149562,41546],"train_count":576},{"ids":[149591,15481],"train_count":576},{"ids":[159881,69729,6240],"train_count":288},{"ids":[216401,13302,150559],"train_count":288},{"ids":[232069,11577,21797,177702],"train_count":192},{"ids":[52687,148700],"train_count":574},{"ids":[70345,148281,148686],"train_count":287},{"ids":[11929,12666],"train_count":573},{"ids":[14942,61324,7358,2040],"train_count":191},{"ids":[148277,154838],"train_count":573},{"ids":[54858,19582],"train_count":572},{"ids":[54858,19582,16362],"train_count":286},{"ids":[26125,1424],"train_count":571},{"ids":[16763,6733],"train_count":570},{"ids":[47552,28696,45527,7754],"train_count":190},{"ids":[53887,57868,65360],"train_count":285},{"ids":[151709,181923],"train_count":570},{"ids":[154245,6240,150849],"train_count":285},{"ids":[69134,2115,34981],"train_count":284},{"ids":[176213,149591,31889],"train_count":284},{"ids":[167797,7721,77969,6240],"train_count":189},{"ids":[158276,61057],"train_count":566},{"ids":[5301,66951],"train_count":564},{"ids":[7422,18372],"train_count":564},{"ids":[12666,148293],"train_count":564},{"ids":[156211,10064],"train_count":564},{"ids":[26125,32401],"train_count":563},{"ids":[4901,15481],"train_count":562},{"ids":[209104,5301,13302],"train_count":281},{"ids":[160847,170946,32401],"train_count":280},{"ids":[17160,10161,8789,16806],"train_count":186},{"ids":[32868,148561,16806],"train_count":279},{"ids":[151509,87148,33422],"train_count":279},{"ids":[148275,149800],"train_count":557},{"ids":[150438,68335],"train_count":556},{"ids":[215471,163045,6733],"train_count":278},{"ids":[53062,32401],"train_count":555},{"ids":[77436,155297],"train_count":555},{"ids":[153031,153009],"train_count":555},{"ids":[157801,15481],"train_count":555},{"ids":[61326,148464,149037],"train_count":277},{"ids":[215194,182503,15630],"train_count":277},{"ids":[150366,19937],"train_count":552},{"ids":[222663,54025,5092],"train_count":275},{"ids":[5624,23514,36103,148317],"train_count":183},{"ids":[17160,184088],"train_count":549},{"ids":[32884,77779,26125,148325],"train_count":183},{"ids":[149097,16806,6733,18110],"train_count":183},{"ids":[65131,148283],"train_count":548},{"ids":[15481,4403],"train_count":547},{"ids":[151183,2175],"train_count":547},{"ids":[36771,148540],"train_count":546},{"ids":[15630,69729],"train_count":545},{"ids":[24653,148380],"train_count":545},{"ids":[151755,36473],"train_count":545},{"ids":[7754,148276,37835],"train_count":272},{"ids":[149560,167664],"train_count":544},{"ids":[7105,42308],"train_count":543},{"ids":[18372,150194,16467,13113],"train_count":181},{"ids":[148275,60038],"train_count":541},{"ids":[1739,28376,16806],"train_count":270},{"ids":[42565,42074],"train_count":540},{"ids":[150919,15481,148301],"train_count":270},{"ids":[151078,154731,15481],"train_count":270},{"ids":[14942,5301,32884],"train_count":269},{"ids":[148294,12666,148293],"train_count":269},{"ids":[151203,1739],"train_count":537},{"ids":[16362,154822,8789],"train_count":268},{"ids":[154822,8789,16806],"train_count":268},{"ids":[7422,18372,148281,149900],"train_count":178},{"ids":[149156,154299],"train_count":534},{"ids":[202973,149800],"train_count":534},{"ids":[65445,1739,28376],"train_count":266},{"ids":[65445,153342,11796],"train_count":266},{"ids":[159277,13302],"train_count":532},{"ids":[7105,4669],"train_count":531},{"ids":[156096,149280],"train_count":531},{"ids":[1403,148524],"train_count":530},{"ids":[26149,47745],"train_count":529},{"ids":[31889,1424],"train_count":529},{"ids":[2941,8789],"train_count":528},{"ids":[149054,10161,8789,16806],"train_count":176},{"ids":[149191,15630,69729,149137],"train_count":176},{"ids":[154424,45587,1739,149352],"train_count":176},{"ids":[177220,18768,34981,26125],"train_count":176},{"ids":[216401,13302,150559,44633],"train_count":176},{"ids":[232069,11577,21797],"train_count":263},{"ids":[50816,7105],"train_count":525},{"ids":[7105,149280],"train_count":524},{"ids":[18372,149280],"train_count":524},{"ids":[181044,6389,44633],"train_count":262},{"ids":[204086,150584],"train_count":524},{"ids":[152153,199375],"train_count":523},{"ids":[7422,148292],"train_count":522},{"ids":[150919,29119,16467,156204],"train_count":174},{"ids":[54724,149591],"train_count":520},{"ids":[151980,14942,5301],"train_count":260},{"ids":[220083,2040],"train_count":520},{"ids":[61324,7358,2040,55389],"train_count":173},{"ids":[95169,152590],"train_count":519},{"ids":[201647,7721],"train_count":519},{"ids":[5301,23514,36103],"train_count":259},{"ids":[155782,150704,32884],"train_count":259},{"ids":[36103,32401],"train_count":517},{"ids":[149085,7105],"train_count":517},{"ids":[7105,1450],"train_count":516},{"ids":[7422,18372,150194,16467],"train_count":172},{"ids":[41635,3652,27112],"train_count":258},{"ids":[50806,17417,2941],"train_count":258},{"ids":[148489,18372,15630,57868],"train_count":172},{"ids":[149233,148288,18372],"train_count":258},{"ids":[222663,91206,1739],"train_count":258},{"ids":[44093,159859],"train_count":515},{"ids":[151982,8789],"train_count":515},{"ids":[10983,23514],"train_count":514},{"ids":[34208,148646,7105],"train_count":257},{"ids":[152134,150947,7105],"train_count":257},{"ids":[42395,148700,68608],"train_count":256},{"ids":[189726,56916],"train_count":512},{"ids":[69729,6240],"train_count":511},{"ids":[148687,150006],"train_count":511},{"ids":[4901,215194,182503],"train_count":255},{"ids":[18727,22078,148276],"train_count":255},{"ids":[56755,64257,150686],"train_count":255},{"ids":[86949,85208,14942,36771],"train_count":170},{"ids":[1424,10161],"train_count":509},{"ids":[218574,171251],"train_count":509},{"ids":[22078,148276,150528],"train_count":254},{"ids":[151290,148489,15630],"train_count":254},{"ids":[154245,6240],"train_count":508},{"ids":[187134,47745],"train_count":508},{"ids":[26149,5092],"train_count":507},{"ids":[9080,7921],"train_count":506},{"ids":[23355,151290,148489],"train_count":253},{"ids":[64535,29119,186983],"train_count":253},{"ids":[149052,149586,148275],"train_count":253},{"ids":[149589,7105,13113],"train_count":253},{"ids":[186361,149208,55389],"train_count":253},{"ids":[19774,65131],"train_count":504},{"ids":[26607,148318,177567,37835],"train_count":168},{"ids":[33422,15481,152993],"train_count":252},{"ids":[148275,48933],"train_count":503},{"ids":[10064,16763,44633],"train_count":251},{"ids":[33422,15481,95306],"train_count":251},{"ids":[173518,45405],"train_count":502},{"ids":[10161,8789],"train_count":501},{"ids":[7958,10161,12666],"train_count":250},{"ids":[13659,148518],"train_count":500},{"ids":[149430,158693,152644],"train_count":250},{"ids":[163542,10161],"train_count":499},{"ids":[227028,54201],"train_count":499},{"ids":[58427,7721,155812,149280],"train_count":166},{"ids":[149167,162292,50800],"train_count":249},{"ids":[157801,15481,19937],"train_count":248},{"ids":[206694,7358,92647],"train_count":248},{"ids":[46807,31889,16763,6733],"train_count":165},{"ids":[151639,7754],"train_count":495},{"ids":[12666,149086],"train_count":494},{"ids":[10983,36473],"train_count":493},{"ids":[16763,44633],"train_count":493},{"ids":[9080,77436],"train_count":492},{"ids":[148378,14942,77436,149537],"train_count":164},{"ids":[231736,7921],"train_count":492},{"ids":[38859,5301],"train_count":491},{"ids":[61326,149454,7105],"train_count":245},{"ids":[157695,1450,148378],"train_count":245},{"ids":[159859,149665],"train_count":490},{"ids":[17050,2941],"train_count":489},{"ids":[201553,9080,77969,6240],"train_count":163},{"ids":[10983,9080,148283],"train_count":244},{"ids":[13532,157279],"train_count":488},{"ids":[15481,148275],"train_count":488},{"ids":[26149,5092,22078],"train_count":244},{"ids":[39427,44633],"train_count":488},{"ids":[65445,149389,159497],"train_count":244},{"ids":[153682,47745],"train_count":488},{"ids":[156125,198071],"train_count":488},{"ids":[162593,152974,158525],"train_count":244},{"ids":[34981,21441],"train_count":486},{"ids":[46807,31889,16763],"train_count":243},{"ids":[189870,7358,178595,26125],"train_count":162},{"ids":[5301,149800],"train_count":485},{"ids":[14942,83147],"train_count":485},{"ids":[20177,66951,197362],"train_count":242},{"ids":[24653,7721,18372],"train_count":242},{"ids":[148292,16763],"train_count":484},{"ids":[148378,14942,151308],"train_count":242},{"ids":[61326,148292,16763,37835],"train_count":161},{"ids":[149430,158693],"train_count":483},{"ids":[157089,148336],"train_count":483},{"ids":[7358,41635,45679],"train_count":241},{"ids":[11777,5092],"train_count":482},{"ids":[11929,21785,149595],"train_count":241},{"ids":[34981,148275],"train_count":482},{"ids":[42317,221196],"train_count":482},{"ids":[77779,26125,45480],"train_count":241},{"ids":[151211,149633],"train_count":482},{"ids":[190964,29119],"train_count":482},{"ids":[5301,148280,148277],"train_count":240},{"ids":[13659,48989],"train_count":480},{"ids":[45587,7358],"train_count":480},{"ids":[155609,149572,31889],"train_count":240},{"ids":[162593,152974,158525,31889],"train_count":160},{"ids":[5301,55389,44633],"train_count":239},{"ids":[5301,162292],"train_count":478},{"ids":[5624,148606],"train_count":478},{"ids":[5624,148757],"train_count":478},{"ids":[10161,12666],"train_count":478},{"ids":[149900,87018,21785],"train_count":239},{"ids":[159489,151639],"train_count":478},{"ids":[178260,171664],"train_count":478},{"ids":[220204,39427,148277],"train_count":239},{"ids":[149810,16467,13113,2175],"train_count":159},{"ids":[5241,34981],"train_count":476},{"ids":[22078,150800],"train_count":476},{"ids":[33638,45587,7358],"train_count":238},{"ids":[39427,6240],"train_count":476},{"ids":[8789,4403],"train_count":475},{"ids":[23947,61628],"train_count":475},{"ids":[36103,42240],"train_count":475},{"ids":[153437,148336],"train_count":475},{"ids":[11577,21797,177702],"train_count":237},{"ids":[58427,7721,155812],"train_count":237},{"ids":[164018,5301,55389],"train_count":237},{"ids":[45587,4403],"train_count":473},{"ids":[226673,83895],"train_count":473},{"ids":[1450,148378,14942],"train_count":236},{"ids":[8789,54025,16669],"train_count":236},{"ids":[13302,151397],"train_count":472},{"ids":[28376,16806,148317],"train_count":236},{"ids":[34981,148275,7105],"train_count":236},{"ids":[41381,22078,59627],"train_count":236},{"ids":[148397,148414],"train_count":472},{"ids":[151029,6240,176249],"train_count":236},{"ids":[52687,156211,10064,16763],"train_count":157},{"ids":[149233,148288,18372,25767],"train_count":157},{"ids":[7105,148325],"train_count":470},{"ids":[56287,154767,15630],"train_count":235},{"ids":[77779,26125,149891],"train_count":235},{"ids":[150917,13302,10161],"train_count":235},{"ids":[163347,18768],"train_count":470},{"ids":[58361,148288],"train_count":469},{"ids":[155260,15630],"train_count":469},{"ids":[16806,6733,18110],"train_count":234},{"ids":[33638,45587,7358,200641],"train_count":156},{"ids":[70667,158121],"train_count":468},{"ids":[149097,16806,6733],"train_count":234},{"ids":[14942,47745],"train_count":466},{"ids":[53062,148317],"train_count":466},{"ids":[149282,36621],"train_count":466},{"ids":[149430,158693,36183],"train_count":233},{"ids":[19774,158401,148390,151828],"train_count":155},{"ids":[151029,6240],"train_count":465},{"ids":[225833,16763],"train_count":465},{"ids":[20289,2175,12751],"train_count":232},{"ids":[77655,148660,16763],"train_count":232},{"ids":[89030,150662,12751],"train_count":232},{"ids":[149191,15630],"train_count":464},{"ids":[149827,13292],"train_count":464},{"ids":[153944,151011,82942],"train_count":232},{"ids":[33422,45679],"train_count":463},{"ids":[22220,64257,225639,6240],"train_count":154},{"ids":[26125,151642],"train_count":462},{"ids":[53062,42240],"train_count":462},{"ids":[148314,155812],"train_count":462},{"ids":[150007,8789,7921],"train_count":231},{"ids":[21720,15481],"train_count":461},{"ids":[151216,32401],"train_count":461},{"ids":[18727,148281,149885],"train_count":230},{"ids":[58427,148294],"train_count":460},{"ids":[150194,16467,13113],"train_count":230},{"ids":[174856,152141],"train_count":460},{"ids":[26125,158668],"train_count":459},{"ids":[33065,6733],"train_count":459},{"ids":[33638,148280,77969,5301],"train_count":153},{"ids":[148301,2175,36473,7105],"train_count":153},{"ids":[150714,15481],"train_count":459},{"ids":[175447,198326],"train_count":459},{"ids":[4901,93834],"train_count":458},{"ids":[18727,148275,148801],"train_count":229},{"ids":[26269,6240],"train_count":458},{"ids":[53887,57868],"train_count":458},{"ids":[149233,148464],"train_count":458},{"ids":[153944,151011,1729],"train_count":229},{"ids":[160149,4640],"train_count":458},{"ids":[225833,16763,4669],"train_count":229},{"ids":[156715,149280],"train_count":457},{"ids":[16763,37835],"train_count":456},{"ids":[148454,200154],"train_count":456},{"ids":[150387,33422,15481,152993],"train_count":152},{"ids":[164018,5301,55389,44633],"train_count":152},{"ids":[166636,12666,149086],"train_count":228},{"ids":[14942,45768],"train_count":454},{"ids":[22078,148569],"train_count":454},{"ids":[35884,15481,148579],"train_count":227},{"ids":[76953,148606,57868],"train_count":227},{"ids":[148318,177567],"train_count":454},{"ids":[202020,2941,24623],"train_count":227},{"ids":[6733,1729],"train_count":453},{"ids":[26125,150366],"train_count":453},{"ids":[149589,7105,13113,150800],"train_count":151},{"ids":[36621,9080],"train_count":452},{"ids":[42565,186408,31889],"train_count":226},{"ids":[58297,53555],"train_count":452},{"ids":[151744,7721,26125],"train_count":226},{"ids":[163045,6733,149591],"train_count":226},{"ids":[182581,149637],"train_count":452},{"ids":[14666,45679],"train_count":451},{"ids":[8789,36473],"train_count":450},{"ids":[13302,10161],"train_count":450},{"ids":[14942,77436,149537],"train_count":225},{"ids":[33638,150853,61757],"train_count":225},{"ids":[34208,148292,148330],"train_count":225},{"ids":[53062,41546],"train_count":450},{"ids":[61324,7358,2040],"train_count":225},{"ids":[64535,29119,186983,10161],"train_count":150},{"ids":[64611,153051,13302],"train_count":225},{"ids":[148518,2175,42565,36747],"train_count":150},{"ids":[148518,2175,156204],"train_count":225},{"ids":[149233,148464,6733,194972],"train_count":150},{"ids":[149501,14666],"train_count":450},{"ids":[151706,14942],"train_count":450},{"ids":[151744,7721,26125,1450],"train_count":150},{"ids":[35884,149345],"train_count":449},{"ids":[181931,6389],"train_count":449},{"ids":[2115,149097,16806],"train_count":224},{"ids":[63081,1739],"train_count":448},{"ids":[149560,167664,149800],"train_count":224},{"ids":[11929,21785],"train_count":447},{"ids":[64535,30206,3652,149383],"train_count":149},{"ids":[156476,148454],"train_count":447},{"ids":[3652,27112,155066],"train_count":223},{"ids":[32392,148277,26125],"train_count":223},{"ids":[148277,26125,150366],"train_count":223},{"ids":[155164,16806],"train_count":446},{"ids":[5301,32884],"train_count":444},{"ids":[5301,55389],"train_count":444},{"ids":[52687,148700,149800],"train_count":222},{"ids":[3652,27112],"train_count":443},{"ids":[2040,54025,149637],"train_count":221},{"ids":[6733,57868,148593],"train_count":221},{"ids":[10983,41635,3652],"train_count":221},{"ids":[7550,26125,158668,16806],"train_count":147},{"ids":[11929,12666,12666,45768],"train_count":147},{"ids":[189962,149800],"train_count":441},{"ids":[209104,5301,13302,2941],"train_count":147},{"ids":[7347,151744,7721],"train_count":220},{"ids":[39243,150813,149665],"train_count":220},{"ids":[148518,2175,42565],"train_count":220},{"ids":[19648,151446],"train_count":439},{"ids":[225993,2941],"train_count":439},{"ids":[7550,26125,39427,37835],"train_count":146},{"ids":[7721,148288],"train_count":438},{"ids":[13659,148464,64257],"train_count":219},{"ids":[41381,7358,41635],"train_count":219},{"ids":[54025,42565,170741],"train_count":219},{"ids":[59627,1739,14666],"train_count":219},{"ids":[81573,148510],"train_count":438},{"ids":[148464,6733,194972],"train_count":219},{"ids":[150926,151032,42565],"train_count":219},{"ids":[154297,1403,148524],"train_count":219},{"ids":[154729,5092],"train_count":437},{"ids":[152973,77436],"train_count":436},{"ids":[4901,151853,148378,14942],"train_count":145},{"ids":[7358,42387],"train_count":435},{"ids":[58297,148518,2175,42565],"train_count":145},{"ids":[148876,26125],"train_count":435},{"ids":[151216,47474,36771,67494],"train_count":145},{"ids":[217038,7754,148292,148569],"train_count":145},{"ids":[9080,148283,148634],"train_count":217},{"ids":[18768,34981,26125],"train_count":217},{"ids":[93834,12666],"train_count":434},{"ids":[150423,153342,4096],"train_count":217},{"ids":[181931,165328,42387],"train_count":217},{"ids":[68608,42387],"train_count":433},{"ids":[6240,150849,14942],"train_count":216},{"ids":[14942,47745,159380],"train_count":216},{"ids":[18727,149095],"train_count":432},{"ids":[28333,11777],"train_count":432},{"ids":[32392,148390,16806],"train_count":216},{"ids":[77655,6733],"train_count":432},{"ids":[150455,2175,153922,150451],"train_count":144},{"ids":[15729,150294],"train_count":431},{"ids":[26125,42308],"train_count":431},{"ids":[153049,42565],"train_count":431},{"ids":[154726,150156],"train_count":431},{"ids":[5092,45527],"train_count":430},{"ids":[9629,11777],"train_count":430},{"ids":[153589,2115,149097],"train_count":215},{"ids":[171132,148325],"train_count":430},{"ids":[7721,18372],"train_count":429},{"ids":[8789,77436],"train_count":429},{"ids":[34960,2040],"train_count":429},{"ids":[82929,148294,152134,245644],"train_count":143},{"ids":[153024,61057],"train_count":429},{"ids":[18372,15630],"train_count":428},{"ids":[52687,74990],"train_count":428},{"ids":[57632,59627,71658],"train_count":214},{"ids":[148303,1450,18768],"train_count":214},{"ids":[150919,29119,16467],"train_count":214},{"ids":[6389,158638],"train_count":427},{"ids":[149095,149827],"train_count":427},{"ids":[151223,74990],"train_count":427},{"ids":[152590,160195],"train_count":427},{"ids":[6240,7721,148281,7721],"train_count":142},{"ids":[7754,23514,7105,148275],"train_count":142},{"ids":[17050,148318,155916],"train_count":213},{"ids":[20177,2040],"train_count":426},{"ids":[25497,62709,3652,21441],"train_count":142},{"ids":[29189,161308,149665],"train_count":213},{"ids":[148277,4403],"train_count":426},{"ids":[149591,7105,148317],"train_count":213},{"ids":[5241,150800],"train_count":425},{"ids":[187589,7721],"train_count":425},{"ids":[7550,26125,158668,71221],"train_count":141},{"ids":[25497,2941,27473,59627],"train_count":141},{"ids":[171011,28696],"train_count":423},{"ids":[148378,14942,168329],"train_count":211},{"ids":[151988,21918],"train_count":422},{"ids":[193330,15630],"train_count":422},{"ids":[162125,181841],"train_count":421},{"ids":[181931,165328],"train_count":421},{"ids":[2941,27473,59627,71658],"train_count":140},{"ids":[7721,32884],"train_count":420},{"ids":[9629,53555,148369],"train_count":210},{"ids":[12751,2941],"train_count":420},{"ids":[14942,83147,4403],"train_count":210},{"ids":[32884,77779,26125,148540],"train_count":140},{"ids":[7754,22078],"train_count":419},{"ids":[2175,42565,36747],"train_count":209},{"ids":[23514,7105,148275],"train_count":209},{"ids":[25497,2941,27473],"train_count":209},{"ids":[70667,158121,2040],"train_count":209},{"ids":[148961,148700,185709],"train_count":209},{"ids":[13302,150559],"train_count":417},{"ids":[19774,29119,173919,77779],"train_count":139},{"ids":[36473,2115,8789,148390],"train_count":139},{"ids":[57632,59627,71658,54201],"train_count":139},{"ids":[148275,149280],"train_count":417},{"ids":[42565,36747],"train_count":416},{"ids":[47745,10161,15481],"train_count":208},{"ids":[148292,149452],"train_count":416},{"ids":[148464,149037,1739],"train_count":208},{"ids":[154245,6240,7721],"train_count":208},{"ids":[14942,22078],"train_count":415},{"ids":[4901,4669],"train_count":414},{"ids":[5092,148317],"train_count":414},{"ids":[16806,5092,41546],"train_count":207},{"ids":[18372,149193],"train_count":414},{"ids":[18727,28696,148518],"train_count":207},{"ids":[52337,64257,148324,181923],"train_count":138},{"ids":[52687,74990,148518],"train_count":207},{"ids":[87018,21785,148554],"train_count":207},{"ids":[150735,159035],"train_count":414},{"ids":[36103,41546],"train_count":413},{"ids":[148337,148489],"train_count":413},{"ids":[150580,201647],"train_count":413},{"ids":[5241,150800,177067],"train_count":206},{"ids":[7721,77436,149537],"train_count":206},{"ids":[10161,3652],"train_count":412},{"ids":[34208,148292],"train_count":412},{"ids":[77779,167596],"train_count":412},{"ids":[91206,1424],"train_count":412},{"ids":[149715,68608],"train_count":412},{"ids":[186747,148634],"train_count":412},{"ids":[52337,36183,170946,41546],"train_count":137},{"ids":[70345,148281,148686,154479],"train_count":137},{"ids":[148700,185709],"train_count":411},{"ids":[5624,148757,61057],"train_count":205},{"ids":[9080,1739],"train_count":410},{"ids":[18727,24623,153218],"train_count":205},{"ids":[45587,1739,149352],"train_count":205},{"ids":[47552,5301],"train_count":410},{"ids":[153682,47745,159380],"train_count":205},{"ids":[26607,148318],"train_count":409},{"ids":[148915,152711],"train_count":409},{"ids":[10983,36473,39605],"train_count":204},{"ids":[13659,148464,64257,149800],"train_count":136},{"ids":[23947,167654,32884],"train_count":204},{"ids":[26607,148318,177567,44633],"train_count":136},{"ids":[148276,44633],"train_count":408},{"ids":[152377,53887],"train_count":408},{"ids":[154424,45587,1739],"train_count":204},{"ids":[196219,201647,149351],"train_count":204},{"ids":[11777,11577],"train_count":407},{"ids":[22220,9080],"train_count":407},{"ids":[42317,163406],"train_count":407},{"ids":[148276,37835],"train_count":407},{"ids":[152759,36771],"train_count":407},{"ids":[3652,21441],"train_count":406},{"ids":[6389,77436],"train_count":406},{"ids":[7347,153194,9629],"train_count":203},{"ids":[28696,45527,148414],"train_count":203},{"ids":[32868,33437,155295],"train_count":203},{"ids":[35884,61628,12014],"train_count":203},{"ids":[46145,149208,37835],"train_count":203},{"ids":[48771,27448,1739],"train_count":203},{"ids":[148604,148303,9080],"train_count":203},{"ids":[148873,6389,36916],"train_count":203},{"ids":[151078,154731],"train_count":406},{"ids":[152986,153501,48046],"train_count":203},{"ids":[162683,148317],"train_count":406},{"ids":[9080,148275],"train_count":405},{"ids":[64535,29119],"train_count":405},{"ids":[160149,4640,34981,21441],"train_count":135},{"ids":[165171,6240],"train_count":405},{"ids":[5301,13302,2941],"train_count":202},{"ids":[17160,10161,8789],"train_count":202},{"ids":[64611,153051,27112],"train_count":202},{"ids":[148280,148277],"train_count":404},{"ids":[150599,6240,84706],"train_count":202},{"ids":[151509,87148],"train_count":404},{"ids":[152135,6240,185709],"train_count":202},{"ids":[26125,26046],"train_count":403},{"ids":[148275,148449],"train_count":403},{"ids":[2175,45587],"train_count":402},{"ids":[13302,150559,44633],"train_count":201},{"ids":[14942,45768,7921],"train_count":201},{"ids":[18372,148281,149900],"train_count":201},{"ids":[47745,6240],"train_count":402},{"ids":[61326,148292,16763,44633],"train_count":134},{"ids":[149413,12666,200641,44633],"train_count":134},{"ids":[154731,15481],"train_count":402},{"ids":[162593,152974,42387],"train_count":201},{"ids":[58361,148275],"train_count":401},{"ids":[148874,11577],"train_count":401},{"ids":[5241,21441,150282],"train_count":200},{"ids":[11929,9080,148518],"train_count":200},{"ids":[45587,148275],"train_count":400},{"ids":[64535,42565,163347],"train_count":200},{"ids":[149746,8789,151308],"train_count":200},{"ids":[160877,2941],"train_count":400},{"ids":[165792,149155],"train_count":400},{"ids":[27473,59627,71658,68335],"train_count":133},{"ids":[39427,66050],"train_count":399},{"ids":[42565,170968],"train_count":399},{"ids":[151509,87148,33422,15481],"train_count":133},{"ids":[153306,14942,22078,150800],"train_count":133},{"ids":[154521,148275,7105,21918],"train_count":133},{"ids":[7958,10161],"train_count":398},{"ids":[47745,10161],"train_count":398},{"ids":[53062,148325],"train_count":398},{"ids":[64257,225639],"train_count":398},{"ids":[64535,2941,8789],"train_count":199},{"ids":[9080,148283,148634,9080],"train_count":132},{"ids":[48629,16763],"train_count":396},{"ids":[54858,158401,148390],"train_count":198},{"ids":[65445,1739,51788],"train_count":198},{"ids":[148427,148336],"train_count":396},{"ids":[148873,6389],"train_count":396},{"ids":[149156,154299,4403],"train_count":198},{"ids":[152024,42565,171158],"train_count":198},{"ids":[155726,61057,152993],"train_count":198},{"ids":[226013,158668,16806],"train_count":198},{"ids":[7422,7721],"train_count":395},{"ids":[26149,34981],"train_count":395},{"ids":[26607,5301],"train_count":395},{"ids":[5092,42308],"train_count":394},{"ids":[15630,42565],"train_count":394},{"ids":[77436,155461],"train_count":394},{"ids":[77655,211593,44633],"train_count":197},{"ids":[90202,1424],"train_count":394},{"ids":[148275,149957],"train_count":394},{"ids":[149644,148336],"train_count":394},{"ids":[152786,151292,157647],"train_count":197},{"ids":[158668,16806],"train_count":394},{"ids":[13113,2175,42565,149633],"train_count":131},{"ids":[148292,15481,7358,42387],"train_count":131},{"ids":[150834,9080,77969,6240],"train_count":131},{"ids":[15630,148275],"train_count":392},{"ids":[148286,33065,95525],"train_count":196},{"ids":[148303,1450],"train_count":392},{"ids":[148309,54724],"train_count":392},{"ids":[149413,156211],"train_count":392},{"ids":[152959,150187,6240],"train_count":196},{"ids":[237856,42565,42074],"train_count":196},{"ids":[23355,148445,36183,149351],"train_count":130},{"ids":[26125,15481],"train_count":390},{"ids":[42565,171935],"train_count":390},{"ids":[16763,32884],"train_count":389},{"ids":[21785,148554],"train_count":389},{"ids":[26125,148325],"train_count":389},{"ids":[44432,162292],"train_count":389},{"ids":[61324,7358],"train_count":389},{"ids":[66951,197362],"train_count":389},{"ids":[148752,53062],"train_count":389},{"ids":[148823,8789],"train_count":389},{"ids":[149098,15481],"train_count":389},{"ids":[149589,7105],"train_count":389},{"ids":[14942,22078,150800],"train_count":194},{"ids":[148294,7921],"train_count":388},{"ids":[149413,12666,200641],"train_count":194},{"ids":[155812,4096,48046],"train_count":194},{"ids":[164085,153987],"train_count":388},{"ids":[186361,149208],"train_count":388},{"ids":[10983,148301,2175,36473],"train_count":129},{"ids":[14942,149827],"train_count":387},{"ids":[16806,6733],"train_count":387},{"ids":[19582,21785,93942,4096],"train_count":129},{"ids":[65445,149389,159497,149800],"train_count":129},{"ids":[68608,47552],"train_count":387},{"ids":[148752,59595],"train_count":387},{"ids":[150022,18768,148276,21441],"train_count":129},{"ids":[151721,152141],"train_count":387},{"ids":[5624,148606,17941],"train_count":193},{"ids":[50806,148294,12666],"train_count":193},{"ids":[54025,42565],"train_count":386},{"ids":[148823,8789,150156],"train_count":193},{"ids":[149591,17460],"train_count":386},{"ids":[151146,158702,225639],"train_count":193},{"ids":[159836,158568,42387],"train_count":193},{"ids":[2040,54025],"train_count":385},{"ids":[34208,74990],"train_count":385},{"ids":[36103,148325],"train_count":385},{"ids":[64535,30206],"train_count":385},{"ids":[148292,151642],"train_count":385},{"ids":[7358,41635,45679,6240],"train_count":128},{"ids":[7721,148281,7721],"train_count":192},{"ids":[22220,64257,225639,66050],"train_count":128},{"ids":[32724,10161,149926,150938],"train_count":128},{"ids":[63081,1739,92647],"train_count":192},{"ids":[150532,3652,182270],"train_count":192},{"ids":[154245,6240,7721,77436],"train_count":128},{"ids":[155048,8789,54025,42565],"train_count":128},{"ids":[167797,7721,77969],"train_count":192},{"ids":[12666,200641],"train_count":383},{"ids":[27448,1739],"train_count":383},{"ids":[148390,151828],"train_count":383},{"ids":[7422,18372,148281],"train_count":191},{"ids":[14942,61324,7358],"train_count":191},{"ids":[161183,149800],"train_count":382},{"ids":[10983,36473,2115,8789],"train_count":127},{"ids":[4096,48046],"train_count":380},{"ids":[5241,34981,21441],"train_count":190},{"ids":[7358,178595,26125],"train_count":190},{"ids":[12014,76069],"train_count":380},{"ids":[15729,23514,36103],"train_count":190},{"ids":[16806,148317],"train_count":380},{"ids":[61326,149454],"train_count":380},{"ids":[149454,150140],"train_count":380},{"ids":[149560,34981],"train_count":380},{"ids":[152179,148378,14942],"train_count":190},{"ids":[179458,6733,149800],"train_count":190},{"ids":[209983,6240,7721],"train_count":190},{"ids":[23355,203141],"train_count":379},{"ids":[68140,155110],"train_count":379},{"ids":[202808,32401],"train_count":379},{"ids":[13113,150800],"train_count":378},{"ids":[21445,151963,218943],"train_count":189},{"ids":[23947,23514,18372,1739],"train_count":126},{"ids":[34960,27112,173910,4403],"train_count":126},{"ids":[148275,148276],"train_count":378},{"ids":[148276,9080,77969,6240],"train_count":126},{"ids":[148395,8688],"train_count":378},{"ids":[148427,148336,67494],"train_count":189},{"ids":[148489,15630],"train_count":378},{"ids":[36771,148524],"train_count":377},{"ids":[148301,2175],"train_count":377},{"ids":[167717,152657],"train_count":377},{"ids":[239017,74359],"train_count":377},{"ids":[5092,22078],"train_count":376},{"ids":[11577,159541],"train_count":376},{"ids":[29119,16467,156204],"train_count":188},{"ids":[64257,198326],"train_count":376},{"ids":[76747,148281],"train_count":376},{"ids":[149032,150704],"train_count":376},{"ids":[151232,59627,71658],"train_count":188},{"ids":[203031,21785,149098],"train_count":188},{"ids":[21445,28696,45527,148414],"train_count":125},{"ids":[26149,47745,10161,92647],"train_count":125},{"ids":[77655,148660,16763,6733],"train_count":125},{"ids":[149644,148336,151186,31889],"train_count":125},{"ids":[7721,32884,71658],"train_count":187},{"ids":[34279,148294],"train_count":374},{"ids":[148510,149055],"train_count":374},{"ids":[148579,4096],"train_count":374},{"ids":[149591,26269,11796],"train_count":187},{"ids":[151216,47474,36771],"train_count":187},{"ids":[218574,174005,5301],"train_count":187},{"ids":[32724,155812],"train_count":373},{"ids":[57272,14942],"train_count":373},{"ids":[61326,148292],"train_count":373},{"ids":[157695,47552],"train_count":373},{"ids":[158401,148390],"train_count":373},{"ids":[198620,246235],"train_count":373},{"ids":[6240,7721,77436],"train_count":186},{"ids":[23355,7358,148380,37855],"train_count":124},{"ids":[28376,16806,32401],"train_count":186},{"ids":[31889,16763,6733],"train_count":186},{"ids":[32884,148275],"train_count":372},{"ids":[42395,148700,68608,42387],"train_count":124},{"ids":[45527,78330,26125],"train_count":186},{"ids":[151853,148378,14942],"train_count":186},{"ids":[157696,151676],"train_count":372},{"ids":[212659,14942,159079],"train_count":186},{"ids":[6240,77436],"train_count":371},{"ids":[34208,15481],"train_count":371},{"ids":[2941,150475],"train_count":370},{"ids":[22220,210216,13785],"train_count":185},{"ids":[54858,155898,1450],"train_count":185},{"ids":[149644,148336,151186],"train_count":185},{"ids":[159993,2941],"train_count":370},{"ids":[37519,148292],"train_count":369},{"ids":[65131,148283,60659,148288],"train_count":123},{"ids":[162292,12751],"train_count":369},{"ids":[189053,15481],"train_count":369},{"ids":[5241,180003,149127],"train_count":184},{"ids":[22346,149962,7921],"train_count":184},{"ids":[148303,1424],"train_count":368},{"ids":[148326,148700],"train_count":368},{"ids":[150387,33422,45679],"train_count":184},{"ids":[203467,151291],"train_count":368},{"ids":[16806,15630],"train_count":367},{"ids":[148721,6240],"train_count":367},{"ids":[149337,86128],"train_count":367},{"ids":[10064,149095],"train_count":366},{"ids":[11929,12666,34981],"train_count":183},{"ids":[26607,148292],"train_count":366},{"ids":[77779,26125,148325],"train_count":183},{"ids":[150396,2115],"train_count":366},{"ids":[192416,161978,190978],"train_count":183},{"ids":[213355,159999],"train_count":366},{"ids":[15481,148579],"train_count":365},{"ids":[22078,1424],"train_count":365},{"ids":[12666,12666,45768],"train_count":182},{"ids":[47552,6240],"train_count":364},{"ids":[148561,148319],"train_count":364},{"ids":[155445,10161],"train_count":364},{"ids":[182670,150198,32401],"train_count":182},{"ids":[237856,42565,148779],"train_count":182},{"ids":[18727,148281],"train_count":363},{"ids":[22346,6733,57868,148593],"train_count":121},{"ids":[34208,2115],"train_count":363},{"ids":[55055,149454],"train_count":363},{"ids":[55389,148288],"train_count":363},{"ids":[61326,149454,7105,149193],"train_count":121},{"ids":[68608,64392],"train_count":363},{"ids":[150947,7105],"train_count":363},{"ids":[159042,148397,13739,9080],"train_count":121},{"ids":[200641,44633],"train_count":363},{"ids":[225381,155611,33437,16669],"train_count":121},{"ids":[15630,69729,149137],"train_count":181},{"ids":[18372,150194,16467],"train_count":181},{"ids":[91206,1739],"train_count":362},{"ids":[24653,7721],"train_count":361},{"ids":[30206,3652],"train_count":361},{"ids":[148915,66353],"train_count":361},{"ids":[153342,11796],"train_count":361},{"ids":[7105,13785],"train_count":360},{"ids":[64611,148319],"train_count":360},{"ids":[149509,25497],"train_count":360},{"ids":[32392,148390],"train_count":359},{"ids":[55389,148275],"train_count":359},{"ids":[148288,18372],"train_count":359},{"ids":[149591,25767],"train_count":359},{"ids":[37519,148292,45587],"train_count":179},{"ids":[61326,2115,34981],"train_count":179},{"ids":[85208,14942,36771],"train_count":179},{"ids":[148309,54724,4403],"train_count":179},{"ids":[148397,148414,95525],"train_count":179},{"ids":[148823,34723,149244],"train_count":179},{"ids":[5241,54724],"train_count":357},{"ids":[7550,26125,158668,148917],"train_count":119},{"ids":[16763,148275],"train_count":357},{"ids":[23514,148518],"train_count":357},{"ids":[32868,8789,7754,148276],"train_count":119},{"ids":[42886,61057],"train_count":357},{"ids":[47552,28696],"train_count":357},{"ids":[71000,7721,32884,71658],"train_count":119},{"ids":[213334,45527,78330,26125],"train_count":119},{"ids":[16806,32401],"train_count":356},{"ids":[64535,2941],"train_count":356},{"ids":[148561,7358],"train_count":356},{"ids":[153931,6733,1729],"train_count":178},{"ids":[189726,28420],"train_count":356},{"ids":[17417,2941],"train_count":355},{"ids":[148286,33065],"train_count":355},{"ids":[152959,74990],"train_count":355},{"ids":[8789,7754,148276,1403],"train_count":118},{"ids":[42565,186408],"train_count":354},{"ids":[43800,5301,13302,155063],"train_count":118},{"ids":[64257,225639,6240],"train_count":177},{"ids":[69134,2115,34981,202246],"train_count":118},{"ids":[149541,33422,1424],"train_count":177},{"ids":[155726,149037,148685],"train_count":177},{"ids":[18727,11577],"train_count":353},{"ids":[38745,58361],"train_count":353},{"ids":[152786,191062],"train_count":353},{"ids":[28376,16806,42240],"train_count":176},{"ids":[55302,42565,6389],"train_count":176},{"ids":[76747,148281,37835],"train_count":176},{"ids":[148336,67494],"train_count":352},{"ids":[149054,10161,8789],"train_count":176},{"ids":[151247,57868,153313],"train_count":176},{"ids":[158216,42387],"train_count":352},{"ids":[177220,18768,34981],"train_count":176},{"ids":[181496,6389],"train_count":352},{"ids":[196503,148325],"train_count":352},{"ids":[232612,7721],"train_count":352},{"ids":[42395,148700],"train_count":351},{"ids":[43800,5301,13302,150938],"train_count":117},{"ids":[53555,160259],"train_count":351},{"ids":[151277,15481],"train_count":351},{"ids":[151639,7754,148276,44633],"train_count":117},{"ids":[152839,6240,7721,2115],"train_count":117},{"ids":[153160,48824,148686,149600],"train_count":117},{"ids":[7347,151744],"train_count":350},{"ids":[12666,45768],"train_count":350},{"ids":[34960,27112,173910],"train_count":175},{"ids":[39427,51788],"train_count":350},{"ids":[50816,7105,149800],"train_count":175},{"ids":[82929,148294,152134],"train_count":175},{"ids":[95525,187677,77779],"train_count":175},{"ids":[149465,148327,148319],"train_count":175},{"ids":[149927,61057,152993],"train_count":175},{"ids":[159806,13113,193219],"train_count":175},{"ids":[170946,32401],"train_count":350},{"ids":[176213,149591],"train_count":350},{"ids":[218394,5092],"train_count":350},{"ids":[18768,15630],"train_count":349},{"ids":[19582,16362],"train_count":349},{"ids":[151831,152141],"train_count":349},{"ids":[9080,148283],"train_count":348},{"ids":[10983,9080],"train_count":348},{"ids":[16362,36103,150396,41546],"train_count":116},{"ids":[18727,148357],"train_count":348},{"ids":[45587,1739],"train_count":348},{"ids":[148518,50800,32884,4403],"train_count":116},{"ids":[150599,6240,84706,10161],"train_count":116},{"ids":[150853,148518,50800,32884],"train_count":116},{"ids":[5301,149193],"train_count":347},{"ids":[26607,23355],"train_count":347},{"ids":[28696,33422],"train_count":347},{"ids":[56755,64257],"train_count":347},{"ids":[148309,50800],"train_count":347},{"ids":[168900,7105],"train_count":347},{"ids":[181044,6389],"train_count":347},{"ids":[7358,2040,55389],"train_count":173},{"ids":[7422,18372,150194],"train_count":173},{"ids":[42565,171158],"train_count":346},{"ids":[86949,85208,14942],"train_count":173},{"ids":[148427,12751],"train_count":346},{"ids":[151283,10064],"train_count":346},{"ids":[208002,7358,42387],"train_count":173},{"ids":[219874,29196,17941],"train_count":173},{"ids":[7958,48941],"train_count":345},{"ids":[11929,9080],"train_count":345},{"ids":[70667,149937],"train_count":345},{"ids":[83147,1424],"train_count":345},{"ids":[149746,8789,77436,149537],"train_count":115},{"ids":[150926,151032],"train_count":345},{"ids":[190964,7358],"train_count":345},{"ids":[209983,6240,7721,32884],"train_count":115},{"ids":[18372,15630,57868],"train_count":172},{"ids":[151828,7105],"train_count":344},{"ids":[152786,151292],"train_count":344},{"ids":[162125,181841,148275],"train_count":172},{"ids":[11577,159032],"train_count":343},{"ids":[16362,16763],"train_count":343},{"ids":[16763,4096],"train_count":343},{"ids":[60567,152149],"train_count":343},{"ids":[150455,2175],"train_count":343},{"ids":[157625,149193],"train_count":343},{"ids":[4901,151853,148378],"train_count":171},{"ids":[6499,45587,7358,200641],"train_count":114},{"ids":[6733,40794],"train_count":342},{"ids":[9080,148518],"train_count":342},{"ids":[11577,21797],"train_count":342},{"ids":[26125,45480],"train_count":342},{"ids":[61757,9080,7358,150849],"train_count":114},{"ids":[148318,177567,37835],"train_count":171},{"ids":[148782,16763],"train_count":342},{"ids":[150839,148275],"train_count":342},{"ids":[150919,29119],"train_count":342},{"ids":[152392,148275],"train_count":342},{"ids":[155260,15630,42565,170968],"train_count":114},{"ids":[6733,18110],"train_count":341},{"ids":[14942,5301],"train_count":341},{"ids":[214629,149213],"train_count":341},{"ids":[15481,148301],"train_count":340},{"ids":[21918,6389],"train_count":340},{"ids":[23947,167654],"train_count":340},{"ids":[38745,58361,58245],"train_count":170},{"ids":[149560,6240,158718],"train_count":170},{"ids":[211778,22078,148569],"train_count":170},{"ids":[8789,5092],"train_count":339},{"ids":[34960,154408,6240,168329],"train_count":113},{"ids":[34981,26125],"train_count":339},{"ids":[71658,54201],"train_count":339},{"ids":[148561,7358,7721,26125],"train_count":113},{"ids":[215194,182503,15630,148288],"train_count":113},{"ids":[6240,77436,149537],"train_count":169},{"ids":[7721,155812,149280],"train_count":169},{"ids":[18727,151963,149338],"train_count":169},{"ids":[43080,3652,35458],"train_count":169},{"ids":[46807,31889],"train_count":338},{"ids":[148646,7105],"train_count":338},{"ids":[153899,14666,6389],"train_count":169},{"ids":[23514,148283],"train_count":337},{"ids":[36473,7105],"train_count":337},{"ids":[213383,16669],"train_count":337},{"ids":[4901,4669,21785],"train_count":168},{"ids":[7105,13113,150800],"train_count":168},{"ids":[9080,48623],"train_count":336},{"ids":[10983,36473,39605,45527],"train_count":112},{"ids":[10983,62709,165171],"train_count":168},{"ids":[21720,150396],"train_count":336},{"ids":[34981,7105],"train_count":336},{"ids":[42565,148779],"train_count":336},{"ids":[58427,7721],"train_count":336},{"ids":[76069,154731,53887,21785],"train_count":112},{"ids":[87148,33422,15481,95306],"train_count":112},{"ids":[148301,2175,36473],"train_count":168},{"ids":[148873,151201,18372,36916],"train_count":112},{"ids":[149052,149586],"train_count":336},{"ids":[149810,16467,13113],"train_count":168},{"ids":[160259,32884],"train_count":336},{"ids":[205606,7105],"train_count":336},{"ids":[22220,150294],"train_count":335},{"ids":[86128,24623],"train_count":335},{"ids":[149939,61057],"train_count":335},{"ids":[4901,32884,148294],"train_count":167},{"ids":[13659,12666,13603],"train_count":167},{"ids":[16467,185709],"train_count":334},{"ids":[26607,42317,67958],"train_count":167},{"ids":[26607,151223,74990],"train_count":167},{"ids":[42886,149576,148357],"train_count":167},{"ids":[46807,60860,7105],"train_count":167},{"ids":[52337,150190,150095],"train_count":167},{"ids":[93834,22078,160162],"train_count":167},{"ids":[149586,148275],"train_count":334},{"ids":[155048,8789,54025],"train_count":167},{"ids":[2941,27473],"train_count":333},{"ids":[6733,149800],"train_count":333},{"ids":[58361,58245],"train_count":333},{"ids":[148292,148277],"train_count":333},{"ids":[150545,245644],"train_count":333},{"ids":[198168,42565],"train_count":333},{"ids":[29119,186983,10161],"train_count":166},{"ids":[86949,27448],"train_count":332},{"ids":[148292,16763,37835],"train_count":166},{"ids":[148326,21441,168937],"train_count":166},{"ids":[148561,16806],"train_count":332},{"ids":[163045,6733],"train_count":332},{"ids":[194874,160195],"train_count":332},{"ids":[4901,215194],"train_count":331},{"ids":[58297,11777],"train_count":331},{"ids":[148276,1403],"train_count":331},{"ids":[151947,151666],"train_count":331},{"ids":[2175,149591],"train_count":330},{"ids":[5241,149147,16763,37835],"train_count":110},{"ids":[7422,15481,5092,152148],"train_count":110},{"ids":[11777,148275],"train_count":330},{"ids":[16806,41546],"train_count":330},{"ids":[148782,93834],"train_count":330},{"ids":[148823,25497],"train_count":330},{"ids":[149269,8789,54025,17941],"train_count":110},{"ids":[151201,18372,36916],"train_count":165},{"ids":[152569,173719,36103,42308],"train_count":110},{"ids":[159881,69729],"train_count":330},{"ids":[185698,86128,24623,58245],"train_count":110},{"ids":[2115,90202],"train_count":329},{"ids":[7958,149035],"train_count":329},{"ids":[19736,90397],"train_count":329},{"ids":[34723,149244],"train_count":329},{"ids":[156476,227012],"train_count":329},{"ids":[52687,12666,200641],"train_count":164},{"ids":[61326,149099],"train_count":328},{"ids":[148275,7105,21918],"train_count":164},{"ids":[148336,31889],"train_count":328},{"ids":[149943,167601,10750],"train_count":164},{"ids":[189870,7358,178595],"train_count":164},{"ids":[201553,9080,77969],"train_count":164},{"ids":[222663,54025],"train_count":328},{"ids":[57272,14942,45768,7921],"train_count":109},{"ids":[150387,33422,15481,19937],"train_count":109},{"ids":[157746,76069,154731,53887],"train_count":109},{"ids":[162125,181841,148275,31889],"train_count":109},{"ids":[13532,152865,4669],"train_count":163},{"ids":[46807,168823,155016],"train_count":163},{"ids":[64257,150686],"train_count":326},{"ids":[77969,8688,166815],"train_count":163},{"ids":[148280,77969,5301],"train_count":163},{"ids":[180575,171994],"train_count":326},{"ids":[210138,150704],"train_count":326},{"ids":[7921,148275],"train_count":325},{"ids":[16806,45587],"train_count":325},{"ids":[148757,61057],"train_count":325},{"ids":[151988,76069],"train_count":325},{"ids":[15729,148606,15630,74990],"train_count":108},{"ids":[20177,66951,2115,92647],"train_count":108},{"ids":[39243,150813],"train_count":324},{"ids":[48941,150937],"train_count":324},{"ids":[148397,13739,9080],"train_count":162},{"ids":[149464,148969],"train_count":324},{"ids":[151062,15481],"train_count":324},{"ids":[205606,7105,13785],"train_count":162},{"ids":[7105,152141],"train_count":323},{"ids":[25497,148294],"train_count":323},{"ids":[26149,148445],"train_count":323},{"ids":[53062,45480],"train_count":323},{"ids":[70345,154641],"train_count":323},{"ids":[153362,218814],"train_count":323},{"ids":[158350,42565],"train_count":323},{"ids":[2941,24623,13292],"train_count":161},{"ids":[19648,53887,150198],"train_count":161},{"ids":[22220,9080,45679],"train_count":161},{"ids":[23355,62215],"train_count":322},{"ids":[25497,148283,148446],"train_count":161},{"ids":[33638,36473,9080],"train_count":161},{"ids":[152974,158525,31889],"train_count":161},{"ids":[164726,148279],"train_count":322},{"ids":[187589,7721,36473],"train_count":161},{"ids":[198502,38183],"train_count":322},{"ids":[7422,18372,149744,7721],"train_count":107},{"ids":[15481,2941],"train_count":321},{"ids":[18372,148281,149900,152973],"train_count":107},{"ids":[23355,151438],"train_count":321},{"ids":[32884,71658],"train_count":321},{"ids":[46145,149208,7721,29119],"train_count":107},{"ids":[57272,14942,45768,42074],"train_count":107},{"ids":[158245,87658],"train_count":321},{"ids":[223847,153922],"train_count":321},{"ids":[2941,24623],"train_count":320},{"ids":[8688,148292],"train_count":320},{"ids":[52687,156211,10064],"train_count":160},{"ids":[61326,2115],"train_count":320},{"ids":[148275,31889,1424],"train_count":160},{"ids":[153618,6733,57868],"train_count":160},{"ids":[161624,8688,92647],"train_count":160},{"ids":[212015,150006],"train_count":320},{"ids":[7358,148288],"train_count":319},{"ids":[57868,42240],"train_count":319},{"ids":[152134,150947],"train_count":319},{"ids":[10983,36473,2115],"train_count":159},{"ids":[14942,83147,1424],"train_count":159},{"ids":[48629,64392],"train_count":318},{"ids":[77655,159489,151639,7754],"train_count":106},{"ids":[148496,9396],"train_count":318},{"ids":[153848,7721],"train_count":318},{"ids":[187134,47745,159380],"train_count":159},{"ids":[1450,18768],"train_count":317},{"ids":[157695,1450],"train_count":317},{"ids":[4901,61057,12666],"train_count":158},{"ids":[5092,148325],"train_count":316},{"ids":[26149,5092,21785],"train_count":158},{"ids":[30206,3652,149383],"train_count":158},{"ids":[32868,7358],"train_count":316},{"ids":[148288,1403,13292],"train_count":158},{"ids":[148288,18372,25767],"train_count":158},{"ids":[148873,151201,18372],"train_count":158},{"ids":[149269,12751,2941],"train_count":158},{"ids":[167821,148614,150800],"train_count":158},{"ids":[191881,8789],"train_count":316},{"ids":[213830,4640],"train_count":316},{"ids":[7105,148288],"train_count":315},{"ids":[9080,4403],"train_count":315},{"ids":[20289,159489],"train_count":315},{"ids":[22220,9080,45679,11796],"train_count":105},{"ids":[57868,65360],"train_count":315},{"ids":[57868,148593],"train_count":315},{"ids":[70667,48623,6733,10064],"train_count":105},{"ids":[154382,8789,7721,26125],"train_count":105},{"ids":[160149,4640,34981,2115],"train_count":105},{"ids":[2175,13113],"train_count":314},{"ids":[6733,5301,66951],"train_count":157},{"ids":[19736,152209,45527],"train_count":157},{"ids":[64611,153051,152134],"train_count":157},{"ids":[149140,148675],"train_count":314},{"ids":[149208,168640],"train_count":314},{"ids":[149586,148275,7105],"train_count":157},{"ids":[155094,152833,17941],"train_count":157},{"ids":[185196,151186],"train_count":314},{"ids":[2175,153922],"train_count":313},{"ids":[6733,10064],"train_count":313},{"ids":[8789,7921],"train_count":313},{"ids":[16806,42240],"train_count":313},{"ids":[57868,148325],"train_count":313},{"ids":[215194,182503],"train_count":313},{"ids":[10983,13113],"train_count":312},{"ids":[21797,177702],"train_count":312},{"ids":[23514,36103,42308],"train_count":156},{"ids":[26607,61057],"train_count":312},{"ids":[28376,16806,148325],"train_count":156},{"ids":[47474,36771,67494],"train_count":156},{"ids":[51844,15630,5092,16669],"train_count":104},{"ids":[149591,7105,148325],"train_count":156},{"ids":[150853,148518,50800],"train_count":156},{"ids":[151438,181923],"train_count":312},{"ids":[151642,19937],"train_count":312},{"ids":[152759,36771,11577],"train_count":156},{"ids":[156427,21918,6389],"train_count":156},{"ids":[171011,28696,1739],"train_count":156},{"ids":[171653,7721,149550],"train_count":156},{"ids":[179837,6733],"train_count":312},{"ids":[148380,60038],"train_count":311},{"ids":[158525,31889],"train_count":311},{"ids":[167654,32884],"train_count":311},{"ids":[2175,36473,7105],"train_count":155},{"ids":[5241,21441,159605],"train_count":155},{"ids":[5624,77779,16763],"train_count":155},{"ids":[9629,53555,149068],"train_count":155},{"ids":[19774,158401,148390],"train_count":155},{"ids":[20177,66951,2115],"train_count":155},{"ids":[21785,149098,15481],"train_count":155},{"ids":[26125,39427,37835],"train_count":155},{"ids":[34960,27112],"train_count":310},{"ids":[42886,65131,29119],"train_count":155},{"ids":[56755,53241,229259],"train_count":155},{"ids":[148380,149800],"train_count":310},{"ids":[149454,7105],"train_count":310},{"ids":[151728,152141],"train_count":310},{"ids":[193330,15630,81839],"train_count":155},{"ids":[7721,2115,26125,15481],"train_count":103},{"ids":[14942,151308],"train_count":309},{"ids":[17050,154713],"train_count":309},{"ids":[21785,149537],"train_count":309},{"ids":[26125,64903],"train_count":309},{"ids":[148288,22078,148301,148811],"train_count":103},{"ids":[148292,29119],"train_count":309},{"ids":[148464,149037],"train_count":309},{"ids":[149052,149586,148275,7105],"train_count":103},{"ids":[150423,153342],"train_count":309},{"ids":[150424,18768,148292,9080],"train_count":103},{"ids":[5301,148280],"train_count":308},{"ids":[7721,26125,1450],"train_count":154},{"ids":[25497,8789],"train_count":308},{"ids":[52337,36183,170946],"train_count":154},{"ids":[52687,34981],"train_count":308},{"ids":[158722,12666,148293],"train_count":154},{"ids":[193790,10161,3652],"train_count":154},{"ids":[203095,9629],"train_count":308},{"ids":[221962,32884],"train_count":308},{"ids":[225381,155611,33437],"train_count":154},{"ids":[7358,2040],"train_count":307},{"ids":[14666,6389],"train_count":307},{"ids":[66951,2115],"train_count":307},{"ids":[69134,149741],"train_count":307},{"ids":[148275,34226],"train_count":307},{"ids":[150237,148388],"train_count":307},{"ids":[154299,3652],"train_count":307},{"ids":[215471,163045],"train_count":307},{"ids":[7721,11777,11577],"train_count":153},{"ids":[9629,11577],"train_count":306},{"ids":[19774,89527,18110,18372],"train_count":102},{"ids":[24653,148380,148288],"train_count":153},{"ids":[25497,148761],"train_count":306},{"ids":[33638,148280,77969],"train_count":153},{"ids":[55055,149454,150140,231448],"train_count":102},{"ids":[58297,33422,150366],"train_count":153},{"ids":[83147,4403],"train_count":306},{"ids":[86022,6240,7721],"train_count":153},{"ids":[148464,6733,2040,185709],"train_count":102},{"ids":[148634,148336,67494],"train_count":153},{"ids":[148823,8789,150156,44633],"train_count":102},{"ids":[149464,148969,54201],"train_count":153},{"ids":[150714,15481,19937],"train_count":153},{"ids":[150828,42317,67958,154838],"train_count":102},{"ids":[151509,87148,33422,92647],"train_count":102},{"ids":[153347,8688,13113],"train_count":153},{"ids":[155260,15630,42565,171935],"train_count":102},{"ids":[155812,4096],"train_count":306},{"ids":[157172,148557,172199],"train_count":153},{"ids":[158245,5301,27448],"train_count":153},{"ids":[208734,152611],"train_count":306},{"ids":[216401,13302],"train_count":306},{"ids":[7721,36473],"train_count":305},{"ids":[17160,10161],"train_count":305},{"ids":[35884,15481],"train_count":305},{"ids":[150007,8789],"train_count":305},{"ids":[153848,40710],"train_count":305},{"ids":[166202,7105],"train_count":305},{"ids":[200496,176321],"train_count":305},{"ids":[211603,16806],"train_count":305},{"ids":[7754,148292,148569],"train_count":152},{"ids":[13659,48989,21918],"train_count":152},{"ids":[23355,7358,148380],"train_count":152},{"ids":[23947,148972],"train_count":304},{"ids":[28696,45527,151201],"train_count":152},{"ids":[32868,148561],"train_count":304},{"ids":[36736,7921,15481],"train_count":152},{"ids":[66951,2115,92647],"train_count":152},{"ids":[148378,14942,158638],"train_count":152},{"ids":[148961,183214],"train_count":304},{"ids":[149572,31889],"train_count":304},{"ids":[149591,31889],"train_count":304},{"ids":[153160,48824,148686],"train_count":152},{"ids":[162546,90202],"train_count":304},{"ids":[211607,48046],"train_count":304},{"ids":[4901,61057],"train_count":303},{"ids":[7358,200641],"train_count":303},{"ids":[16763,10161],"train_count":303},{"ids":[19774,65131,148283,60659],"train_count":101},{"ids":[32392,148377,62709,149351],"train_count":101},{"ids":[55055,149454,150140,36771],"train_count":101},{"ids":[65445,153342,11796,149800],"train_count":101},{"ids":[148604,148303,9080,148676],"train_count":101},{"ids":[151011,82942],"train_count":303},{"ids":[151069,12666],"train_count":303},{"ids":[152377,53887,21785,149075],"train_count":101},{"ids":[2175,163347,18768],"train_count":151},{"ids":[5241,149147,1450],"train_count":151},{"ids":[7358,2175],"train_count":302},{"ids":[34226,148685],"train_count":302},{"ids":[41635,3652],"train_count":302},{"ids":[61057,2941],"train_count":302},{"ids":[149208,55389,149280],"train_count":151},{"ids":[151001,149068],"train_count":302},{"ids":[177678,155427],"train_count":302},{"ids":[18768,61057],"train_count":301},{"ids":[32884,1424],"train_count":301},{"ids":[61326,53555],"train_count":301},{"ids":[148378,5301],"train_count":301},{"ids":[153387,148284],"train_count":301},{"ids":[171733,2941],"train_count":301},{"ids":[11929,12666,12666],"train_count":150},{"ids":[18768,148276,21441],"train_count":150},{"ids":[21720,15481,149351],"train_count":150},{"ids":[26149,21785,11777],"train_count":150},{"ids":[53555,148369],"train_count":300},{"ids":[58297,148518,2175],"train_count":150},{"ids":[61127,23858,172384],"train_count":150},{"ids":[149589,7105,13113,29354],"train_count":100},{"ids":[150853,34723,149244],"train_count":150},{"ids":[154665,148276],"train_count":300},{"ids":[156444,222837,149193],"train_count":150},{"ids":[171885,201647,7721],"train_count":150},{"ids":[192379,11577],"train_count":300},{"ids":[1739,14666],"train_count":299},{"ids":[39596,1424],"train_count":299},{"ids":[87148,33422],"train_count":299},{"ids":[148782,156204],"train_count":299},{"ids":[15630,4403],"train_count":298},{"ids":[16362,37483,166595],"train_count":149},{"ids":[22220,39427,150842],"train_count":149},{"ids":[23514,18372,1739],"train_count":149},{"ids":[32868,33437],"train_count":298},{"ids":[56145,148388],"train_count":298},{"ids":[57272,148279,149137],"train_count":149},{"ids":[61326,24623],"train_count":298},{"ids":[64611,148294,151666],"train_count":149},{"ids":[69134,2115],"train_count":298},{"ids":[148395,8688,47474],"train_count":149},{"ids":[148823,148294],"train_count":298},{"ids":[150919,15481],"train_count":298},{"ids":[5624,150425,14666,7105],"train_count":99},{"ids":[48629,48989,4640,18110],"train_count":99},{"ids":[56755,65131],"train_count":297},{"ids":[70345,148281],"train_count":297},{"ids":[148275,33477],"train_count":297},{"ids":[148526,148298],"train_count":297},{"ids":[148537,12751],"train_count":297},{"ids":[148782,93834,22078,160162],"train_count":99},{"ids":[150387,33422,153056,33477],"train_count":99},{"ids":[160149,4640,34981,149633],"train_count":99},{"ids":[160149,149637],"train_count":297},{"ids":[173286,87018],"train_count":297},{"ids":[1739,155055],"train_count":296},{"ids":[5301,42565,171158],"train_count":148},{"ids":[6240,84706,10161],"train_count":148},{"ids":[7754,22078,148284],"train_count":148},{"ids":[23947,15481],"train_count":296},{"ids":[32868,148516,172094],"train_count":148},{"ids":[148281,7721],"train_count":296},{"ids":[148496,154838],"train_count":296},{"ids":[149137,14942,58503],"train_count":148},{"ids":[149591,15481,19937],"train_count":148},{"ids":[155094,152833],"train_count":296},{"ids":[165331,15481,19937],"train_count":148},{"ids":[208002,7358],"train_count":296},{"ids":[5092,2175],"train_count":295},{"ids":[6240,150849],"train_count":295},{"ids":[7721,148281],"train_count":295},{"ids":[22220,36771],"train_count":295},{"ids":[70749,148464],"train_count":295},{"ids":[71658,68335],"train_count":295},{"ids":[149560,6240],"train_count":295},{"ids":[150530,197375],"train_count":295},{"ids":[151292,157647],"train_count":295},{"ids":[154390,54025],"train_count":295},{"ids":[198528,16467],"train_count":295},{"ids":[209104,5301],"train_count":295},{"ids":[222663,91206],"train_count":295},{"ids":[2115,8789,148390],"train_count":147},{"ids":[4901,93834,12666,148317],"train_count":98},{"ids":[5301,23514,36103,148317],"train_count":98},{"ids":[26125,158668,16806],"train_count":147},{"ids":[28696,7105],"train_count":294},{"ids":[53062,42308],"train_count":294},{"ids":[148280,77969,5301,149193],"train_count":98},{"ids":[148326,21441,168937,33422],"train_count":98},{"ids":[151011,1729],"train_count":294},{"ids":[152786,151292,53555],"train_count":147},{"ids":[160761,173910,148275],"train_count":147},{"ids":[186361,149208,55389,149280],"train_count":98},{"ids":[202020,2941,24623,13292],"train_count":98},{"ids":[15630,81839],"train_count":293},{"ids":[29354,90219],"train_count":293},{"ids":[34208,18372],"train_count":293},{"ids":[148292,148330],"train_count":293},{"ids":[152959,150187],"train_count":293},{"ids":[2175,42565,149633],"train_count":146},{"ids":[7105,45480],"train_count":292},{"ids":[7422,15481,5092],"train_count":146},{"ids":[7958,149035,37835],"train_count":146},{"ids":[12666,148288],"train_count":292},{"ids":[13532,154793],"train_count":292},{"ids":[16362,36103,150396],"train_count":146},{"ids":[32868,8789],"train_count":292},{"ids":[39243,60909,150366],"train_count":146},{"ids":[53887,5092],"train_count":292},{"ids":[148281,148686],"train_count":292},{"ids":[150853,61757],"train_count":292},{"ids":[179458,6733],"train_count":292},{"ids":[217038,7754,148292],"train_count":146},{"ids":[220204,39427],"train_count":292},{"ids":[231736,149005],"train_count":292},{"ids":[232069,11577],"train_count":292},{"ids":[23514,7105,148275,161782],"train_count":97},{"ids":[26607,42317,67958,42565],"train_count":97},{"ids":[36736,7921,15481,2941],"train_count":97},{"ids":[42565,2175],"train_count":291},{"ids":[55389,44633],"train_count":291},{"ids":[148526,16763],"train_count":291},{"ids":[154521,148303,1450,18768],"train_count":97},{"ids":[181622,148343],"train_count":291},{"ids":[2115,23514],"train_count":290},{"ids":[6240,7721,148281],"train_count":145},{"ids":[7754,148276,1403],"train_count":145},{"ids":[10983,41635],"train_count":290},{"ids":[11777,10750],"train_count":290},{"ids":[19582,16763,148277],"train_count":145},{"ids":[19774,29119,173919],"train_count":145},{"ids":[21445,151963],"train_count":290},{"ids":[23947,23514,18372],"train_count":145},{"ids":[41635,151883],"train_count":290},{"ids":[93614,28376,16806],"train_count":145},{"ids":[148569,149800],"train_count":290},{"ids":[50806,17417],"train_count":289},{"ids":[2175,153922,150451],"train_count":144},{"ids":[4640,34981,21441],"train_count":144},{"ids":[5301,149280],"train_count":288},{"ids":[6240,33065],"train_count":288},{"ids":[7358,92647],"train_count":288},{"ids":[15481,7358,42387],"train_count":144},{"ids":[18727,148275],"train_count":288},{"ids":[18768,39427],"train_count":288},{"ids":[19774,65131,148283,152933],"train_count":96},{"ids":[22220,25497,16362],"train_count":144},{"ids":[23355,148445,36183],"train_count":144},{"ids":[23947,148288,22078,148301],"train_count":96},{"ids":[46145,149208,55389,148288],"train_count":96},{"ids":[87018,21785,150621],"train_count":144},{"ids":[148782,58361],"train_count":288},{"ids":[149208,55389,149193],"train_count":144},{"ids":[149663,16806],"train_count":288},{"ids":[162564,4640,34981,7105],"train_count":96},{"ids":[187134,47745,5301],"train_count":144},{"ids":[5241,11777],"train_count":287},{"ids":[6733,149499],"train_count":287},{"ids":[34208,23514],"train_count":287},{"ids":[52337,36183],"train_count":287},{"ids":[149576,148275],"train_count":287},{"ids":[13113,29354,90219],"train_count":143},{"ids":[23355,81839,150282],"train_count":143},{"ids":[25497,62709,3652],"train_count":143},{"ids":[26125,158668,71221],"train_count":143},{"ids":[29119,173919,77779],"train_count":143},{"ids":[37519,61324],"train_count":286},{"ids":[37519,148281],"train_count":286},{"ids":[38745,58361,65360],"train_count":143},{"ids":[61326,148464],"train_count":286},{"ids":[68608,15630],"train_count":286},{"ids":[148294,152134,245644],"train_count":143},{"ids":[149269,229903],"train_count":286},{"ids":[152959,74990,148288],"train_count":143},{"ids":[152974,158525],"train_count":286},{"ids":[152996,150842,11796],"train_count":143},{"ids":[164467,6733,149499],"train_count":143},{"ids":[220232,11802,148292],"train_count":143},{"ids":[4901,93834,12666,4403],"train_count":95},{"ids":[15481,9080],"train_count":285},{"ids":[17050,150178],"train_count":285},{"ids":[26125,42240],"train_count":285},{"ids":[42886,149095,61057,42565],"train_count":95},{"ids":[84400,148561,7358,7721],"train_count":95},{"ids":[148700,42565],"train_count":285},{"ids":[149191,15630,69729,182032],"train_count":95},{"ids":[151146,158702,225639,149193],"train_count":95},{"ids":[151183,196290],"train_count":285},{"ids":[151459,155812,4096,48046],"train_count":95},{"ids":[158693,36183],"train_count":285},{"ids":[2941,27473,59627],"train_count":142},{"ids":[7754,23514,7105],"train_count":142},{"ids":[10161,149926,150938],"train_count":142},{"ids":[17569,53887],"train_count":284},{"ids":[20289,2175],"train_count":284},{"ids":[39427,148277],"train_count":284},{"ids":[62709,3652,21441],"train_count":142},{"ids":[151728,149193],"train_count":284},{"ids":[151891,81839],"train_count":284},{"ids":[152130,2941,171780],"train_count":142},{"ids":[155609,149572],"train_count":284},{"ids":[7105,13113],"train_count":283},{"ids":[22220,210216],"train_count":283},{"ids":[46145,150456],"train_count":283},{"ids":[148390,16806],"train_count":283},{"ids":[4901,32884,148294,7921],"train_count":94},{"ids":[6240,7721,77436,149537],"train_count":94},{"ids":[7347,2115,200641],"train_count":141},{"ids":[13532,151919],"train_count":282},{"ids":[23355,151290],"train_count":282},{"ids":[25497,148294,42565,37855],"train_count":94},{"ids":[29119,10161],"train_count":282},{"ids":[32724,10161,149926,153174],"train_count":94},{"ids":[46807,149099],"train_count":282},{"ids":[61326,149099,5092],"train_count":141},{"ids":[148526,148298,7754,14666],"train_count":94},{"ids":[149430,158693,36183,1450],"train_count":94},{"ids":[150853,34723,149244,149665],"train_count":94},{"ids":[151161,152824,57868],"train_count":141},{"ids":[154849,77969,8688,166815],"train_count":94},{"ids":[157862,15481,19937],"train_count":141},{"ids":[190964,29119,150532],"train_count":141},{"ids":[202020,2941],"train_count":282},{"ids":[208086,149345,1739,155055],"train_count":94},{"ids":[233244,148275,31889],"train_count":141},{"ids":[18727,22078],"train_count":281},{"ids":[22220,61057],"train_count":281},{"ids":[22220,148390],"train_count":281},{"ids":[150047,19937],"train_count":281},{"ids":[14942,2040],"train_count":280},{"ids":[16797,44432],"train_count":280},{"ids":[19582,47474],"train_count":280},{"ids":[19736,150270,161907],"train_count":140},{"ids":[27473,59627,71658],"train_count":140},{"ids":[36183,170946,41546],"train_count":140},{"ids":[36473,2115,8789],"train_count":140},{"ids":[48629,2115,148877],"train_count":140},{"ids":[77779,26125,148540],"train_count":140},{"ids":[86128,24623,58245],"train_count":140},{"ids":[149720,148367,149880],"train_count":140},{"ids":[153306,14942,22078],"train_count":140},{"ids":[154521,148275,7105],"train_count":140},{"ids":[158291,152898],"train_count":280},{"ids":[166897,148569],"train_count":280},{"ids":[167717,7721],"train_count":280},{"ids":[241496,198597,19937],"train_count":140},{"ids":[2115,11777],"train_count":279},{"ids":[6389,44633],"train_count":279},{"ids":[69729,5301],"train_count":279},{"ids":[149233,148464,6733,2040],"train_count":93},{"ids":[17050,150178,149572],"train_count":139},{"ids":[18768,34981],"train_count":278},{"ids":[21441,168937,33422],"train_count":139},{"ids":[23355,16467],"train_count":278},{"ids":[23947,148972,149232],"train_count":139},{"ids":[35884,83147,1424],"train_count":139},{"ids":[36771,73781],"train_count":278},{"ids":[38859,66951],"train_count":278},{"ids":[38859,148700,42565],"train_count":139},{"ids":[39243,60909],"train_count":278},{"ids":[41381,149095,7105],"train_count":139},{"ids":[42565,149633],"train_count":278},{"ids":[43080,3652],"train_count":278},{"ids":[57632,59627],"train_count":278},{"ids":[63081,1739,15481],"train_count":139},{"ids":[148292,61057,16806],"train_count":139},{"ids":[148292,150047],"train_count":278},{"ids":[148318,155916],"train_count":278},{"ids":[148526,8789,5092],"train_count":139},{"ids":[149560,174531],"train_count":278},{"ids":[157892,6240,84706],"train_count":139},{"ids":[181931,165328,44633],"train_count":139},{"ids":[203031,21785],"train_count":278},{"ids":[1739,38183],"train_count":277},{"ids":[6733,5301],"train_count":277},{"ids":[10161,1729],"train_count":277},{"ids":[13532,148757],"train_count":277},{"ids":[34208,17417],"train_count":277},{"ids":[34981,2115],"train_count":277},{"ids":[50806,148292],"train_count":277},{"ids":[149167,82557],"train_count":277},{"ids":[168937,33422],"train_count":277},{"ids":[182503,15630],"train_count":277},{"ids":[5624,10750,86128,24623],"train_count":92},{"ids":[7754,148276,44633],"train_count":138},{"ids":[16362,154822],"train_count":276},{"ids":[22078,59627],"train_count":276},{"ids":[45527,148414],"train_count":276},{"ids":[52337,36183,157759,151407],"train_count":92},{"ids":[52337,64257,148324],"train_count":138},{"ids":[64257,148324,181923],"train_count":138},{"ids":[81573,202403],"train_count":276},{"ids":[87148,33422,15481],"train_count":138},{"ids":[148283,148634,9080],"train_count":138},{"ids":[148292,16763,44633],"train_count":138},{"ids":[148823,34723,149244,67494],"train_count":92},{"ids":[149095,61057,42565,5301],"train_count":92},{"ids":[149389,159497,149800],"train_count":138},{"ids":[150270,161907],"train_count":276},{"ids":[151706,14942,47745,172637],"train_count":92},{"ids":[152155,149089,34723,44633],"train_count":92},{"ids":[158245,5301,27448,26125],"train_count":92},{"ids":[211778,22078],"train_count":276},{"ids":[5241,180003],"train_count":275},{"ids":[10983,59627],"train_count":275},{"ids":[33437,155295],"train_count":275},{"ids":[46807,34723],"train_count":275},{"ids":[148281,149885],"train_count":275},{"ids":[150198,32401],"train_count":275},{"ids":[14942,36771],"train_count":274},{"ids":[34960,154408,6240],"train_count":137},{"ids":[39625,6733,58475],"train_count":137},{"ids":[148281,148686,154479],"train_count":137},{"ids":[148464,64257,149800],"train_count":137},{"ids":[149269,12751],"train_count":274},{"ids":[149720,148367],"train_count":274},{"ids":[149746,8789,158585],"train_count":137},{"ids":[150237,148388,16806],"train_count":137},{"ids":[153020,148276,61057],"train_count":137},{"ids":[214629,148939],"train_count":274},{"ids":[4901,151853,1739,7721],"train_count":91},{"ids":[7105,13113,29354,90219],"train_count":91},{"ids":[7921,15481,2941,157647],"train_count":91},{"ids":[14942,22078,150800,44633],"train_count":91},{"ids":[15481,148288],"train_count":273},{"ids":[16362,22078,42565,171144],"train_count":91},{"ids":[16362,148381],"train_count":273},{"ids":[34279,5301,42565,171158],"train_count":91},{"ids":[34960,2040,10161,36771],"train_count":91},{"ids":[89030,4640],"train_count":273},{"ids":[148303,9080],"train_count":273},{"ids":[148378,14942,77436,150621],"train_count":91},{"ids":[148395,8688,47474,4669],"train_count":91},{"ids":[148526,148288,1403,13292],"train_count":91},{"ids":[149454,150140,36771,45480],"train_count":91},{"ids":[191297,7754,84920,55055],"train_count":91},{"ids":[8789,77436,149537],"train_count":136},{"ids":[18768,15630,148275],"train_count":136},{"ids":[26607,148378],"train_count":272},{"ids":[28376,16806,41546],"train_count":136},{"ids":[32392,7105,150528],"train_count":136},{"ids":[36473,148288],"train_count":272},{"ids":[38859,150348,149809],"train_count":136},{"ids":[55302,69729,11796],"train_count":136},{"ids":[65445,153342],"train_count":272},{"ids":[77655,211593],"train_count":272},{"ids":[77969,5301],"train_count":272},{"ids":[148275,13785],"train_count":272},{"ids":[148318,177567,44633],"train_count":136},{"ids":[154665,148276,185709],"train_count":136},{"ids":[220630,36771],"train_count":272},{"ids":[2115,92647],"train_count":271},{"ids":[16763,42074],"train_count":271},{"ids":[33422,92647],"train_count":271},{"ids":[152986,153501],"train_count":271},{"ids":[154822,8789],"train_count":271},{"ids":[156476,150584],"train_count":271},{"ids":[218321,148317],"train_count":271},{"ids":[1424,155899],"train_count":270},{"ids":[1739,28376],"train_count":270},{"ids":[2040,10161,36771,1729],"train_count":90},{"ids":[5301,13302,150938],"train_count":135},{"ids":[6733,149591],"train_count":270},{"ids":[9080,148622],"train_count":270},{"ids":[18727,150270,198597,19937],"train_count":90},{"ids":[20289,2175,12751,60659],"train_count":90},{"ids":[22078,148276],"train_count":270},{"ids":[23514,36103,42240],"train_count":135},{"ids":[34981,26125,1424,13302],"train_count":90},{"ids":[54025,16669],"train_count":270},{"ids":[57868,32401],"train_count":270},{"ids":[150455,2175,153922,51788],"train_count":90},{"ids":[150926,151032,42565,42074],"train_count":90},{"ids":[152870,148489,15630,53887],"train_count":90},{"ids":[153020,148276,9080,77969],"train_count":90},{"ids":[153682,47745,5301],"train_count":135},{"ids":[155224,13292,148943],"train_count":135},{"ids":[161420,42565,155130],"train_count":135},{"ids":[203031,21785,149098,15481],"train_count":90},{"ids":[217650,160195],"train_count":270},{"ids":[234932,14942,45679,1739],"train_count":90},{"ids":[1450,148378],"train_count":269},{"ids":[17816,45144],"train_count":269},{"ids":[29119,186983],"train_count":269},{"ids":[37519,13603],"train_count":269},{"ids":[41381,7358],"train_count":269},{"ids":[148292,1739],"train_count":269},{"ids":[148336,15481],"train_count":269},{"ids":[148362,9080],"train_count":269},{"ids":[149541,39596],"train_count":269},{"ids":[151980,14942],"train_count":269},{"ids":[163731,3652],"train_count":269},{"ids":[8789,7721,26125],"train_count":134},{"ids":[11796,149800],"train_count":268},{"ids":[12666,200641,44633],"train_count":134},{"ids":[23514,36103,148325],"train_count":134},{"ids":[37519,11577],"train_count":268},{"ids":[42565,90219],"train_count":268},{"ids":[148660,16763,6733],"train_count":134},{"ids":[149867,10064,16763],"train_count":134},{"ids":[151069,12666,4403],"train_count":134},{"ids":[154729,37855],"train_count":268},{"ids":[157089,148336,7105],"train_count":134},{"ids":[9080,148288],"train_count":267},{"ids":[13302,2941],"train_count":267},{"ids":[20578,159541,5301,55389],"train_count":89},{"ids":[26125,17460],"train_count":267},{"ids":[32724,148292,35458,151418],"train_count":89},{"ids":[148362,148276],"train_count":267},{"ids":[149092,167834,22078,148318],"train_count":89},{"ids":[149538,148558,72884,68608],"train_count":89},{"ids":[151283,155066],"train_count":267},{"ids":[205914,152175],"train_count":267},{"ids":[207347,150029,1403,181923],"train_count":89},{"ids":[5301,13302,155063],"train_count":133},{"ids":[7358,7721,26125],"train_count":133},{"ids":[14942,9080,148622],"train_count":133},{"ids":[35105,14942,149827],"train_count":133},{"ids":[42886,187056],"train_count":266},{"ids":[48629,48989,4640],"train_count":133},{"ids":[69729,11796],"train_count":266},{"ids":[148362,9080,7921],"train_count":133},{"ids":[153047,77436],"train_count":266},{"ids":[154626,74990,57868],"train_count":133},{"ids":[155100,148378,14942],"train_count":133},{"ids":[157391,148281,16806],"train_count":133},{"ids":[206694,7358,152027],"train_count":133},{"ids":[9396,2040],"train_count":265},{"ids":[15481,5092],"train_count":265},{"ids":[19736,2115],"train_count":265},{"ids":[148390,2040],"train_count":265},{"ids":[148972,149232],"train_count":265},{"ids":[149389,159497],"train_count":265},{"ids":[154767,15630],"train_count":265},{"ids":[4901,4669,21785,4403],"train_count":88},{"ids":[7347,153194,9629,53062],"train_count":88},{"ids":[10161,5092],"train_count":264},{"ids":[16797,32884],"train_count":264},{"ids":[17160,157977,154838],"train_count":132},{"ids":[22220,210216,13785,66050],"train_count":88},{"ids":[31889,16763],"train_count":264},{"ids":[42886,149576,148357,66050],"train_count":88},{"ids":[54724,149591,7105],"train_count":132},{"ids":[57272,42565],"train_count":264},{"ids":[64257,225639,66050],"train_count":132},{"ids":[148606,15630,74990],"train_count":132},{"ids":[149037,1739],"train_count":264},{"ids":[149591,26269],"train_count":264},{"ids":[149622,148294,16806,5831],"train_count":88},{"ids":[154178,2115,23514,7105],"train_count":88},{"ids":[154527,148280,148277],"train_count":132},{"ids":[167791,155610,31889],"train_count":132},{"ids":[15729,148294],"train_count":263},{"ids":[36103,5092],"train_count":263},{"ids":[151728,150584],"train_count":263},{"ids":[155032,198325],"train_count":263},{"ids":[205914,16467],"train_count":263},{"ids":[7958,219510],"train_count":262},{"ids":[18727,24623],"train_count":262},{"ids":[25497,160827,7721],"train_count":131},{"ids":[71000,7721,32884],"train_count":131},{"ids":[81573,151223,152156],"train_count":131},{"ids":[87018,21785,149537],"train_count":131},{"ids":[149098,15481,4403],"train_count":131},{"ids":[150022,18768,148276],"train_count":131},{"ids":[150834,9080,77969],"train_count":131},{"ids":[155039,14942],"train_count":262},{"ids":[175120,148778],"train_count":262},{"ids":[190964,29119,10161],"train_count":131},{"ids":[198597,19937],"train_count":262},{"ids":[5624,23514,36103,42240],"train_count":87},{"ids":[10161,15481],"train_count":261},{"ids":[15630,5092],"train_count":261},{"ids":[23947,23514],"train_count":261},{"ids":[23947,148676],"train_count":261},{"ids":[37519,16763],"train_count":261},{"ids":[46145,149208,55389,149193],"train_count":87},{"ids":[53555,148380,34981,67494],"train_count":87},{"ids":[57272,149137,14942,58503],"train_count":87},{"ids":[148390,151828,7105,149800],"train_count":87},{"ids":[150175,152138,149663,16806],"train_count":87},{"ids":[151945,16763,155374,84706],"train_count":87},{"ids":[152011,19937],"train_count":261},{"ids":[155782,150704],"train_count":261},{"ids":[159733,15481],"train_count":261},{"ids":[224036,152417,157759,6389],"train_count":87},{"ids":[18768,61057,6733],"train_count":130},{"ids":[20177,55986,5092],"train_count":130},{"ids":[33638,36473,148319],"train_count":130},{"ids":[148336,151186,31889],"train_count":130},{"ids":[148362,148276,44633],"train_count":130},{"ids":[148445,36183,149351],"train_count":130},{"ids":[150401,84480],"train_count":260},{"ids":[150757,148369],"train_count":260},{"ids":[150800,177067],"train_count":260},{"ids":[158441,82557],"train_count":260},{"ids":[158503,1424,10161],"train_count":130},{"ids":[158693,152644],"train_count":260},{"ids":[164018,5301],"train_count":260},{"ids":[6389,158585],"train_count":259},{"ids":[36736,18372],"train_count":259},{"ids":[48629,2115],"train_count":259},{"ids":[55389,4403],"train_count":259},{"ids":[148276,150528],"train_count":259},{"ids":[148823,3652],"train_count":259},{"ids":[150745,153002],"train_count":259},{"ids":[3652,65131],"train_count":258},{"ids":[10983,148301,2175],"train_count":129},{"ids":[18768,34981,26125,1424],"train_count":86},{"ids":[19582,21785,93942],"train_count":129},{"ids":[21785,93942,4096],"train_count":129},{"ids":[23514,148283,53062],"train_count":129},{"ids":[23947,9080,148622],"train_count":129},{"ids":[25497,148292,15481,172094],"train_count":86},{"ids":[26125,151642,19937],"train_count":129},{"ids":[26607,5301,148280,148277],"train_count":86},{"ids":[35884,5092,7358],"train_count":129},{"ids":[43080,5301,23514],"train_count":129},{"ids":[53555,148275],"train_count":258},{"ids":[58427,153051],"train_count":258},{"ids":[70749,148464,150187],"train_count":129},{"ids":[148277,26125],"train_count":258},{"ids":[148288,1403,13292,180298],"train_count":86},{"ids":[148464,6733,194972,155130],"train_count":86},{"ids":[149233,148288],"train_count":258},{"ids":[149563,62847,29119,7754],"train_count":86},{"ids":[154521,148303,1450],"train_count":129},{"ids":[164467,56090],"train_count":258},{"ids":[165328,42387],"train_count":258},{"ids":[205914,16467,150167],"train_count":129},{"ids":[210370,156977],"train_count":258},{"ids":[5301,27448],"train_count":257},{"ids":[10567,150544],"train_count":257},{"ids":[26125,7358],"train_count":257},{"ids":[31889,42308],"train_count":257},{"ids":[33638,45587],"train_count":257},{"ids":[42565,172453],"train_count":257},{"ids":[61057,1450],"train_count":257},{"ids":[151232,59627],"train_count":257},{"ids":[161509,15481],"train_count":257},{"ids":[7347,9629,2040],"train_count":128},{"ids":[9396,2040,153987],"train_count":128},{"ids":[15630,148275,31889],"train_count":128},{"ids":[26149,87658,148453],"train_count":128},{"ids":[34208,22078],"train_count":256},{"ids":[41635,45679,6240],"train_count":128},{"ids":[42886,242110,10161],"train_count":128},{"ids":[56755,2115],"train_count":256},{"ids":[58427,15481],"train_count":256},{"ids":[84400,148561,7358],"train_count":128},{"ids":[148276,9080,77969],"train_count":128},{"ids":[148337,11802,150198],"train_count":128},{"ids":[148700,68608],"train_count":256},{"ids":[150828,148331],"train_count":256},{"ids":[151032,42565],"train_count":256},{"ids":[153935,7105],"train_count":256},{"ids":[155611,33437,16669],"train_count":128},{"ids":[159133,8688,172518],"train_count":128},{"ids":[171132,148317],"train_count":256},{"ids":[7754,148280,77779,15630],"train_count":85},{"ids":[16806,148325],"train_count":255},{"ids":[18110,18372],"train_count":255},{"ids":[19774,33437,16467,149591],"train_count":85},{"ids":[20177,55986],"train_count":255},{"ids":[26125,149891],"train_count":255},{"ids":[27112,155066],"train_count":255},{"ids":[148279,16467,13113,2175],"train_count":85},{"ids":[148288,1403],"train_count":255},{"ids":[150187,6240],"train_count":255},{"ids":[150198,148317],"train_count":255},{"ids":[151718,148288],"train_count":255},{"ids":[152786,151292,53555,149068],"train_count":85},{"ids":[167834,22078,148318,177567],"train_count":85},{"ids":[179837,6733,40794,64392],"train_count":85},{"ids":[2175,12751,60659],"train_count":127},{"ids":[3652,182270],"train_count":254},{"ids":[5241,150800,4096],"train_count":127},{"ids":[13532,149679,150469],"train_count":127},{"ids":[18727,151223,74990],"train_count":127},{"ids":[21445,168114],"train_count":254},{"ids":[21785,149595],"train_count":254},{"ids":[33422,158605],"train_count":254},{"ids":[39427,149591],"train_count":254},{"ids":[58427,148292,29119],"train_count":127},{"ids":[65131,148283,152933],"train_count":127},{"ids":[148427,42565,42074],"train_count":127},{"ids":[148873,148489,18372],"train_count":127},{"ids":[148961,148700],"train_count":254},{"ids":[151290,148489],"train_count":254},{"ids":[152959,148275,15481],"train_count":127},{"ids":[153020,148276,53062],"train_count":127},{"ids":[157695,45768,150456],"train_count":127},{"ids":[158350,42565,148539],"train_count":127},{"ids":[208086,149345,1739],"train_count":127},{"ids":[215774,12666,36103],"train_count":127},{"ids":[21785,149098],"train_count":253},{"ids":[47474,42565],"train_count":253},{"ids":[60659,148288],"train_count":253},{"ids":[93834,148275],"train_count":253},{"ids":[148275,16763],"train_count":253},{"ids":[148337,220495],"train_count":253},{"ids":[148524,8789],"train_count":253},{"ids":[149455,149800],"train_count":253},{"ids":[152024,42565],"train_count":253},{"ids":[2115,23514,7105],"train_count":126},{"ids":[14942,61324],"train_count":252},{"ids":[17417,148275],"train_count":252},{"ids":[19774,65131,148283,157853],"train_count":84},{"ids":[20177,153683,2941,53062],"train_count":84},{"ids":[22220,210216,13785,148446],"train_count":84},{"ids":[23355,62215,47552],"train_count":126},{"ids":[27112,173910,4403],"train_count":126},{"ids":[34208,65131],"train_count":252},{"ids":[38859,182867,150006],"train_count":126},{"ids":[41381,22078,59627,148275],"train_count":84},{"ids":[47474,42565,5301],"train_count":126},{"ids":[53555,149068],"train_count":252},{"ids":[68608,148702],"train_count":252},{"ids":[148298,2040],"train_count":252},{"ids":[148782,16763,27112],"train_count":126},{"ids":[151283,153238,148369],"train_count":126},{"ids":[161420,42565,171096],"train_count":126},{"ids":[163868,14942,72884],"train_count":126},{"ids":[164085,153987,149280],"train_count":126},{"ids":[166266,150479,21785,5092],"train_count":84},{"ids":[171593,39596,42565,34083],"train_count":84},{"ids":[215471,163045,6733,74990],"train_count":84},{"ids":[7105,68335],"train_count":251},{"ids":[7721,29119],"train_count":251},{"ids":[10064,90202],"train_count":251},{"ids":[25497,21918],"train_count":251},{"ids":[56287,154767],"train_count":251},{"ids":[148915,18372],"train_count":251},{"ids":[148974,86128],"train_count":251},{"ids":[150772,148276],"train_count":251},{"ids":[150800,4096],"train_count":251},{"ids":[162292,50800],"train_count":251},{"ids":[175447,4096],"train_count":251},{"ids":[7358,41635],"train_count":250},{"ids":[7721,155812],"train_count":250},{"ids":[10064,148554],"train_count":250},{"ids":[11929,47745,6240],"train_count":125},{"ids":[18372,36916],"train_count":250},{"ids":[34208,148303],"train_count":250},{"ids":[47745,10161,92647],"train_count":125},{"ids":[53887,21785,149075],"train_count":125},{"ids":[60860,7105],"train_count":250},{"ids":[61326,148280],"train_count":250},{"ids":[65131,148283,60659],"train_count":125},{"ids":[74990,148288],"train_count":250},{"ids":[148868,68813],"train_count":250},{"ids":[149454,7105,149193],"train_count":125},{"ids":[150704,32884,2115],"train_count":125},{"ids":[150984,166449],"train_count":250},{"ids":[154731,53887,21785],"train_count":125},{"ids":[163731,1450],"train_count":250},{"ids":[218574,171251,149280],"train_count":125},{"ids":[7754,148292,148569,12751],"train_count":83},{"ids":[11929,12666,34981,7105],"train_count":83},{"ids":[14942,156977],"train_count":249},{"ids":[15729,23514,36103,148317],"train_count":83},{"ids":[18727,28696],"train_count":249},{"ids":[37519,148281,148518,77779],"train_count":83},{"ids":[41381,22078],"train_count":249},{"ids":[42886,149576],"train_count":249},{"ids":[45587,7358,200641,4403],"train_count":83},{"ids":[45587,7358,200641,148275],"train_count":83},{"ids":[54858,89527,151844,12014],"train_count":83},{"ids":[56755,64257,4096,66050],"train_count":83},{"ids":[65445,149389],"train_count":249},{"ids":[74990,148518],"train_count":249},{"ids":[82383,153298,13113,4403],"train_count":83},{"ids":[148294,42565],"train_count":249},{"ids":[148378,14942,77436,153232],"train_count":83},{"ids":[148397,13739,9080,67494],"train_count":83},{"ids":[4640,34981,2115],"train_count":124},{"ids":[4901,151853,158283],"train_count":124},{"ids":[7105,21918],"train_count":248},{"ids":[7358,148380,37855],"train_count":124},{"ids":[7754,23514],"train_count":248},{"ids":[7921,15481,2941],"train_count":124},{"ids":[7958,149035,42387],"train_count":124},{"ids":[28696,148518],"train_count":248},{"ids":[42886,149095,61057],"train_count":124},{"ids":[48824,148686,149600],"train_count":124},{"ids":[59627,1739],"train_count":248},{"ids":[64257,149800],"train_count":248},{"ids":[148319,18768],"train_count":248},{"ids":[148378,14942,158585],"train_count":124},{"ids":[148700,68608,42387],"train_count":124},{"ids":[150828,148331,69729],"train_count":124},{"ids":[152973,77436,149537],"train_count":124},{"ids":[157862,15481],"train_count":248},{"ids":[224895,48623],"train_count":248},{"ids":[227138,149086,53062],"train_count":124},{"ids":[7921,15481],"train_count":247},{"ids":[8789,148288],"train_count":247},{"ids":[10064,11777],"train_count":247},{"ids":[11802,148292],"train_count":247},{"ids":[21441,150282],"train_count":247},{"ids":[23947,149717],"train_count":247},{"ids":[36736,7921],"train_count":247},{"ids":[36771,67494],"train_count":247},{"ids":[80730,36183],"train_count":247},{"ids":[94491,7358],"train_count":247},{"ids":[167664,149800],"train_count":247},{"ids":[6733,10064,17460],"train_count":123},{"ids":[10983,62709],"train_count":246},{"ids":[10983,176068,159236,150051],"train_count":82},{"ids":[11929,8789],"train_count":246},{"ids":[15630,1450],"train_count":246},{"ids":[18372,4096],"train_count":246},{"ids":[18372,149744,7721],"train_count":123},{"ids":[21445,28696,45527,151201],"train_count":82},{"ids":[25497,21918,148388],"train_count":123},{"ids":[35770,154731,90202],"train_count":123},{"ids":[51844,15630,5092],"train_count":123},{"ids":[54025,149637],"train_count":246},{"ids":[61057,10750],"train_count":246},{"ids":[61757,9396,2040,153987],"train_count":82},{"ids":[62709,165171],"train_count":246},{"ids":[92173,152371],"train_count":246},{"ids":[148283,60659,148288],"train_count":123},{"ids":[148292,4403],"train_count":246},{"ids":[149092,15630,1450],"train_count":123},{"ids":[149199,8688,47745],"train_count":123},{"ids":[150401,8789,53887],"train_count":123},{"ids":[151945,16763,42074],"train_count":123},{"ids":[155132,159400,42565],"train_count":123},{"ids":[159541,5301,55389],"train_count":123},{"ids":[167994,42565],"train_count":246},{"ids":[170866,148561,16763],"train_count":123},{"ids":[171133,10064],"train_count":246},{"ids":[215384,4669],"train_count":246},{"ids":[234932,14942,45679],"train_count":123},{"ids":[26269,61628],"train_count":245},{"ids":[26607,7754],"train_count":245},{"ids":[34960,5301],"train_count":245},{"ids":[148288,4403],"train_count":245},{"ids":[148823,7358],"train_count":245},{"ids":[149382,220149],"train_count":245},{"ids":[149541,33422],"train_count":245},{"ids":[154145,186469],"train_count":245},{"ids":[220083,159881],"train_count":245},{"ids":[6240,7721,32884],"train_count":122},{"ids":[8789,54025,17941],"train_count":122},{"ids":[12666,13603],"train_count":244},{"ids":[15481,2941,157647],"train_count":122},{"ids":[15481,5092,152148],"train_count":122},{"ids":[23514,36103,148524],"train_count":122},{"ids":[26125,150047],"train_count":244},{"ids":[26125,158668,148917],"train_count":122},{"ids":[26607,45587],"train_count":244},{"ids":[33422,153056,33477],"train_count":122},{"ids":[42317,67958,17941],"train_count":122},{"ids":[42886,148646,150937],"train_count":122},{"ids":[61326,16806],"train_count":244},{"ids":[148314,155812,4096],"train_count":122},{"ids":[148489,18372,4096],"train_count":122},{"ids":[148868,68813,149591],"train_count":122},{"ids":[151014,1403,148524],"train_count":122},{"ids":[157089,148336,15481],"train_count":122},{"ids":[159836,158568],"train_count":244},{"ids":[161775,152973],"train_count":244},{"ids":[185942,148537,151229],"train_count":122},{"ids":[203717,148317],"train_count":244},{"ids":[239142,200641,44633],"train_count":122},{"ids":[16763,42387],"train_count":243},{"ids":[22220,197641],"train_count":243},{"ids":[57272,18768],"train_count":243},{"ids":[57868,148317],"train_count":243},{"ids":[61057,12666],"train_count":243},{"ids":[64535,61057],"train_count":243},{"ids":[77655,148660],"train_count":243},{"ids":[150532,3652],"train_count":243},{"ids":[151250,148293],"train_count":243},{"ids":[153342,4096],"train_count":243},{"ids":[154626,74990],"train_count":243},{"ids":[155611,33437],"train_count":243},{"ids":[157445,7721,77969,6240],"train_count":81},{"ids":[158401,148390,151828,148841],"train_count":81},{"ids":[173286,87018,21785,148554],"train_count":81},{"ids":[204269,10161,1729,156204],"train_count":81},{"ids":[228583,3652],"train_count":243},{"ids":[7721,149098],"train_count":242},{"ids":[7721,151308],"train_count":242},{"ids":[11802,7105],"train_count":242},{"ids":[26149,148281],"train_count":242},{"ids":[28376,16806,42308],"train_count":121},{"ids":[36473,148293],"train_count":242},{"ids":[48989,4640,18110],"train_count":121},{"ids":[58427,45679],"train_count":242},{"ids":[148288,4403,11802],"train_count":121},{"ids":[148380,148277],"train_count":242},{"ids":[148660,16763],"train_count":242},{"ids":[149746,8789,151907],"train_count":121},{"ids":[159042,148397,13739],"train_count":121},{"ids":[170341,7721,2115],"train_count":121},{"ids":[175447,148380],"train_count":242},{"ids":[182670,150198,148317],"train_count":121},{"ids":[193790,150931],"train_count":242},{"ids":[5301,152141],"train_count":241},{"ids":[10496,151666],"train_count":241},{"ids":[16362,21785],"train_count":241},{"ids":[29119,16467],"train_count":241},{"ids":[41635,45679],"train_count":241},{"ids":[53062,148540],"train_count":241},{"ids":[57868,41546],"train_count":241},{"ids":[76953,148606],"train_count":241},{"ids":[148281,37835],"train_count":241},{"ids":[149156,183334],"train_count":241},{"ids":[152143,148453],"train_count":241},{"ids":[158610,148275],"train_count":241},{"ids":[2115,34981,202246],"train_count":120},{"ids":[5241,54724,149591,15481],"train_count":80},{"ids":[7105,148275,149800],"train_count":120},{"ids":[14666,7105],"train_count":240},{"ids":[14942,77436,150621],"train_count":120},{"ids":[17160,14942],"train_count":240},{"ids":[18768,148292,9080],"train_count":120},{"ids":[21785,149075],"train_count":240},{"ids":[22220,39427,150842,1739],"train_count":80},{"ids":[22220,149099,15630],"train_count":120},{"ids":[32868,149086,174558],"train_count":120},{"ids":[35770,85268],"train_count":240},{"ids":[35884,5092,7358,15481],"train_count":80},{"ids":[35884,151709,152973,77436],"train_count":80},{"ids":[42395,28696,7105],"train_count":120},{"ids":[46807,148277,200530],"train_count":120},{"ids":[64535,2115,156454],"train_count":120},{"ids":[148337,148489,18372,4096],"train_count":80},{"ids":[148526,45768],"train_count":240},{"ids":[148700,149800],"train_count":240},{"ids":[149900,87018],"train_count":240},{"ids":[150175,7721,148319,198326],"train_count":80},{"ids":[153944,151011,148779],"train_count":120},{"ids":[156211,10064,16763,149800],"train_count":80},{"ids":[158503,1424],"train_count":240},{"ids":[160530,10064,57868],"train_count":120},{"ids":[160761,173910,1450],"train_count":120},{"ids":[200814,148537],"train_count":240},{"ids":[237773,151937,44633],"train_count":120},{"ids":[11777,29354],"train_count":239},{"ids":[34723,148275],"train_count":239},{"ids":[40794,42387],"train_count":239},{"ids":[74176,5092],"train_count":239},{"ids":[77436,161995],"train_count":239},{"ids":[154019,6240],"train_count":239},{"ids":[2040,12666],"train_count":238},{"ids":[7921,77436],"train_count":238},{"ids":[8688,148292,148277],"train_count":119},{"ids":[8789,7754,148276],"train_count":119},{"ids":[16362,36103],"train_count":238},{"ids":[20177,6733,5301],"train_count":119},{"ids":[26125,148540],"train_count":238},{"ids":[26125,150083],"train_count":238},{"ids":[32868,8789,7754],"train_count":119},{"ids":[34723,149244,149665],"train_count":119},{"ids":[58297,148701],"train_count":238},{"ids":[61757,9396,2040],"train_count":119},{"ids":[148605,26125],"train_count":238},{"ids":[149208,7721,29119],"train_count":119},{"ids":[149269,153870],"train_count":238},{"ids":[150413,149595,16467],"train_count":119},{"ids":[151002,1729],"train_count":238},{"ids":[151221,32401],"train_count":238},{"ids":[152569,173719,36103],"train_count":119},{"ids":[153437,16256],"train_count":238},{"ids":[158666,37835],"train_count":238},{"ids":[170700,148537,16467],"train_count":119},{"ids":[175120,9629],"train_count":238},{"ids":[213334,45527,78330],"train_count":119},{"ids":[223847,7358,42074],"train_count":119},{"ids":[6240,176249],"train_count":237},{"ids":[9080,148283,148634,148648],"train_count":79},{"ids":[34279,17417,2941,54025],"train_count":79},{"ids":[42886,7721,5301,47745],"train_count":79},{"ids":[46807,168823],"train_count":237},{"ids":[61326,148298],"train_count":237},{"ids":[64535,2941,8789,5092],"train_count":79},{"ids":[149156,153462],"train_count":237},{"ids":[150424,18768,148292,148648],"train_count":79},{"ids":[157391,148281,16806,44633],"train_count":79},{"ids":[159133,8688,172518,148275],"train_count":79},{"ids":[200641,149800],"train_count":237},{"ids":[2115,10161],"train_count":236},{"ids":[6389,77436,149537],"train_count":118},{"ids":[7105,2040],"train_count":236},{"ids":[32724,148292,35458],"train_count":118},{"ids":[42317,67958,42565],"train_count":118},{"ids":[47474,36771],"train_count":236},{"ids":[50806,148294],"train_count":236},{"ids":[148427,4640],"train_count":236},{"ids":[148427,25135,172533],"train_count":118},{"ids":[148427,42565],"train_count":236},{"ids":[149097,16806],"train_count":236},{"ids":[149460,19937],"train_count":236},{"ids":[149488,186671],"train_count":236},{"ids":[151201,18372],"train_count":236},{"ids":[151247,152000],"train_count":236},{"ids":[152128,9629,64257],"train_count":118},{"ids":[152155,149089,34723],"train_count":118},{"ids":[168214,148275,148276],"train_count":118},{"ids":[220495,16467,151269],"train_count":118},{"ids":[223847,7358],"train_count":236},{"ids":[7921,148288],"train_count":235},{"ids":[14942,7921],"train_count":235},{"ids":[14942,168329],"train_count":235},{"ids":[42886,65131],"train_count":235},{"ids":[63046,27112],"train_count":235},{"ids":[148318,1729],"train_count":235},{"ids":[149591,151979],"train_count":235},{"ids":[150917,13302],"train_count":235},{"ids":[164597,151385],"train_count":235},{"ids":[182465,156204],"train_count":235},{"ids":[187392,150174],"train_count":235},{"ids":[7422,58475,16763],"train_count":117},{"ids":[11777,17941],"train_count":234},{"ids":[17050,148318,82942],"train_count":117},{"ids":[24653,148277],"train_count":234},{"ids":[34981,7358],"train_count":234},{"ids":[35884,8789],"train_count":234},{"ids":[52687,74990,148518,42317],"train_count":78},{"ids":[64535,29119,186983,150532],"train_count":78},{"ids":[70345,148281,148686,16806],"train_count":78},{"ids":[91016,16669],"train_count":234},{"ids":[148319,149800],"train_count":234},{"ids":[148337,220495,16467,151269],"train_count":78},{"ids":[149233,148288,18372,36916],"train_count":78},{"ids":[149715,68608,47552],"train_count":117},{"ids":[149900,87018,21785,148554],"train_count":78},{"ids":[150194,16467],"train_count":234},{"ids":[150580,14942,160086],"train_count":117},{"ids":[152018,61628],"train_count":234},{"ids":[152839,6240,7721],"train_count":117},{"ids":[153294,1424,74359,16763],"train_count":78},{"ids":[154299,4403],"train_count":234},{"ids":[154726,8789,16806],"train_count":117},{"ids":[155382,16806,53062],"train_count":117},{"ids":[167994,42565,192611],"train_count":117},{"ids":[183479,50800],"train_count":234},{"ids":[198168,42565,21441],"train_count":117},{"ids":[199593,10161,25135],"train_count":117},{"ids":[224036,152417,157759],"train_count":117},{"ids":[1450,9629],"train_count":233},{"ids":[8789,149280],"train_count":233},{"ids":[22220,148319],"train_count":233},{"ids":[26607,15481],"train_count":233},{"ids":[34208,36771],"train_count":233},{"ids":[55302,162292],"train_count":233},{"ids":[57868,153313],"train_count":233},{"ids":[89030,150662],"train_count":233},{"ids":[150175,12014],"train_count":233},{"ids":[163961,6240],"train_count":233},{"ids":[175563,53062],"train_count":233},{"ids":[198063,172332],"train_count":233},{"ids":[201647,149351],"train_count":233},{"ids":[212659,14942],"train_count":233},{"ids":[240350,7358],"train_count":233},{"ids":[5831,11777],"train_count":232},{"ids":[7347,153194],"train_count":232},{"ids":[7958,28696,33422],"train_count":116},{"ids":[13659,148464],"train_count":232},{"ids":[13659,148645],"train_count":232},{"ids":[14942,158638],"train_count":232},{"ids":[18372,153195],"train_count":232},{"ids":[22078,42565,171144],"train_count":116},{"ids":[23514,148275],"train_count":232},{"ids":[36103,150396,41546],"train_count":116},{"ids":[36473,2175],"train_count":232},{"ids":[36473,39605,45527],"train_count":116},{"ids":[46401,13292,152973],"train_count":116},{"ids":[50800,32884,4403],"train_count":116},{"ids":[74353,18372,18110],"train_count":116},{"ids":[89030,4640,31608],"train_count":116},{"ids":[148275,12751],"train_count":232},{"ids":[148326,151446],"train_count":232},{"ids":[148518,50800,32884],"train_count":116},{"ids":[149208,37835],"train_count":232},{"ids":[150413,149595],"train_count":232},{"ids":[150662,12751],"train_count":232},{"ids":[150714,15481,152993],"train_count":116},{"ids":[151078,148366,148298],"train_count":116},{"ids":[151161,152824],"train_count":232},{"ids":[152003,45679,6389],"train_count":116},{"ids":[154285,170744,241144],"train_count":116},{"ids":[154382,8789,7721],"train_count":116},{"ids":[171133,180269],"train_count":232},{"ids":[215384,4096],"train_count":232},{"ids":[2175,36473],"train_count":231},{"ids":[13659,168333],"train_count":231},{"ids":[37519,8688,55638,18372],"train_count":77},{"ids":[42395,152019,2175,149591],"train_count":77},{"ids":[50022,16669],"train_count":231},{"ids":[57272,18768,15630,148275],"train_count":77},{"ids":[76747,21785,152139,170627],"train_count":77},{"ids":[148275,1739],"train_count":231},{"ids":[148275,42317],"train_count":231},{"ids":[148275,148801],"train_count":231},{"ids":[148366,12751],"train_count":231},{"ids":[150365,151666],"train_count":231},{"ids":[150387,33422,153056,60038],"train_count":77},{"ids":[150438,150585],"train_count":231},{"ids":[153589,2115],"train_count":231},{"ids":[153931,6733],"train_count":231},{"ids":[1403,152973],"train_count":230},{"ids":[9080,45679,11796],"train_count":115},{"ids":[13785,10161],"train_count":230},{"ids":[15481,149351],"train_count":230},{"ids":[15630,42565,170968],"train_count":115},{"ids":[19774,162684,151011],"train_count":115},{"ids":[26149,2175,45587],"train_count":115},{"ids":[33207,150270,161907],"train_count":115},{"ids":[89602,18110,2115],"train_count":115},{"ids":[148276,53062],"train_count":230},{"ids":[148362,151993],"train_count":230},{"ids":[148397,148414,172332],"train_count":115},{"ids":[148606,57868],"train_count":230},{"ids":[149509,148377,16763],"train_count":115},{"ids":[150396,32401],"train_count":230},{"ids":[151058,16467,150198],"train_count":115},{"ids":[152009,158518,157348],"train_count":115},{"ids":[154287,151676],"train_count":230},{"ids":[155039,50800,182503],"train_count":115},{"ids":[156069,7921,37835],"train_count":115},{"ids":[179993,4669,149075],"train_count":115},{"ids":[197263,152754,14666],"train_count":115},{"ids":[201826,153277,5301],"train_count":115},{"ids":[209071,7754,47474],"train_count":115},{"ids":[218473,74990],"train_count":230},{"ids":[236400,54201],"train_count":230},{"ids":[3652,1424],"train_count":229},{"ids":[16763,4403],"train_count":229},{"ids":[18727,150838],"train_count":229},{"ids":[18768,148276],"train_count":229},{"ids":[21441,168937],"train_count":229},{"ids":[37483,148331],"train_count":229},{"ids":[66951,1739],"train_count":229},{"ids":[148823,148390],"train_count":229},{"ids":[158702,225639],"train_count":229},{"ids":[160558,81839],"train_count":229},{"ids":[170341,7721],"train_count":229},{"ids":[198529,159497],"train_count":229},{"ids":[212796,47474],"train_count":229},{"ids":[6499,45587,7358],"train_count":114},{"ids":[8789,18768],"train_count":228},{"ids":[9080,7358,150849],"train_count":114},{"ids":[15630,60038],"train_count":228},{"ids":[15729,34981,18768,45679],"train_count":76},{"ids":[16763,149800],"train_count":228},{"ids":[19582,47474,1739,42308],"train_count":76},{"ids":[21720,91206,47474,42565],"train_count":76},{"ids":[24623,153218],"train_count":228},{"ids":[31889,13292],"train_count":228},{"ids":[32392,148377,62709],"train_count":114},{"ids":[35884,22078,150968],"train_count":114},{"ids":[35884,151709,152973],"train_count":114},{"ids":[42565,174192],"train_count":228},{"ids":[46145,149208,7721],"train_count":114},{"ids":[57272,18768,15630],"train_count":114},{"ids":[61326,151978,7754,44432],"train_count":76},{"ids":[148275,45679],"train_count":228},{"ids":[148294,12666,34226],"train_count":114},{"ids":[148309,149208,85208],"train_count":114},{"ids":[148378,6240],"train_count":228},{"ids":[148526,8789],"train_count":228},{"ids":[148526,66951,1739],"train_count":114},{"ids":[148561,7358,7721],"train_count":114},{"ids":[149430,182503,148377,44633],"train_count":76},{"ids":[149943,167601,10750,159031],"train_count":76},{"ids":[150270,198597,19937],"train_count":114},{"ids":[151508,148634,9080,148293],"train_count":76},{"ids":[154408,6240,168329],"train_count":114},{"ids":[154768,22078,150800],"train_count":114},{"ids":[157156,93309],"train_count":228},{"ids":[164597,151385,155012],"train_count":114},{"ids":[166636,12666],"train_count":228},{"ids":[185698,86128,24623],"train_count":114},{"ids":[204723,168062],"train_count":228},{"ids":[15729,148303],"train_count":227},{"ids":[42565,148327],"train_count":227},{"ids":[74990,149574],"train_count":227},{"ids":[148526,148292],"train_count":227},{"ids":[148676,10161],"train_count":227},{"ids":[151744,7721],"train_count":227},{"ids":[153051,13302],"train_count":227},{"ids":[213383,80228],"train_count":227},{"ids":[13302,150938],"train_count":226},{"ids":[15729,148606,15630],"train_count":113},{"ids":[25497,2941],"train_count":226},{"ids":[26125,148317],"train_count":226},{"ids":[32884,148294,7921],"train_count":113},{"ids":[76069,154731,53887],"train_count":113},{"ids":[77509,149099,148336],"train_count":113},{"ids":[148464,64257],"train_count":226},{"ids":[148526,21785],"train_count":226},{"ids":[148535,8789],"train_count":226},{"ids":[151247,154660],"train_count":226},{"ids":[151763,14942,7921],"train_count":113},{"ids":[155453,23514,7105],"train_count":113},{"ids":[173719,36103,42308],"train_count":113},{"ids":[182503,15630,148288],"train_count":113},{"ids":[186408,31889],"train_count":226},{"ids":[187197,159075],"train_count":226},{"ids":[190964,7358,151302],"train_count":113},{"ids":[196219,201647],"train_count":226},{"ids":[201826,153277],"train_count":226},{"ids":[230253,148275],"train_count":226},{"ids":[2040,54025,6733,75896],"train_count":75},{"ids":[2115,149097],"train_count":225},{"ids":[5241,149147,16763,42387],"train_count":75},{"ids":[5241,150800,177067,150584],"train_count":75},{"ids":[10161,25135,4640,159142],"train_count":75},{"ids":[13739,9080],"train_count":225},{"ids":[14942,156977,149095,8789],"train_count":75},{"ids":[14942,159079],"train_count":225},{"ids":[20177,2040,54025,6733],"train_count":75},{"ids":[33638,150853],"train_count":225},{"ids":[36736,18372,32884,36747],"train_count":75},{"ids":[38859,150348,14942,149827],"train_count":75},{"ids":[42565,82942],"train_count":225},{"ids":[148526,26125],"train_count":225},{"ids":[149095,5092],"train_count":225},{"ids":[150387,33422,153056,34226],"train_count":75},{"ids":[150842,11796],"train_count":225},{"ids":[154758,4096],"train_count":225},{"ids":[154849,26125,12666,41546],"train_count":75},{"ids":[211778,22078,2941,54025],"train_count":75},{"ids":[233244,148275],"train_count":225},{"ids":[6389,36916],"train_count":224},{"ids":[10064,150621],"train_count":224},{"ids":[13659,48989,37855],"train_count":112},{"ids":[15729,45587],"train_count":224},{"ids":[19736,89527,151946],"train_count":112},{"ids":[22078,148301,148811],"train_count":112},{"ids":[26607,5301,148280],"train_count":112},{"ids":[53887,21785,148554],"train_count":112},{"ids":[148283,148634],"train_count":224},{"ids":[148314,148634,148336],"train_count":112},{"ids":[148326,151446,1450],"train_count":112},{"ids":[148327,148319],"train_count":224},{"ids":[148496,9396,148336],"train_count":112},{"ids":[148782,93834,44633],"train_count":112},{"ids":[149156,153462,2941],"train_count":112},{"ids":[150029,1403,181923],"train_count":112},{"ids":[152143,148453,15481],"train_count":112},{"ids":[152731,148942,155055],"train_count":112},{"ids":[153342,11796,149800],"train_count":112},{"ids":[171011,28696,154800],"train_count":112},{"ids":[177153,60659,148293],"train_count":112},{"ids":[180371,79142],"train_count":224},{"ids":[184702,10750,76069],"train_count":112},{"ids":[223134,38183,210740],"train_count":112},{"ids":[237072,2040],"train_count":224},{"ids":[237773,151937],"train_count":224},{"ids":[11777,2941],"train_count":223},{"ids":[26125,148803],"train_count":223},{"ids":[61326,7721],"train_count":223},{"ids":[71000,7105],"train_count":223},{"ids":[148292,45587],"train_count":223},{"ids":[149591,148841],"train_count":223},{"ids":[150479,33422],"train_count":223},{"ids":[150559,44633],"train_count":223},{"ids":[154729,155131],"train_count":223},{"ids":[155018,148336],"train_count":223},{"ids":[155132,175740],"train_count":223},{"ids":[194771,42565],"train_count":223},{"ids":[206746,11577],"train_count":223},{"ids":[243223,154800],"train_count":223},{"ids":[4901,61324,87018],"train_count":111},{"ids":[5301,27448,26125],"train_count":111},{"ids":[7721,2115,26125,64903],"train_count":74},{"ids":[8688,172518],"train_count":222},{"ids":[15729,23514],"train_count":222},{"ids":[16362,58255,236525],"train_count":111},{"ids":[17160,148729,13302],"train_count":111},{"ids":[18372,148281],"train_count":222},{"ids":[26149,21785,11777,5092],"train_count":74},{"ids":[30206,3652,29354],"train_count":111},{"ids":[42565,163347],"train_count":222},{"ids":[56755,61057],"train_count":222},{"ids":[64535,30206,3652,29354],"train_count":74},{"ids":[77655,6733,148972],"train_count":111},{"ids":[84400,84475,7721],"train_count":111},{"ids":[86949,85208],"train_count":222},{"ids":[93834,22078,160162,16256],"train_count":74},{"ids":[148280,77779,15630,4403],"train_count":74},{"ids":[148397,148414,148660],"train_count":111},{"ids":[148427,47174,148327,148277],"train_count":74},{"ids":[148823,34723],"train_count":222},{"ids":[148920,148281,9396],"train_count":111},{"ids":[149147,16763,37835],"train_count":111},{"ids":[149413,7358],"train_count":222},{"ids":[150175,7721,148319],"train_count":111},{"ids":[150387,33422,153056,48933],"train_count":74},{"ids":[154849,39427,149497],"train_count":111},{"ids":[156242,64257,148380],"train_count":111},{"ids":[159042,23514,148327],"train_count":111},{"ids":[161308,149665],"train_count":222},{"ids":[165263,50022],"train_count":222},{"ids":[219481,233559],"train_count":222},{"ids":[239300,150947,7105,1729],"train_count":74},{"ids":[7422,15481],"train_count":221},{"ids":[34208,148275],"train_count":221},{"ids":[38859,150348],"train_count":221},{"ids":[38859,182867],"train_count":221},{"ids":[42387,149280],"train_count":221},{"ids":[52687,152238],"train_count":221},{"ids":[54858,38183],"train_count":221},{"ids":[58427,148292],"train_count":221},{"ids":[65131,29119],"train_count":221},{"ids":[149009,152236],"train_count":221},{"ids":[149958,148293],"train_count":221},{"ids":[150387,149137],"train_count":221},{"ids":[150813,149665],"train_count":221},{"ids":[151078,148366],"train_count":221},{"ids":[152973,158585],"train_count":221},{"ids":[152974,42387],"train_count":221},{"ids":[200496,2040],"train_count":221},{"ids":[4901,150800],"train_count":220},{"ids":[6389,77436,150621],"train_count":110},{"ids":[7721,160115,32884],"train_count":110},{"ids":[8789,6733],"train_count":220},{"ids":[14942,9080],"train_count":220},{"ids":[23514,7105,54201],"train_count":110},{"ids":[26125,1450],"train_count":220},{"ids":[26607,61057,148377],"train_count":110},{"ids":[50563,55638],"train_count":220},{"ids":[148294,7921,148275],"train_count":110},{"ids":[149092,15630],"train_count":220},{"ids":[149796,13603,6389],"train_count":110},{"ids":[150237,148388,71221],"train_count":110},{"ids":[151146,167664,149193],"train_count":110},{"ids":[151277,61057],"train_count":220},{"ids":[152946,12666,148377],"train_count":110},{"ids":[154297,1403],"train_count":220},{"ids":[154691,10064],"train_count":220},{"ids":[158013,36771,45480],"train_count":110},{"ids":[160847,170946,148317],"train_count":110},{"ids":[167834,22078,148318],"train_count":110},{"ids":[181841,148275],"train_count":220},{"ids":[181841,148275,31889],"train_count":110},{"ids":[185252,12014],"train_count":220},{"ids":[188583,162127],"train_count":220},{"ids":[198620,148569],"train_count":220},{"ids":[217720,56090],"train_count":220},{"ids":[221634,18110],"train_count":220},{"ids":[226960,152833],"train_count":220},{"ids":[6733,194972],"train_count":219},{"ids":[10161,25135],"train_count":219},{"ids":[21785,149012],"train_count":219},{"ids":[24653,150396],"train_count":219},{"ids":[29189,161308],"train_count":219},{"ids":[33065,95525],"train_count":219},{"ids":[36771,11577],"train_count":219},{"ids":[47745,172637],"train_count":219},{"ids":[52687,12666,200641,149800],"train_count":73},{"ids":[58297,53555,148380,34981],"train_count":73},{"ids":[77655,6733,148972,6733],"train_count":73},{"ids":[148292,15481,7358,44633],"train_count":73},{"ids":[148873,148489,18372,15630],"train_count":73},{"ids":[149644,2040],"train_count":219},{"ids":[152179,7721,2941,151281],"train_count":73},{"ids":[153437,150577],"train_count":219},{"ids":[156211,10064,16763,37835],"train_count":73},{"ids":[156427,21918],"train_count":219},{"ids":[162683,148280,7921,16806],"train_count":73},{"ids":[4901,4669,157815],"train_count":109},{"ids":[7358,151302],"train_count":218},{"ids":[12696,14666,6389],"train_count":109},{"ids":[15729,45587,4403],"train_count":109},{"ids":[19774,89527,18110],"train_count":109},{"ids":[36473,39605],"train_count":218},{"ids":[36771,6733],"train_count":218},{"ids":[38859,66951,197362],"train_count":109},{"ids":[40794,64392],"train_count":218},{"ids":[42886,148646],"train_count":218},{"ids":[45768,148275],"train_count":218},{"ids":[61757,31889,42308],"train_count":109},{"ids":[63890,4403],"train_count":218},{"ids":[77969,5301,149193],"train_count":109},{"ids":[82557,15481],"train_count":218},{"ids":[82929,27473],"train_count":218},{"ids":[89527,18110,18372],"train_count":109},{"ids":[93607,148489,7921],"train_count":109},{"ids":[148281,149900,152973],"train_count":109},{"ids":[149454,150140,36771],"train_count":109},{"ids":[150926,151032,150396],"train_count":109},{"ids":[151146,158702],"train_count":218},{"ids":[156204,1729],"train_count":218},{"ids":[157746,76069,154731],"train_count":109},{"ids":[158291,152898,149005],"train_count":109},{"ids":[158466,32401],"train_count":218},{"ids":[165331,15481],"train_count":218},{"ids":[166592,149095,57868],"train_count":109},{"ids":[3652,13785],"train_count":217},{"ids":[22346,149962],"train_count":217},{"ids":[29196,36771],"train_count":217},{"ids":[32884,12751],"train_count":217},{"ids":[36736,7721],"train_count":217},{"ids":[148561,16763],"train_count":217},{"ids":[149413,12666],"train_count":217},{"ids":[149718,150396],"train_count":217},{"ids":[150849,14942],"train_count":217},{"ids":[152179,148378],"train_count":217},{"ids":[153194,9629],"train_count":217},{"ids":[164590,151713],"train_count":217},{"ids":[178260,170627],"train_count":217},{"ids":[7754,84920,55055,41546],"train_count":72},{"ids":[7958,28696,151979],"train_count":108},{"ids":[13659,148518,2175,42565],"train_count":72},{"ids":[15729,154285,170744],"train_count":108},{"ids":[16362,21785,21918],"train_count":108},{"ids":[26149,44432,10750],"train_count":108},{"ids":[26607,151223],"train_count":216},{"ids":[28696,33422,1424],"train_count":108},{"ids":[29189,2175],"train_count":216},{"ids":[36103,12751],"train_count":216},{"ids":[41381,149095,7105,2040],"train_count":72},{"ids":[42886,7721,149098],"train_count":108},{"ids":[47552,28696,45527,148414],"train_count":72},{"ids":[58297,11777,8789,41635],"train_count":72},{"ids":[87018,21785,149012],"train_count":108},{"ids":[94491,7358,15481],"train_count":108},{"ids":[148292,150366],"train_count":216},{"ids":[148782,16763,27112,8688],"train_count":72},{"ids":[148873,6389,25767],"train_count":108},{"ids":[149344,9080,77969,6240],"train_count":72},{"ids":[149563,62847,29119],"train_count":108},{"ids":[149591,7105,42308],"train_count":108},{"ids":[150398,149037,148685],"train_count":108},{"ids":[151232,59627,71658,54201],"train_count":72},{"ids":[151639,7754,148276,77436],"train_count":72},{"ids":[151853,1739,7721],"train_count":108},{"ids":[154732,32884],"train_count":216},{"ids":[155164,16806,45587,148275],"train_count":72},{"ids":[162564,4640,34981],"train_count":108},{"ids":[163765,148841],"train_count":216},{"ids":[166266,150479,21785],"train_count":108},{"ids":[171133,10064,158732],"train_count":108},{"ids":[173980,53555,5092,2175],"train_count":72},{"ids":[178874,159069],"train_count":216},{"ids":[181834,151896,41546],"train_count":108},{"ids":[227058,6733],"train_count":216},{"ids":[1739,149352],"train_count":215},{"ids":[8789,151308],"train_count":215},{"ids":[16806,44633],"train_count":215},{"ids":[22078,150968],"train_count":215},{"ids":[26125,41546],"train_count":215},{"ids":[31889,41546],"train_count":215},{"ids":[38859,148700],"train_count":215},{"ids":[45768,7921],"train_count":215},{"ids":[61326,74990],"train_count":215},{"ids":[77509,149099],"train_count":215},{"ids":[149962,7921],"train_count":215},{"ids":[153501,48046],"train_count":215},{"ids":[186643,149075],"train_count":215},{"ids":[187392,61057],"train_count":215},{"ids":[4640,34981,149633],"train_count":107},{"ids":[5624,23514,148518],"train_count":107},{"ids":[7358,1403],"train_count":214},{"ids":[7422,18372,149744],"train_count":107},{"ids":[14942,45768,42074],"train_count":107},{"ids":[16763,155374,84706],"train_count":107},{"ids":[22346,151408,17941],"train_count":107},{"ids":[23947,7921],"train_count":214},{"ids":[23947,148676,10161],"train_count":107},{"ids":[25497,36747,155298],"train_count":107},{"ids":[33422,1424,159488],"train_count":107},{"ids":[33422,153056,60038],"train_count":107},{"ids":[39427,1424],"train_count":214},{"ids":[42565,11796],"train_count":214},{"ids":[53887,148317],"train_count":214},{"ids":[56755,148294],"train_count":214},{"ids":[148634,148336],"train_count":214},{"ids":[149147,16763,42387],"train_count":107},{"ids":[151775,162292],"train_count":214},{"ids":[151853,148378],"train_count":214},{"ids":[152973,77436,153232],"train_count":107},{"ids":[155066,149193],"train_count":214},{"ids":[155651,18110,149591],"train_count":107},{"ids":[157391,4640,156211],"train_count":107},{"ids":[158216,42387,89771],"train_count":107},{"ids":[165498,221151],"train_count":214},{"ids":[190730,148284],"train_count":214},{"ids":[204723,168062,170139],"train_count":107},{"ids":[220232,11802],"train_count":214},{"ids":[239300,9629],"train_count":214},{"ids":[1403,13292],"train_count":213},{"ids":[1424,155099],"train_count":213},{"ids":[4403,11802],"train_count":213},{"ids":[5092,148540],"train_count":213},{"ids":[5624,77779,16763,4669],"train_count":71},{"ids":[6240,185709],"train_count":213},{"ids":[16362,37483],"train_count":213},{"ids":[18768,148318],"train_count":213},{"ids":[19774,170759,1403,45587],"train_count":71},{"ids":[36771,149195],"train_count":213},{"ids":[51844,166844,152950,37835],"train_count":71},{"ids":[150155,148405,21918,6389],"train_count":71},{"ids":[150926,151032,42565,148779],"train_count":71},{"ids":[151945,16763,42074,89315],"train_count":71},{"ids":[155548,172332],"train_count":213},{"ids":[156181,4640],"train_count":213},{"ids":[159541,5301,55389,44633],"train_count":71},{"ids":[159806,13113],"train_count":213},{"ids":[179837,6733,40794,148702],"train_count":71},{"ids":[179993,4669],"train_count":213},{"ids":[1424,74359,16763],"train_count":106},{"ids":[4640,34981,7105],"train_count":106},{"ids":[8789,150156],"train_count":212},{"ids":[16256,1739],"train_count":212},{"ids":[20177,66951,148381],"train_count":106},{"ids":[21918,155223],"train_count":212},{"ids":[25497,148761,13292],"train_count":106},{"ids":[26149,7358,33422],"train_count":106},{"ids":[26149,34981,148283],"train_count":106},{"ids":[32392,7105],"train_count":212},{"ids":[46401,13292],"train_count":212},{"ids":[48989,1424],"train_count":212},{"ids":[61326,33437],"train_count":212},{"ids":[77655,159489,151639],"train_count":106},{"ids":[148276,148275],"train_count":212},{"ids":[149085,148841],"train_count":212},{"ids":[153899,14666],"train_count":212},{"ids":[209521,6733,1729],"train_count":106},{"ids":[2115,8789],"train_count":211},{"ids":[7358,55389],"train_count":211},{"ids":[15481,149095],"train_count":211},{"ids":[23947,9080],"train_count":211},{"ids":[33065,38183],"train_count":211},{"ids":[33207,150270],"train_count":211},{"ids":[34208,148561],"train_count":211},{"ids":[36103,45480],"train_count":211},{"ids":[37835,10161],"train_count":211},{"ids":[72884,44633],"train_count":211},{"ids":[149796,13603],"train_count":211},{"ids":[151945,16763],"train_count":211},{"ids":[156476,152141],"train_count":211},{"ids":[241330,151828],"train_count":211},{"ids":[2040,60659],"train_count":210},{"ids":[7105,14942],"train_count":210},{"ids":[7721,2115,26125,26046],"train_count":70},{"ids":[10983,176068,159236],"train_count":105},{"ids":[11929,21785,149595,148275],"train_count":70},{"ids":[15630,5092,16669],"train_count":105},{"ids":[15630,42565,171935],"train_count":105},{"ids":[18110,1450],"train_count":210},{"ids":[19736,152209],"train_count":210},{"ids":[20289,61628,228174],"train_count":105},{"ids":[23514,18372],"train_count":210},{"ids":[23947,16806,16256],"train_count":105},{"ids":[28696,15481],"train_count":210},{"ids":[39625,6733,58475,29119],"train_count":70},{"ids":[48623,6733,10064],"train_count":105},{"ids":[51844,148288,4403,11802],"train_count":70},{"ids":[54724,4403],"train_count":210},{"ids":[70667,48623,6733],"train_count":105},{"ids":[89030,162151,215816],"train_count":105},{"ids":[148292,150083],"train_count":210},{"ids":[148314,148634,148336,67494],"train_count":70},{"ids":[148606,17941],"train_count":210},{"ids":[149095,61057,42565],"train_count":105},{"ids":[149560,6240,42387],"train_count":105},{"ids":[150175,173049],"train_count":210},{"ids":[150828,42317,67958,185709],"train_count":70},{"ids":[153306,2175,47174,148275],"train_count":70},{"ids":[154424,45587],"train_count":210},{"ids":[155152,7754,42317],"train_count":105},{"ids":[155382,16806],"train_count":210},{"ids":[158121,2040],"train_count":210},{"ids":[160877,29354],"train_count":210},{"ids":[173910,4403],"train_count":210},{"ids":[198528,56090],"train_count":210},{"ids":[242409,151397],"train_count":210},{"ids":[6733,2175],"train_count":209},{"ids":[7358,152027],"train_count":209},{"ids":[19582,16763],"train_count":209},{"ids":[23355,151979],"train_count":209},{"ids":[26125,150281],"train_count":209},{"ids":[42395,202031],"train_count":209},{"ids":[45763,151666],"train_count":209},{"ids":[60659,148275],"train_count":209},{"ids":[148303,6733],"train_count":209},{"ids":[150441,151642],"train_count":209},{"ids":[150853,16763],"train_count":209},{"ids":[152959,68608],"train_count":209},{"ids":[155726,61057],"train_count":209},{"ids":[162280,14942],"train_count":209},{"ids":[173890,29354],"train_count":209},{"ids":[3652,29354],"train_count":208},{"ids":[4901,74990],"train_count":208},{"ids":[7422,47552,5301],"train_count":104},{"ids":[10064,16763,149800],"train_count":104},{"ids":[14942,45679,1739],"train_count":104},{"ids":[21720,19160,172340],"train_count":104},{"ids":[22220,25135,9629],"train_count":104},{"ids":[23514,36103,32401],"train_count":104},{"ids":[32482,66694,7105],"train_count":104},{"ids":[33422,153056,48933],"train_count":104},{"ids":[34723,149244,67494],"train_count":104},{"ids":[42317,67958,154838],"train_count":104},{"ids":[48989,21918],"train_count":208},{"ids":[54858,21441],"train_count":208},{"ids":[57272,178537,45405],"train_count":104},{"ids":[148397,177720],"train_count":208},{"ids":[149454,150140,231448],"train_count":104},{"ids":[149490,10064,90202],"train_count":104},{"ids":[150359,9396],"train_count":208},{"ids":[150425,14666,7105],"train_count":104},{"ids":[151078,154731,92647],"train_count":104},{"ids":[153618,6733],"train_count":208},{"ids":[154726,219129,5092],"train_count":104},{"ids":[156476,153195],"train_count":208},{"ids":[157695,26125],"train_count":208},{"ids":[176201,155195],"train_count":208},{"ids":[198576,77779],"train_count":208},{"ids":[205914,171730],"train_count":208},{"ids":[1739,149053],"train_count":207},{"ids":[4640,149900,87018,21785],"train_count":69},{"ids":[7754,148464,6733,194972],"train_count":69},{"ids":[12666,34981],"train_count":207},{"ids":[16763,38183],"train_count":207},{"ids":[16806,42308],"train_count":207},{"ids":[18727,5301,148280,148277],"train_count":69},{"ids":[26125,171149],"train_count":207},{"ids":[48941,11777],"train_count":207},{"ids":[84400,84475],"train_count":207},{"ids":[148319,149280],"train_count":207},{"ids":[149282,36621,9080,148293],"train_count":69},{"ids":[151183,148776],"train_count":207},{"ids":[160530,10064],"train_count":207},{"ids":[162684,151011],"train_count":207},{"ids":[164184,1403,58805,17941],"train_count":69},{"ids":[2115,26125,15481],"train_count":103},{"ids":[2941,149519,11577],"train_count":103},{"ids":[7105,13113,29354],"train_count":103},{"ids":[13113,58361,58245],"train_count":103},{"ids":[17569,34723],"train_count":206},{"ids":[18110,148288],"train_count":206},{"ids":[18727,150270,198597],"train_count":103},{"ids":[22078,1450],"train_count":206},{"ids":[23947,15481,148579],"train_count":103},{"ids":[25497,148294,42565],"train_count":103},{"ids":[26607,2941,54025],"train_count":103},{"ids":[33638,148280],"train_count":206},{"ids":[39625,6733],"train_count":206},{"ids":[52687,148390],"train_count":206},{"ids":[57272,178537],"train_count":206},{"ids":[68553,40794,48623],"train_count":103},{"ids":[74990,15481,19937],"train_count":103},{"ids":[148288,22078,148301],"train_count":103},{"ids":[149054,10161],"train_count":206},{"ids":[149715,148275],"train_count":206},{"ids":[150580,14942],"train_count":206},{"ids":[151032,42565,42074],"train_count":103},{"ids":[152135,6240],"train_count":206},{"ids":[154326,6389,17417],"train_count":103},{"ids":[156234,21918,148762],"train_count":103},{"ids":[158291,152898,7921],"train_count":103},{"ids":[161159,16362],"train_count":206},{"ids":[162292,16256],"train_count":206},{"ids":[220495,16467],"train_count":206},{"ids":[1739,148366],"train_count":205},{"ids":[7550,61324],"train_count":205},{"ids":[9080,45679],"train_count":205},{"ids":[21785,37855],"train_count":205},{"ids":[22078,148275],"train_count":205},{"ids":[54858,10161],"train_count":205},{"ids":[68813,149591],"train_count":205},{"ids":[148453,15481],"train_count":205},{"ids":[153277,5301],"train_count":205},{"ids":[155039,10161],"train_count":205},{"ids":[158568,42387],"train_count":205},{"ids":[225679,55389],"train_count":205},{"ids":[6733,2040,185709],"train_count":102},{"ids":[7754,148303],"train_count":204},{"ids":[7921,37835],"train_count":204},{"ids":[8789,54025,5092],"train_count":102},{"ids":[8789,150156,44633],"train_count":102},{"ids":[13659,12666,13603,4640],"train_count":68},{"ids":[16362,22078,11777,17941],"train_count":68},{"ids":[20578,9080],"train_count":204},{"ids":[21720,19160],"train_count":204},{"ids":[22078,59627,148275,31889],"train_count":68},{"ids":[22346,151408],"train_count":204},{"ids":[24623,13292],"train_count":204},{"ids":[24653,149095],"train_count":204},{"ids":[32724,155812,17941],"train_count":102},{"ids":[33638,30206,72884],"train_count":102},{"ids":[34208,148319],"train_count":204},{"ids":[36103,148524],"train_count":204},{"ids":[39427,148281,148524,8789],"train_count":68},{"ids":[43800,5301,13302,153174],"train_count":68},{"ids":[53241,5301],"train_count":204},{"ids":[61057,42565,5301],"train_count":102},{"ids":[87148,33422,92647],"train_count":102},{"ids":[89527,18110,18372,149800],"train_count":68},{"ids":[148288,1403,13292,149297],"train_count":68},{"ids":[148464,6733,2040],"train_count":102},{"ids":[148561,16806,15630,4403],"train_count":68},{"ids":[149140,148675,15481],"train_count":102},{"ids":[149208,55389,148288],"train_count":102},{"ids":[149269,8789,54025,5092],"train_count":68},{"ids":[149572,77436],"train_count":204},{"ids":[149589,156454,10750],"train_count":102},{"ids":[149679,150469],"train_count":204},{"ids":[149943,167601,10750,150006],"train_count":68},{"ids":[150175,213134,1403],"train_count":102},{"ids":[150325,9080],"train_count":204},{"ids":[150387,33422,153056,148293],"train_count":68},{"ids":[151078,148366,149885],"train_count":102},{"ids":[152569,173719],"train_count":204},{"ids":[154178,2115,23514],"train_count":102},{"ids":[154285,170744,150757,148369],"train_count":68},{"ids":[154731,90202],"train_count":204},{"ids":[158013,15481,9080,7921],"train_count":68},{"ids":[5092,7358],"train_count":203},{"ids":[7105,153195],"train_count":203},{"ids":[7358,42074],"train_count":203},{"ids":[13113,29354],"train_count":203},{"ids":[16763,148277],"train_count":203},{"ids":[34208,153935],"train_count":203},{"ids":[35884,61628],"train_count":203},{"ids":[41381,14942],"train_count":203},{"ids":[48771,27448],"train_count":203},{"ids":[66050,14942],"train_count":203},{"ids":[148281,16806],"train_count":203},{"ids":[148281,149900],"train_count":203},{"ids":[148557,15481],"train_count":203},{"ids":[150599,6240],"train_count":203},{"ids":[151001,14942],"train_count":203},{"ids":[151002,42074],"train_count":203},{"ids":[156110,12751],"train_count":203},{"ids":[174856,149280],"train_count":203},{"ids":[178595,26125],"train_count":203},{"ids":[201826,156685],"train_count":203},{"ids":[232069,66694],"train_count":203},{"ids":[2941,8789,45587],"train_count":101},{"ids":[5092,9080],"train_count":202},{"ids":[5241,34981,36747],"train_count":101},{"ids":[6733,10064,7721],"train_count":101},{"ids":[8688,55638,18372],"train_count":101},{"ids":[10064,17460],"train_count":202},{"ids":[14942,45679],"train_count":202},{"ids":[15630,74990],"train_count":202},{"ids":[18372,152141],"train_count":202},{"ids":[19736,2175],"train_count":202},{"ids":[21720,148518],"train_count":202},{"ids":[21785,54330],"train_count":202},{"ids":[21918,6389,158585],"train_count":101},{"ids":[24653,148380,33477],"train_count":101},{"ids":[25497,21918,148369],"train_count":101},{"ids":[34208,148390],"train_count":202},{"ids":[45587,1450],"train_count":202},{"ids":[55302,45587],"train_count":202},{"ids":[95169,148380,148277],"train_count":101},{"ids":[95169,159061],"train_count":202},{"ids":[148294,16806,5831],"train_count":101},{"ids":[148303,9080,148676],"train_count":101},{"ids":[148377,62709,149351],"train_count":101},{"ids":[150479,21785,5092],"train_count":101},{"ids":[151784,165444,19937],"train_count":101},{"ids":[152024,36916],"train_count":202},{"ids":[152628,5831,149961],"train_count":101},{"ids":[153051,27112],"train_count":202},{"ids":[155548,95525],"train_count":202},{"ids":[158693,36183,1450],"train_count":101},{"ids":[159222,155421],"train_count":202},{"ids":[162546,90202,1450],"train_count":101},{"ids":[168686,22078,148569],"train_count":101},{"ids":[174005,5301],"train_count":202},{"ids":[192416,161978],"train_count":202},{"ids":[194731,50800],"train_count":202},{"ids":[199160,36103,37835],"train_count":101},{"ids":[206488,182503],"train_count":202},{"ids":[220630,36771,71221],"train_count":101},{"ids":[224775,1450],"train_count":202},{"ids":[1739,7105],"train_count":201},{"ids":[4669,42565],"train_count":201},{"ids":[6240,15481],"train_count":201},{"ids":[7105,54201],"train_count":201},{"ids":[15729,22078],"train_count":201},{"ids":[18372,148281,149900,148510],"train_count":67},{"ids":[20177,6733,5301,149601],"train_count":67},{"ids":[32392,148277,61057,152993],"train_count":67},{"ids":[33422,94309],"train_count":201},{"ids":[39243,7754,148464,6733],"train_count":67},{"ids":[42395,148700,68608,15630],"train_count":67},{"ids":[52687,74990,148518,31953],"train_count":67},{"ids":[55302,42565],"train_count":201},{"ids":[68553,40794,5092,32401],"train_count":67},{"ids":[77873,154575,15481,18110],"train_count":67},{"ids":[89527,18110],"train_count":201},{"ids":[148510,95525],"train_count":201},{"ids":[149348,150195],"train_count":201},{"ids":[149589,7721,32884,71658],"train_count":67},{"ids":[153987,149280],"train_count":201},{"ids":[155544,4640,63046,27112],"train_count":67},{"ids":[173890,37483],"train_count":201},{"ids":[199160,36103],"train_count":201},{"ids":[224036,152417],"train_count":201},{"ids":[234771,148518],"train_count":201},{"ids":[1739,51788],"train_count":200},{"ids":[1739,92647],"train_count":200},{"ids":[7754,148464,6733],"train_count":100},{"ids":[10161,149926,153174],"train_count":100},{"ids":[10496,9080,172199],"train_count":100},{"ids":[16467,21785],"train_count":200},{"ids":[16797,44432,149800],"train_count":100},{"ids":[19582,47474,150182],"train_count":100},{"ids":[20289,205175,54201],"train_count":100},{"ids":[23355,62215,42317],"train_count":100},{"ids":[26125,223999],"train_count":200},{"ids":[31889,32401],"train_count":200},{"ids":[38859,148397,36473],"train_count":100},{"ids":[39596,42565,34083],"train_count":100},{"ids":[47745,90219],"train_count":200},{"ids":[50806,36771,10064],"train_count":100},{"ids":[56145,149068],"train_count":200},{"ids":[58297,7358],"train_count":200},{"ids":[62847,29119,7754],"train_count":100},{"ids":[69134,149741,149591],"train_count":100},{"ids":[95169,153156],"train_count":200},{"ids":[148425,148705],"train_count":200},{"ids":[149591,15481,152993],"train_count":100},{"ids":[149715,168906],"train_count":200},{"ids":[149746,8789,158638],"train_count":100},{"ids":[151029,6240,155044],"train_count":100},{"ids":[151183,2040],"train_count":200},{"ids":[151963,218943],"train_count":200},{"ids":[152568,171636],"train_count":200},{"ids":[154849,77969,8688],"train_count":100},{"ids":[155032,148341,155055],"train_count":100},{"ids":[159101,149586,168906],"train_count":100},{"ids":[190964,149578,42387],"train_count":100},{"ids":[236006,149193],"train_count":200},{"ids":[7105,82942],"train_count":199},{"ids":[7358,33422],"train_count":199},{"ids":[7358,44633],"train_count":199},{"ids":[8789,53887],"train_count":199},{"ids":[10496,218814],"train_count":199},{"ids":[10983,151196],"train_count":199},{"ids":[18372,170391],"train_count":199},{"ids":[22220,39427],"train_count":199},{"ids":[53887,150198],"train_count":199},{"ids":[54858,89527],"train_count":199},{"ids":[54858,155898],"train_count":199},{"ids":[150757,1729],"train_count":199},{"ids":[151014,1403],"train_count":199},{"ids":[159042,23514],"train_count":199},{"ids":[163776,2040],"train_count":199},{"ids":[180003,149127],"train_count":199},{"ids":[181834,151896],"train_count":199},{"ids":[191183,159796],"train_count":199},{"ids":[208002,18110],"train_count":199},{"ids":[3652,35458],"train_count":198},{"ids":[4669,21785,4403],"train_count":99},{"ids":[5241,21441,150307],"train_count":99},{"ids":[5241,54724,149591,7105],"train_count":66},{"ids":[5301,42565],"train_count":198},{"ids":[5624,150425,14666],"train_count":99},{"ids":[6733,58475,29119],"train_count":99},{"ids":[7358,148298],"train_count":198},{"ids":[8688,13113],"train_count":198},{"ids":[10567,2115],"train_count":198},{"ids":[12696,14666,6389,149193],"train_count":66},{"ids":[16362,158537],"train_count":198},{"ids":[18096,149340,53062],"train_count":99},{"ids":[18727,32482,151666,36103],"train_count":66},{"ids":[18727,158689,152149],"train_count":99},{"ids":[18768,157146,6389],"train_count":99},{"ids":[23947,16806,50800],"train_count":99},{"ids":[27448,9396],"train_count":198},{"ids":[32482,151666,36103,42308],"train_count":66},{"ids":[32884,77779,26125,42240],"train_count":66},{"ids":[32884,148288],"train_count":198},{"ids":[33422,1424,155099],"train_count":99},{"ids":[34279,5301,42565],"train_count":99},{"ids":[42395,28696,15481],"train_count":99},{"ids":[54858,158401],"train_count":198},{"ids":[58427,7105],"train_count":198},{"ids":[77969,8688],"train_count":198},{"ids":[78330,26125],"train_count":198},{"ids":[85208,14942],"train_count":198},{"ids":[148292,15481,172094],"train_count":99},{"ids":[148294,16806,5831,58503],"train_count":66},{"ids":[148298,7754,14666],"train_count":99},{"ids":[148303,6733,149499],"train_count":99},{"ids":[148526,45768,154800],"train_count":99},{"ids":[148782,93834,22078],"train_count":99},{"ids":[149146,149538,148558,72884],"train_count":66},{"ids":[149269,8789,54025,80228],"train_count":66},{"ids":[149644,148336,148284,148331],"train_count":66},{"ids":[150466,150281],"train_count":198},{"ids":[152674,148343,185709],"train_count":99},{"ids":[153020,148276,9080],"train_count":99},{"ids":[153298,13113,4403],"train_count":99},{"ids":[155132,175740,2941],"train_count":99},{"ids":[156181,44633],"train_count":198},{"ids":[164085,148510,95525],"train_count":99},{"ids":[172753,152999,49224],"train_count":99},{"ids":[173218,42565,11796],"train_count":99},{"ids":[187134,47745,5301,170391],"train_count":66},{"ids":[205606,7105,2040],"train_count":99},{"ids":[218293,18372,170391],"train_count":99},{"ids":[225736,148524],"train_count":198},{"ids":[226013,158668],"train_count":198},{"ids":[236487,7358],"train_count":198},{"ids":[4640,148761],"train_count":197},{"ids":[21785,148797],"train_count":197},{"ids":[23947,149827],"train_count":197},{"ids":[34960,11796],"train_count":197},{"ids":[43080,9080],"train_count":197},{"ids":[58427,41635],"train_count":197},{"ids":[61757,9396],"train_count":197},{"ids":[74990,12666],"train_count":197},{"ids":[90202,1450],"train_count":197},{"ids":[148336,151186],"train_count":197},{"ids":[148380,48933],"train_count":197},{"ids":[148526,22078],"train_count":197},{"ids":[154285,170744],"train_count":197},{"ids":[163961,201286],"train_count":197},{"ids":[167797,7721],"train_count":197},{"ids":[187197,155045],"train_count":197},{"ids":[208544,16763],"train_count":197},{"ids":[211593,44633],"train_count":197},{"ids":[218574,174005],"train_count":197},{"ids":[4901,80378],"train_count":196},{"ids":[7105,150584],"train_count":196},{"ids":[9629,2941],"train_count":196},{"ids":[10496,151666,13292],"train_count":98},{"ids":[12666,12666],"train_count":196},{"ids":[14942,148972,159712],"train_count":98},{"ids":[16467,156204],"train_count":196},{"ids":[20177,6733],"train_count":196},{"ids":[21785,150621],"train_count":196},{"ids":[23514,148327],"train_count":196},{"ids":[25497,148292,15481],"train_count":98},{"ids":[33422,153056,34226],"train_count":98},{"ids":[33638,148380],"train_count":196},{"ids":[34208,18372,14942],"train_count":98},{"ids":[35884,5092],"train_count":196},{"ids":[38241,53062,148317],"train_count":98},{"ids":[53062,73781],"train_count":196},{"ids":[71000,7105,68335],"train_count":98},{"ids":[77779,61057],"train_count":196},{"ids":[93834,12666,148317],"train_count":98},{"ids":[148281,148518,77779],"train_count":98},{"ids":[148526,7105],"train_count":196},{"ids":[148658,205403],"train_count":196},{"ids":[149244,150198],"train_count":196},{"ids":[149538,148558,72884],"train_count":98},{"ids":[150470,5301,158442],"train_count":98},{"ids":[150853,16763,149463],"train_count":98},{"ids":[151001,159484],"train_count":196},{"ids":[208002,18110,18372],"train_count":98},{"ids":[2040,34981],"train_count":195},{"ids":[7721,11777],"train_count":195},{"ids":[7721,148281,7721,77436],"train_count":65},{"ids":[13532,152865,4669,148812],"train_count":65},{"ids":[21441,168937,33422,15481],"train_count":65},{"ids":[22220,148292,61057,16806],"train_count":65},{"ids":[32724,10161,149926,155063],"train_count":65},{"ids":[41381,182032],"train_count":195},{"ids":[58427,148294,157150,34723],"train_count":65},{"ids":[148280,8789],"train_count":195},{"ids":[148388,16806],"train_count":195},{"ids":[149095,7105],"train_count":195},{"ids":[149509,16763],"train_count":195},{"ids":[149576,148357,66050,6733],"train_count":65},{"ids":[149900,87018,21785,150621],"train_count":65},{"ids":[150198,42240],"train_count":195},{"ids":[150749,13785],"train_count":195},{"ids":[152824,57868],"train_count":195},{"ids":[155726,149037],"train_count":195},{"ids":[158390,7105],"train_count":195},{"ids":[184702,155421],"train_count":195},{"ids":[198326,1424],"train_count":195},{"ids":[226564,15481],"train_count":195},{"ids":[2040,150584],"train_count":194},{"ids":[7105,148275,161782],"train_count":97},{"ids":[7721,149098,15481],"train_count":97},{"ids":[8789,41635,155511],"train_count":97},{"ids":[10983,59627,158283],"train_count":97},{"ids":[13113,45527],"train_count":194},{"ids":[13739,9080,67494],"train_count":97},{"ids":[17160,157977],"train_count":194},{"ids":[18727,5301,148280],"train_count":97},{"ids":[21785,11777],"train_count":194},{"ids":[23947,148288,22078],"train_count":97},{"ids":[34981,26125,36771],"train_count":97},{"ids":[42886,9080],"train_count":194},{"ids":[52687,27473,1739],"train_count":97},{"ids":[53887,21785,149537],"train_count":97},{"ids":[58427,153051,29119],"train_count":97},{"ids":[61057,7921],"train_count":194},{"ids":[148425,148705,148964],"train_count":97},{"ids":[148526,148288,1403],"train_count":97},{"ids":[148535,148676],"train_count":194},{"ids":[148972,159712],"train_count":194},{"ids":[149413,21797],"train_count":194},{"ids":[149576,148357],"train_count":194},{"ids":[150387,33422,94309],"train_count":97},{"ids":[150641,151666],"train_count":194},{"ids":[150852,61057],"train_count":194},{"ids":[151459,155812,4096],"train_count":97},{"ids":[153049,42565,155669],"train_count":97},{"ids":[158702,225639,149193],"train_count":97},{"ids":[159297,148277,148280],"train_count":97},{"ids":[160847,170946,148325],"train_count":97},{"ids":[207234,41546],"train_count":194},{"ids":[218574,171251,149193],"train_count":97},{"ids":[2941,171780],"train_count":193},{"ids":[7358,178595],"train_count":193},{"ids":[7921,7358],"train_count":193},{"ids":[14942,58503],"train_count":193},{"ids":[19582,21785],"train_count":193},{"ids":[24653,148276],"train_count":193},{"ids":[35884,83147],"train_count":193},{"ids":[36771,10064],"train_count":193},{"ids":[148369,3652],"train_count":193},{"ids":[152865,4669],"train_count":193},{"ids":[183258,150757],"train_count":193},{"ids":[185063,177447],"train_count":193},{"ids":[189962,149280],"train_count":193},{"ids":[217880,16256],"train_count":193},{"ids":[218582,151713],"train_count":193},{"ids":[2175,149098,15481],"train_count":96},{"ids":[4901,151853,1739],"train_count":96},{"ids":[7358,15481,4403],"train_count":96},{"ids":[7422,2115,34981,148275],"train_count":64},{"ids":[7754,148292],"train_count":192},{"ids":[11777,5092,148317],"train_count":96},{"ids":[15630,69729,182032],"train_count":96},{"ids":[16467,150167],"train_count":192},{"ids":[16467,185709,58245],"train_count":96},{"ids":[17160,148729],"train_count":192},{"ids":[20289,61628],"train_count":192},{"ids":[22078,160162],"train_count":192},{"ids":[22078,160162,16256,1739],"train_count":64},{"ids":[32482,151666],"train_count":192},{"ids":[32884,77436],"train_count":192},{"ids":[33422,153056,148293],"train_count":96},{"ids":[34208,12666],"train_count":192},{"ids":[34208,148646,7105,149193],"train_count":64},{"ids":[34981,26125,1424],"train_count":96},{"ids":[50806,148357],"train_count":192},{"ids":[54858,19160,188954,37855],"train_count":64},{"ids":[58427,79860,7358,171408],"train_count":64},{"ids":[61757,148288,1403,13292],"train_count":64},{"ids":[64535,30206,3652,148284],"train_count":64},{"ids":[68553,40794,69729],"train_count":96},{"ids":[71000,16763],"train_count":192},{"ids":[77969,8688,166815,159327],"train_count":64},{"ids":[93834,12666,4403],"train_count":96},{"ids":[148298,7754,14666,14942],"train_count":64},{"ids":[148526,148292,2115,18110],"train_count":64},{"ids":[148569,12751],"train_count":192},{"ids":[148752,53062,152225],"train_count":96},{"ids":[149382,220149,71221],"train_count":96},{"ids":[149509,148377,16763,4096],"train_count":64},{"ids":[149855,166449],"train_count":192},{"ids":[150022,36621],"train_count":192},{"ids":[150175,152138],"train_count":192},{"ids":[150545,245644,148275],"train_count":96},{"ids":[153034,1729,148917],"train_count":96},{"ids":[153683,2941,53062],"train_count":96},{"ids":[153730,177702,95306],"train_count":96},{"ids":[162564,4640],"train_count":192},{"ids":[173486,11777,7921],"train_count":96},{"ids":[204269,10161,1729],"train_count":96},{"ids":[210215,149900,87018,21785],"train_count":64},{"ids":[240350,7358,84920],"train_count":96},{"ids":[8789,93629],"train_count":191},{"ids":[13113,151883],"train_count":191},{"ids":[25497,148283],"train_count":191},{"ids":[35884,33437],"train_count":191},{"ids":[42886,148298],"train_count":191},{"ids":[153056,33477],"train_count":191},{"ids":[157991,42308],"train_count":191},{"ids":[162597,175101],"train_count":191},{"ids":[209104,151631],"train_count":191},{"ids":[10161,1729,156204],"train_count":95},{"ids":[10161,36771,1729],"train_count":95},{"ids":[10750,86128,24623],"train_count":95},{"ids":[13532,1729,77779],"train_count":95},{"ids":[13532,167465],"train_count":190},{"ids":[14942,149827,61057],"train_count":95},{"ids":[16362,148288,149076],"train_count":95},{"ids":[17816,45144,4403],"train_count":95},{"ids":[18372,25767],"train_count":190},{"ids":[18727,32482,151666],"train_count":95},{"ids":[19160,2040],"train_count":190},{"ids":[22078,2941,54025],"train_count":95},{"ids":[28696,148298],"train_count":190},{"ids":[34279,5301],"train_count":190},{"ids":[42886,36473],"train_count":190},{"ids":[45679,6389,77436],"train_count":95},{"ids":[61326,24623,151229],"train_count":95},{"ids":[61326,151673],"train_count":190},{"ids":[61757,7921,148288],"train_count":95},{"ids":[69134,149741,51788],"train_count":95},{"ids":[148283,148446],"train_count":190},{"ids":[148294,4403],"train_count":190},{"ids":[148326,148700,16256],"train_count":95},{"ids":[148526,66951],"train_count":190},{"ids":[148526,148298,7754],"train_count":95},{"ids":[148558,72884],"train_count":190},{"ids":[148876,26125,151642],"train_count":95},{"ids":[149089,34723,44633],"train_count":95},{"ids":[149345,1739,155055],"train_count":95},{"ids":[149576,148357,66050],"train_count":95},{"ids":[149591,151990],"train_count":190},{"ids":[149885,2040],"train_count":190},{"ids":[152004,7721],"train_count":190},{"ids":[152418,77779],"train_count":190},{"ids":[152786,7358],"train_count":190},{"ids":[155113,231736,149155],"train_count":95},{"ids":[155898,1450],"train_count":190},{"ids":[156476,148454,181997],"train_count":95},{"ids":[156744,13113,150800],"train_count":95},{"ids":[163961,6240,77436],"train_count":95},{"ids":[174240,18372,149591],"train_count":95},{"ids":[177153,60659],"train_count":190},{"ids":[182670,150198,42240],"train_count":95},{"ids":[186643,148554],"train_count":190},{"ids":[191297,7754,84920],"train_count":95},{"ids":[209983,6240],"train_count":190},{"ids":[211778,22078,2941],"train_count":95},{"ids":[212015,159031],"train_count":190},{"ids":[219764,29354],"train_count":190},{"ids":[227355,152027],"train_count":190},{"ids":[2175,36473,7105,149800],"train_count":63},{"ids":[6240,158718],"train_count":189},{"ids":[7358,150849],"train_count":189},{"ids":[7721,2115,26125,17460],"train_count":63},{"ids":[7754,42317,8688,148716],"train_count":63},{"ids":[9629,64257],"train_count":189},{"ids":[15481,2941,24623,13292],"train_count":63},{"ids":[16467,149280],"train_count":189},{"ids":[23947,148288],"train_count":189},{"ids":[32884,148294],"train_count":189},{"ids":[34208,79734,148812,201406],"train_count":63},{"ids":[37519,148380],"train_count":189},{"ids":[42387,149800],"train_count":189},{"ids":[57632,59627,1739,14666],"train_count":63},{"ids":[61057,148377],"train_count":189},{"ids":[64611,153624],"train_count":189},{"ids":[68140,148327],"train_count":189},{"ids":[70345,13113,58361,58245],"train_count":63},{"ids":[70345,148281,148686,232067],"train_count":63},{"ids":[148288,1450],"train_count":189},{"ids":[148292,148569],"train_count":189},{"ids":[148380,148288],"train_count":189},{"ids":[148569,149280],"train_count":189},{"ids":[148752,9080,77969,6240],"train_count":63},{"ids":[149413,148301,31608,151246],"train_count":63},{"ids":[150029,1403],"train_count":189},{"ids":[151709,152973,77436,153232],"train_count":63},{"ids":[151831,148413],"train_count":189},{"ids":[153271,28696],"train_count":189},{"ids":[155152,7754,42317,8688],"train_count":63},{"ids":[155891,45527,148397,28696],"train_count":63},{"ids":[161997,167193],"train_count":189},{"ids":[184702,42565,148675,12751],"train_count":63},{"ids":[2115,148877,4096],"train_count":94},{"ids":[6733,148841],"train_count":188},{"ids":[7721,71658],"train_count":188},{"ids":[7958,28696,7105],"train_count":94},{"ids":[9080,148293],"train_count":188},{"ids":[13659,12666],"train_count":188},{"ids":[14942,77436,153232],"train_count":94},{"ids":[16362,13113,158140],"train_count":94},{"ids":[16362,22078,11777],"train_count":94},{"ids":[20177,1729,21785],"train_count":94},{"ids":[23355,7921],"train_count":188},{"ids":[26125,150047,19937],"train_count":94},{"ids":[26149,44432,29354],"train_count":94},{"ids":[26149,148281,82557],"train_count":94},{"ids":[32392,148288],"train_count":188},{"ids":[34960,154408],"train_count":188},{"ids":[36736,18372,32884],"train_count":94},{"ids":[42886,61057,7105],"train_count":94},{"ids":[46807,60860],"train_count":188},{"ids":[54724,149591,15481],"train_count":94},{"ids":[54858,14942],"train_count":188},{"ids":[66050,14942,156977],"train_count":94},{"ids":[74353,18372],"train_count":188},{"ids":[89315,4403],"train_count":188},{"ids":[148294,42565,37855],"train_count":94},{"ids":[148336,7105],"train_count":188},{"ids":[148414,95525],"train_count":188},{"ids":[148526,84920,13113],"train_count":94},{"ids":[148686,16806],"train_count":188},{"ids":[149413,148301,31608],"train_count":94},{"ids":[150281,19937],"train_count":188},{"ids":[150749,58245],"train_count":188},{"ids":[150764,149550,148939],"train_count":94},{"ids":[150836,26269,11796],"train_count":94},{"ids":[152418,50022],"train_count":188},{"ids":[156595,148811,1424],"train_count":94},{"ids":[157172,148557],"train_count":188},{"ids":[161978,190978],"train_count":188},{"ids":[205914,8688,149885],"train_count":94},{"ids":[210216,13785],"train_count":188},{"ids":[4640,77779],"train_count":187},{"ids":[6733,58475],"train_count":187},{"ids":[16797,152703],"train_count":187},{"ids":[18727,18372],"train_count":187},{"ids":[29196,17941],"train_count":187},{"ids":[34960,2941],"train_count":187},{"ids":[41381,148629],"train_count":187},{"ids":[45679,1739],"train_count":187},{"ids":[66353,6240],"train_count":187},{"ids":[150190,150095],"train_count":187},{"ids":[151216,47474],"train_count":187},{"ids":[153056,60038],"train_count":187},{"ids":[167821,148614],"train_count":187},{"ids":[189378,54201],"train_count":187},{"ids":[2175,149591,7105],"train_count":93},{"ids":[4669,21785],"train_count":186},{"ids":[5624,10750,86128],"train_count":93},{"ids":[7721,5301,47745],"train_count":93},{"ids":[8688,172518,148275],"train_count":93},{"ids":[11777,8789,41635],"train_count":93},{"ids":[12666,34981,7105,1729],"train_count":62},{"ids":[14942,45587],"train_count":186},{"ids":[15481,2941,24623],"train_count":93},{"ids":[15481,9080,7921],"train_count":93},{"ids":[15729,154285,170744,241144],"train_count":62},{"ids":[17050,148318,1729],"train_count":93},{"ids":[18768,15630,148275,31889],"train_count":62},{"ids":[19160,188954,37855,2040],"train_count":62},{"ids":[20578,159541,5301],"train_count":93},{"ids":[26607,150800],"train_count":186},{"ids":[35105,160776,185709],"train_count":93},{"ids":[39427,149591,7105],"train_count":93},{"ids":[41381,148397,9629,11777],"train_count":62},{"ids":[42565,6389],"train_count":186},{"ids":[45527,78330],"train_count":186},{"ids":[48629,2115,148877,4096],"train_count":62},{"ids":[53555,148380,34981],"train_count":93},{"ids":[57272,178537,45405,4403],"train_count":62},{"ids":[63081,1739,150188],"train_count":93},{"ids":[65445,1739,7105,22078],"train_count":62},{"ids":[74990,148518,42317],"train_count":93},{"ids":[75246,148378,5301],"train_count":93},{"ids":[86022,6240],"train_count":186},{"ids":[148390,151828,7105,149193],"train_count":62},{"ids":[148646,150937,10064,153232],"train_count":62},{"ids":[149085,15481],"train_count":186},{"ids":[149282,36621,9080,33477],"train_count":62},{"ids":[149430,182503,148377],"train_count":93},{"ids":[150140,36771,45480],"train_count":93},{"ids":[150580,201647,7721,148288],"train_count":62},{"ids":[150919,29119,197306],"train_count":93},{"ids":[152731,148942],"train_count":186},{"ids":[153347,8688,13113,1424],"train_count":62},{"ids":[154382,8789],"train_count":186},{"ids":[155132,159400,42565,5301],"train_count":62},{"ids":[155812,17941],"train_count":186},{"ids":[156444,222837],"train_count":186},{"ids":[163028,148303,1450],"train_count":93},{"ids":[164590,149904,61057,148377],"train_count":62},{"ids":[171133,10064,87658],"train_count":93},{"ids":[176215,148275,13785],"train_count":93},{"ids":[185510,171717],"train_count":186},{"ids":[207347,150029,1403],"train_count":93},{"ids":[225381,220495],"train_count":186},{"ids":[237856,42565,82942],"train_count":93},{"ids":[7550,168823],"train_count":185},{"ids":[16362,148288],"train_count":185},{"ids":[18372,16256],"train_count":185},{"ids":[23355,7358],"train_count":185},{"ids":[24623,1450],"train_count":185},{"ids":[37835,61628],"train_count":185},{"ids":[57272,148279],"train_count":185},{"ids":[64535,148420],"train_count":185},{"ids":[150714,158605],"train_count":185},{"ids":[164904,36473],"train_count":185},{"ids":[2040,153195],"train_count":184},{"ids":[2115,148877],"train_count":184},{"ids":[2175,9629,53555],"train_count":92},{"ids":[7347,2115],"train_count":184},{"ids":[7422,27473],"train_count":184},{"ids":[7721,158638],"train_count":184},{"ids":[7754,15481],"train_count":184},{"ids":[7958,204086],"train_count":184},{"ids":[10983,172422,8789],"train_count":92},{"ids":[11777,7921],"train_count":184},{"ids":[14942,47745,172637],"train_count":92},{"ids":[16362,148280,148276],"train_count":92},{"ids":[18727,82577,69632],"train_count":92},{"ids":[19736,150270],"train_count":184},{"ids":[22220,212703],"train_count":184},{"ids":[26149,148445,148381],"train_count":92},{"ids":[32392,150405],"train_count":184},{"ids":[33422,1424,153177],"train_count":92},{"ids":[34208,2115,34981],"train_count":92},{"ids":[34960,2040,10161],"train_count":92},{"ids":[36183,157759,151407],"train_count":92},{"ids":[39427,37835],"train_count":184},{"ids":[41381,149340,149749],"train_count":92},{"ids":[43080,5301],"train_count":184},{"ids":[52337,36183,157759],"train_count":92},{"ids":[53062,65360],"train_count":184},{"ids":[58361,6733],"train_count":184},{"ids":[61324,18372],"train_count":184},{"ids":[62709,165171,6240],"train_count":92},{"ids":[74176,5092,89771],"train_count":92},{"ids":[75117,48941,41580],"train_count":92},{"ids":[91206,47474,42565],"train_count":92},{"ids":[148275,16763,6733],"train_count":92},{"ids":[148646,16763],"train_count":184},{"ids":[149679,150469,42565],"train_count":92},{"ids":[150545,152578],"train_count":184},{"ids":[151963,149338],"train_count":184},{"ids":[152211,159545],"train_count":184},{"ids":[152595,6240,84706],"train_count":92},{"ids":[155651,18110],"train_count":184},{"ids":[155812,149280],"train_count":184},{"ids":[168396,148676],"train_count":184},{"ids":[201085,148276],"train_count":184},{"ids":[225833,16763,4096],"train_count":92},{"ids":[2040,55389],"train_count":183},{"ids":[2175,12751,60659,5092],"train_count":61},{"ids":[3652,149383],"train_count":183},{"ids":[7347,7358],"train_count":183},{"ids":[7958,10161,12666,4403],"train_count":61},{"ids":[12696,14666],"train_count":183},{"ids":[13113,1450],"train_count":183},{"ids":[14942,39427],"train_count":183},{"ids":[14942,72884],"train_count":183},{"ids":[16797,93834,22078,160162],"train_count":61},{"ids":[21785,93942],"train_count":183},{"ids":[25497,158231,148558,148276],"train_count":61},{"ids":[26125,36916],"train_count":183},{"ids":[34208,148646,7105,149280],"train_count":61},{"ids":[35105,14942,149827,61057],"train_count":61},{"ids":[37519,148292,1739,38183],"train_count":61},{"ids":[47552,28696,45527,151201],"train_count":61},{"ids":[61326,2115,34981,149280],"train_count":61},{"ids":[62847,22078,24623,149280],"train_count":61},{"ids":[74990,15481],"train_count":183},{"ids":[148526,149576,148275,1739],"train_count":61},{"ids":[148879,148303,11777,72884],"train_count":61},{"ids":[149147,1450],"train_count":183},{"ids":[149269,162547],"train_count":183},{"ids":[149810,16467],"train_count":183},{"ids":[151211,157991],"train_count":183},{"ids":[151232,59627,1739,14666],"train_count":61},{"ids":[151519,6733],"train_count":183},{"ids":[154390,53887],"train_count":183},{"ids":[154692,3652],"train_count":183},{"ids":[156114,67494],"train_count":183},{"ids":[158231,148558,148276,44633],"train_count":61},{"ids":[163028,148303,1450,18768],"train_count":61},{"ids":[166636,12666,149086,16669],"train_count":61},{"ids":[166897,246235],"train_count":183},{"ids":[203031,21785,149098,92647],"train_count":61},{"ids":[209071,7754],"train_count":183},{"ids":[221647,10064,13113,45527],"train_count":61},{"ids":[1739,83895],"train_count":182},{"ids":[2040,10161,36771],"train_count":91},{"ids":[2115,34981,148275],"train_count":91},{"ids":[6240,168329],"train_count":182},{"ids":[7721,26125,148325],"train_count":91},{"ids":[7754,84920,55055],"train_count":91},{"ids":[8688,47474,4669],"train_count":91},{"ids":[8688,92647],"train_count":182},{"ids":[11777,148288],"train_count":182},{"ids":[12696,189994],"train_count":182},{"ids":[16362,22078,42565],"train_count":91},{"ids":[18372,150194],"train_count":182},{"ids":[22078,150800,44633],"train_count":91},{"ids":[24623,42240],"train_count":182},{"ids":[31889,1739],"train_count":182},{"ids":[33207,149208],"train_count":182},{"ids":[42565,179888],"train_count":182},{"ids":[42886,148301,2175],"train_count":91},{"ids":[56755,64257,4096],"train_count":91},{"ids":[57868,45480],"train_count":182},{"ids":[61757,31889],"train_count":182},{"ids":[64535,2115],"train_count":182},{"ids":[76747,148281,154299],"train_count":91},{"ids":[148275,149193],"train_count":182},{"ids":[148292,61057],"train_count":182},{"ids":[148558,72884,68608],"train_count":91},{"ids":[148561,7358,55389],"train_count":91},{"ids":[148782,71658,63046],"train_count":91},{"ids":[149086,53062],"train_count":182},{"ids":[149269,162547,6389],"train_count":91},{"ids":[150083,19937],"train_count":182},{"ids":[150198,148325],"train_count":182},{"ids":[150466,148330,33422],"train_count":91},{"ids":[153306,149213,198326],"train_count":91},{"ids":[153848,152657],"train_count":182},{"ids":[154330,148317],"train_count":182},{"ids":[156234,148524,8789],"train_count":91},{"ids":[158013,15481,9080],"train_count":91},{"ids":[162683,148280,7921],"train_count":91},{"ids":[185120,9629,11577],"train_count":91},{"ids":[185942,148537],"train_count":182},{"ids":[220630,36771,16806],"train_count":91},{"ids":[225833,16763,151666],"train_count":91},{"ids":[239017,74359,1424],"train_count":91},{"ids":[239300,150947,7105],"train_count":91},{"ids":[244015,158731,21785],"train_count":91},{"ids":[16256,13292],"train_count":181},{"ids":[16806,12751],"train_count":181},{"ids":[18727,151963],"train_count":181},{"ids":[24653,148292],"train_count":181},{"ids":[25497,62709],"train_count":181},{"ids":[26125,4403],"train_count":181},{"ids":[36473,9080],"train_count":181},{"ids":[42886,149099],"train_count":181},{"ids":[69729,149137],"train_count":181},{"ids":[74990,57868],"train_count":181},{"ids":[82929,148294],"train_count":181},{"ids":[149937,31889],"train_count":181},{"ids":[150635,2941],"train_count":181},{"ids":[162499,157991],"train_count":181},{"ids":[182670,84480],"train_count":181},{"ids":[237856,150396],"train_count":181},{"ids":[2115,10161,5092],"train_count":90},{"ids":[2175,153922,51788],"train_count":90},{"ids":[2941,22078],"train_count":180},{"ids":[7550,148458],"train_count":180},{"ids":[7721,26125,42240],"train_count":90},{"ids":[7721,77436,153232],"train_count":90},{"ids":[9629,16806],"train_count":180},{"ids":[12666,34981,7105],"train_count":90},{"ids":[13532,13785,149161,148275],"train_count":60},{"ids":[14942,26125],"train_count":180},{"ids":[16362,148280,148276,153009],"train_count":60},{"ids":[16362,148381,153009],"train_count":90},{"ids":[19736,167339],"train_count":180},{"ids":[21785,11777,5092],"train_count":90},{"ids":[22346,6733,168548],"train_count":90},{"ids":[24653,150396,159796],"train_count":90},{"ids":[26125,1424,13302],"train_count":90},{"ids":[29189,152950,168329],"train_count":90},{"ids":[32868,154346,152973,158585],"train_count":60},{"ids":[34208,148561,5092,148369],"train_count":60},{"ids":[36473,2115],"train_count":180},{"ids":[39427,150842],"train_count":180},{"ids":[42886,148646,150937,238492],"train_count":60},{"ids":[50806,36771,10064,163082],"train_count":60},{"ids":[50806,148294,12666,148293],"train_count":60},{"ids":[61326,33437,149637],"train_count":90},{"ids":[61326,149244],"train_count":180},{"ids":[64088,7721,36103],"train_count":90},{"ids":[64257,4096,66050],"train_count":90},{"ids":[66058,72884,66353,5301],"train_count":60},{"ids":[148277,61057,152993],"train_count":90},{"ids":[148292,13113,15630],"train_count":90},{"ids":[148489,9080,148518],"train_count":90},{"ids":[148535,74990,15481],"train_count":90},{"ids":[149092,167834,22078],"train_count":90},{"ids":[149208,7721,29119,155055],"train_count":60},{"ids":[149282,36621,9080,148449],"train_count":60},{"ids":[149746,8789,77436,150621],"train_count":60},{"ids":[150595,45527,78330,26125],"train_count":60},{"ids":[150764,149595,58503],"train_count":90},{"ids":[150828,42317,67958,17941],"train_count":60},{"ids":[151519,6733,57868,148593],"train_count":60},{"ids":[152870,148489,15630],"train_count":90},{"ids":[153942,16467],"train_count":180},{"ids":[154385,68608],"train_count":180},{"ids":[156476,170391],"train_count":180},{"ids":[159096,149648],"train_count":180},{"ids":[163961,6240,77436,149537],"train_count":60},{"ids":[174249,155812,4096,48046],"train_count":60},{"ids":[177220,18768],"train_count":180},{"ids":[178147,7721,77969,6240],"train_count":60}]'''
merges = json.loads(MERGES_JSON)
print(f"Словарь: {len(tokenizer)} токенов")
print(f"Объединений: {len(merges)}")


Словарь: 248077 токенов
Объединений: 4096


In [3]:
import re

WORD = re.compile(r" ?[А-Яа-яЁё]+")


def encode(text, tokenizer, regions=None):
    """Assign each eligible token to one word inside an allowed region."""
    encoded = tokenizer(text, add_special_tokens=False,
                        return_offsets_mapping=True)
    owners = [0] * len(text)
    regions = [(0, len(text))] if regions is None else regions
    for number, match in enumerate(WORD.finditer(text), 1):
        start, end = match.span()
        if any(left <= start and end <= right for left, right in regions):
            owners[start:end] = [number] * (end - start)
    groups = []
    specials = set(tokenizer.all_special_ids)
    for token, (start, end) in zip(encoded["input_ids"],
                                   encoded["offset_mapping"], strict=True):
        owner = owners[start] if 0 <= start < end <= len(text) else 0
        if (token in specials
                or (owner and any(x != owner for x in owners[start:end]))):
            owner = 0
        groups.append(owner)
    return encoded["input_ids"], groups


class Merger:
    """Tuples are input embedding groups, never IDs sent to output head."""

    def __init__(self, entries):
        self.by_first = {}
        seen = set()
        for item in entries:
            seq = tuple(item["ids"])
            if (len(seq) < 2 or seq in seen
                    or any(type(x) is not int or x < 0 for x in seq)):
                raise ValueError("Invalid or duplicate merge")
            seen.add(seq)
            self.by_first.setdefault(seq[0], []).append(seq)
        for values in self.by_first.values():
            values.sort(key=lambda seq: (-len(seq), seq))

    def apply(self, ids, owners):
        if len(ids) != len(owners):
            raise ValueError("IDs and owners must align")
        result, index = [], 0
        while index < len(ids):
            selected = None
            if owners[index]:
                for seq in self.by_first.get(ids[index], ()):
                    end = index + len(seq)
                    if (tuple(ids[index:end]) == seq
                            and all(owner == owners[index]
                                    for owner in owners[index:end])):
                        selected = seq
                        break
            result.append(ids[index] if selected is None else selected)
            index += 1 if selected is None else len(selected)
        return result


merger = Merger(merges)

sentence = "Ссылки на дополнительные Примечания были также в конце статьи"
ids, owners = encode(sentence, tokenizer)
merged = merger.apply(ids, owners)

print(f"Предложение: {sentence!r}")
print(f"Токенов до объединения:  {len(ids)}")
print(f"Позиций после объединения: {len(merged)}")
print()
print("Восстановление исходных ID (проверка обратимости):")
restored = [t for group in merged
            for t in (group if isinstance(group, tuple) else (group,))]
print(f"  restored == original: {restored == ids}")
print(f"  decode(restored) == decode(original): "
      f"{tokenizer.decode(restored) == tokenizer.decode(ids)}")


Предложение: 'Ссылки на дополнительные Примечания были также в конце статьи'
Токенов до объединения:  13
Позиций после объединения: 10

Восстановление исходных ID (проверка обратимости):
  restored == original: True
  decode(restored) == decode(original): True


На этом предложении 13 токенов схлопнулись в 10 позиций, и исходные ID полностью восстанавливаются (обратимость — обязательное условие метода). Вопрос — насколько такой эффект воспроизводится на полном корпусе и на реальных вопросах теста.

## Результат

Агрегированные числа — результат уже проведённого измерения на полном корпусе (10 000 train + 1 000 dev документов Wikipedia RU и 70 реальных вопросах MMLU-ProX-Lite RU validation). Здесь они не пересчитываются заново, чтобы не тянуть весь корпус повторно — механизм выше выполнен вживую, на настоящих данных.

In [4]:
report = {
    "train": {"reduction": 0.09671582220485009},
    "dev": {"reduction": 0.09332831375737205},
    "validation": {"reduction": 0.02473520644383842},
    "timing": {"overhead_ms_per_document": 0.7392235400038771},
    "passed": False,
    "decision": "no_go",
}

rows = [
    ("train (10 000 док.)", report["train"]["reduction"]),
    ("dev (1 000 док.)", report["dev"]["reduction"]),
    ("validation MCQ (70 промптов)", report["validation"]["reduction"]),
]
print(f"{'набор':<32}{'сокращение позиций':>22}")
for name, reduction in rows:
    print(f"{name:<32}{reduction * 100:>21.2f}%")

print()
print("Порог перехода к GPU: сокращение >= 5.0% И на dev, И на validation")
print(f"Итог gate: passed = {report['passed']}, "
      f"decision = {report['decision']}")
print(f"CPU overhead: {report['timing']['overhead_ms_per_document']:.3f} "
      f"мс/документ")


набор                               сокращение позиций
train (10 000 док.)                              9.67%
dev (1 000 док.)                                 9.33%
validation MCQ (70 промптов)                     2.47%

Порог перехода к GPU: сокращение >= 5.0% И на dev, И на validation
Итог gate: passed = False, decision = no_go
CPU overhead: 0.739 мс/документ


Порог (≥5%) пройден на корпусе (9.33%), но не на реальных вопросах
задачи (2.47%) — короткие вопросы с вариантами ответа дают заметно
меньше повторяющихся длинных n-грам, чем связный текст Wikipedia.

## Дополнительная диагностика

Отдельно (пост-хок, вне этого прогона) проверен теоретический потолок
для этого класса методов на тех же MCQ-вопросах: оптимальная
фиксированная сегментация практически равна жадной (~2.47%), условный
верхний предел (объединение каждого доступного слова в одну позицию) —
15.02%. То есть даже у идеальной версии этого механизма запас невелик
именно для коротких вопросов с вариантами ответа.

## Вывод

Кандидат не отправлялся на GPU-обучение — порог сокращения на реальных
данных задачи не пройден.
